# Structural aliasing in Chronos-Bolt
## The five Bayesian models of Deliverable 2, fitted on the fifteen retrained geometries

**PATCHALIAS, final implementation.** This notebook fits the five model families specified in
`coursework/deliverable2_v0/sections/deliverable2.tex` and stated in full in
`coursework/deliverable3/sections/B_bayesian_appendix.tex`. Its population is the fifteen
uniformly retrained patch/stride geometries of `tab:hfModels`; every analysis context is 480
samples and closes on a stride boundary.

| Part | Claim | Model | Estimand | The claim predicts | Response |
|---|---|---|---|---|---|
| 4.1 | H1 behavioural | A | $e^{\bar\beta}$ | $<1$ | paired recovery contrast |
| 4.1 | M1 mitigation | A$'$ | $\delta_O$ | $>0$ | the same contrast, read across geometries |
| 4.2 | H1 representational | B | $e^{\theta_{\mathrm{lock}}}$ | $>1$ | probe codelength in bits |
| 4.3 | H2 phase invariance | C | $\sigma_\phi$ | $\approx 0$ | the per-phase deficit |
| 4.4 | H3 location | D1 | $\theta_S,\ \theta_P$ | both $<0$ | log token collapse |
| 4.5 | H3a, H3b movement | D2 | $\kappa_S,\ \kappa_P$ | $=1$ | detected site spacing |

---

### How to run it

1. **Runtime > Run all.** The first cell installs a pinned environment and restarts the runtime
   once; Colab reports a crashed session, which is that restart. Choose **Run all** again.
2. That is the whole procedure. Everything else is automatic: the notebook decides whether it
   needs to collect observations or only to analyse them, resumes any stage already finished, and
   stops with a named reason if a gate fails.
3. All settings live in one form, **Part 0.4**. Nothing elsewhere needs editing.

**It is a long run, and the notebook is built for that rather than around it.** From an empty
folder it collects 1,113,000 forecasts before it fits anything, and then fits about sixty-five
posteriors, eleven of them on the full 371,000 observations. On a CPU runtime the collection is of the
order of ten hours and the sampling is of the same order again, so a complete run is a day's work
in wall time and not a single sitting. Nothing about that requires supervision: every stage is
checkpointed, an interrupted session costs only the table or the pair of chains it was writing, and
re-running the notebook continues from where it stopped. Part 2.1 measures the collection rate on
this runtime and prints a session plan before committing to it, and Part 0.9 projects the remainder
of the whole run as it goes.

Two levers shorten it, both in the forms above. A GPU runtime collects in under an hour instead of
ten, and the analysis afterwards is unaffected because it reads the collected tables from Drive.
And `NUTS_BACKEND` in Part 0.2 can be set to `nutpie`, a Rust implementation of the same sampler
that is several times faster on the same CPU: it targets the same posterior with the same priors
and the same number of draws, and every diagnostic gate applies to it unchanged.

---

### What to look at when it finishes

| Output | Where |
|---|---|
| One row per claim: pre-gate reading, every gate separately, post-gate verdict | Part 5.5 |
| The same, as a dashboard with the gate matrix | Part 6.6 |
| Every estimand against its own decision threshold | Part 6.1 |
| How far the data moved each estimand from its prior | Part 6.2 |
| Whether the sampler can be believed | Parts 5.1, 6.3 and 6.4 |
| Whether the models can reproduce the data | Part 5.2 |
| Whether the answers depend on the prior | Part 5.3 |
| Everything, as files | `tables/*.csv` and `figures/*.png` in the run folder |

**Part 6 runs on its own.** It reads only the saved artifacts, so the figures can be regenerated
in a fresh runtime with no Chronos, no PyMC sampling and no collection: set
`STANDALONE_FIGURES = True` in Part 0.4 and run all.

---

### Two things this notebook does that its predecessors did not

Appendix B requires each set of group offsets to sum to zero, because the offsets enter one linear
predictor additively and a constant moved from one set to another leaves the likelihood unchanged.
No earlier notebook implemented that for Models A and C, and the consequence was visible in their
output: $\hat R$ up to 2.23 at an effective sample size of 5 **with zero divergences**, which is
the signature of a flat ridge, not of a funnel. `u_harm`, `u_bg` and `u_phase` are `ZeroSumNormal`
here, and Part 3.2 says exactly which sets are constrained and why `beta_c` is not one of them.

Model A carries 371,000 observations, so its `log_likelihood` at four chains and 2000 retained
draws is 23.7 GiB against the 10 GB this runtime has. No fit computes one. The log likelihood is
reconstructed from the posterior in observation slices instead. Part 3.5 proves that reconstruction
against `pm.compute_log_likelihood`, and Part 3.6 proves the streaming PSIS-LOO built on it against
`az.loo`, both to better than $10^{-8}$ and before anything is allowed to depend on either. That is
what makes full-data leave-one-out affordable without subsampling any of it.

Smoke output and synthetic parameter recovery validate the pipeline. They are never evidence
about Chronos, and the verdict table marks them as non-reportable by construction.

## 0.1, Repository

Locate the checkout that holds the support scripts, cloning it on a fresh hosted runtime. Nothing
under `support_scripts/` is written by this notebook: the scripts are read, hashed, and recorded
in the run manifest, so a later change to any of them is visible as a changed fingerprint rather
than as a silently different result.

`collect.py` and `probe_lib.py` do carry three changes made for this analysis, listed in section
8.1 of the companion report: the tone amplitude moved into the collection's configuration so that
it enters the design fingerprint, the contrast collector forwards candidate frequencies in blocks
so that a geometry's working set is bounded, and the design gate takes the response it is being
applied for. They are library properties rather than properties of one analysis, so they were fixed
there instead of patched at run time from here. Because both files are hashed into every
collection's fingerprint, a collection made before those changes will refuse to be resumed and will
say so; this run collects into a new folder, so nothing is lost.

In [ ]:
#@title 0.1  Locate (or clone) the repository
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

MARKER = Path("chronos") / "bayesian" / "support_scripts" / "probe_lib.py"


def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                               REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target


REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
SUPPORT = BAYES_DIR / "support_scripts"
if str(SUPPORT) not in sys.path:
    sys.path.insert(0, str(SUPPORT))

try:
    REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                          text=True).strip()
except Exception:
    REPO_COMMIT = "unknown"

print("repository :", REPO)
print("commit     :", REPO_COMMIT)
print("modules    :", SUPPORT)

## 0.2, Environment

One pinned install, one deliberate restart, verified in a clean subprocess.

The alternative, exporting `uv.lock` and installing all of it, is what earlier notebooks did and
is avoided here for two reasons their own comments record. Colab imports numpy into the kernel
process before any user cell runs, so installing a different numpy leaves an already-compiled
extension resident in memory beside newer Python source, and a later `import arviz` fails several
frames deep inside `scipy` with a message that blames neither. And installing a locked `torch`
over Colab's own build leaves Colab's `torchvision` behind with a mismatched ABI, so
`import chronos` dies on `operator torchvision::nms does not exist`. A short pinned list plus one
restart avoids both; `torchvision` and `torchaudio`, which this project does not depend on at all,
are removed rather than version-matched.

> **When the runtime restarts, Colab shows a red "session crashed" banner. That is this cell
> restarting on purpose.** Choose *Runtime > Run all* again: the cell detects that the environment
> is already installed and verified, and steps straight past it.

In [ ]:
#@title 0.2  Install the pinned environment (restarts the runtime once)
import importlib.util, json, tempfile

INSTALL_CHRONOS = True  #@param {type:"boolean"}
NUTS_BACKEND = "pymc"  #@param ["pymc", "nutpie"]

REQUIREMENTS = [
    "pymc==5.28.5", "arviz==0.22.0",
    "numpy>=1.26,<3", "pandas>=2,<4", "scipy>=1.11,<2",
    "pyarrow", "h5netcdf", "h5py", "matplotlib", "packaging",
]
STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_final_env.json"


def _on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


def _clean_import_healthy() -> bool:
    """Import the numpy/scipy/arviz/pymc chain in a NEW process, so the answer describes what is
    on disk rather than whatever this (possibly already poisoned) process has cached."""
    probe = subprocess.run(
        [sys.executable, "-c", "import numpy, scipy.special, scipy.stats, arviz, pymc"],
        capture_output=True, text=True)
    if probe.returncode != 0:
        print(probe.stderr[-1500:])
    return probe.returncode == 0


def _unsatisfied() -> list[str]:
    from packaging.requirements import Requirement
    from importlib import metadata
    missing = []
    for text in REQUIREMENTS:
        requirement = Requirement(text)
        try:
            if metadata.version(requirement.name) not in requirement.specifier:
                missing.append(text)
        except metadata.PackageNotFoundError:
            missing.append(text)
    return missing


def _restart(reason: str) -> None:
    print("\n" + "=" * 78)
    print(f"RESTARTING THE RUNTIME NOW ({reason}).")
    print("Colab will report a crashed session. That is this cell restarting deliberately.")
    print("When it reconnects choose Runtime > Run all again; the install is then skipped.")
    print("=" * 78)
    sys.stdout.flush()
    import time as _time
    _time.sleep(2)
    os.kill(os.getpid(), 9)


_state = json.loads(STATE_PATH.read_text()) if STATE_PATH.is_file() else {"stage": "fresh"}

if _state["stage"] == "fresh":
    _missing = _unsatisfied()
    if _missing or INSTALL_CHRONOS:
        with tempfile.TemporaryDirectory() as _tmp:
            _constraints = Path(_tmp) / "constraints.txt"
            _constraints.write_text("\n".join(REQUIREMENTS) + "\n")
            if _missing:
                print("installing:", _missing)
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                       "-c", str(_constraints), *REQUIREMENTS])
            if INSTALL_CHRONOS:
                # chronos-forecasting brings torch and transformers; the constraints hold the
                # statistics stack at the pinned versions while it resolves.
                print("installing: chronos-forecasting (for Part 2)")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                       "-c", str(_constraints), "chronos-forecasting"])
                subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                                "torchvision", "torchaudio"], check=False, capture_output=True)
            if NUTS_BACKEND != "pymc":
                print(f"installing: {NUTS_BACKEND} (the sampler chosen above)")
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                       "-c", str(_constraints), NUTS_BACKEND])
    STATE_PATH.write_text(json.dumps({"stage": "installed"}))
    if _on_colab():
        _restart("pinned environment just installed")
    else:
        print("Environment installed. Restart the kernel by hand and resume at this cell.")
elif _state["stage"] == "installed":
    if _clean_import_healthy():
        STATE_PATH.write_text(json.dumps({"stage": "verified"}))
        print("Environment installed and verified clean in this runtime.")
    else:
        STATE_PATH.write_text(json.dumps({"stage": "failed"}))
        raise RuntimeError(
            "numpy/scipy/arviz/pymc still do not import cleanly after one restart. Use "
            "Runtime > Disconnect and delete runtime for a genuinely fresh VM, then Run all once.")
else:
    print("Environment already verified in this runtime." if _state["stage"] == "verified"
          else "Environment previously failed to verify; see the message above.")


def _real_module(name: str) -> bool:
    try:
        spec = importlib.util.find_spec(name)
    except (ImportError, ValueError):
        return False
    return spec is not None and spec.origin is not None


HAVE_TORCH = _real_module("torch") and _real_module("chronos")
print({"Chronos can be loaded in this runtime": HAVE_TORCH})

## 0.3, Imports, seed and house style

In [ ]:
#@title 0.3  Imports, seed and house style
from __future__ import annotations

import gc, hashlib, math, random, shutil, time, types, uuid
from dataclasses import dataclass, replace
from importlib import metadata as importlib_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import arviz as az
import pymc as pm
import pytensor.tensor as pt
import xarray as xr
from scipy import stats
from scipy.special import gammaln

try:
    from IPython.display import HTML, display
    _RICH = True
except ImportError:                                   # a plain interpreter, e.g. a CI check
    _RICH = False

    def display(*args, **kwargs):
        for item in args:
            print(item)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

plt.rcParams.update({"figure.dpi": 115, "savefig.dpi": 140,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.18, "grid.linewidth": 0.6,
                     "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
                     "legend.frameon": False, "figure.facecolor": "white"})
INK = {"primary": "#245B83", "accent": "#A85D21", "good": "#447C61", "bad": "#8E3B3B",
       "muted": "#8A939C", "line": "#333A42", "grid_s": "#6A1B9A", "grid_p": "#1565C0"}

VERSIONS = {}
for _name in ("pymc", "arviz", "numpy", "pandas", "scipy", "pytensor", "xarray"):
    try:
        VERSIONS[_name] = importlib_metadata.version(_name)
    except importlib_metadata.PackageNotFoundError:
        VERSIONS[_name] = None
print("versions:", VERSIONS)


def show(frame, rows: int | None = None, title: str | None = None, caption: str | None = None):
    """Render a table. In a notebook this is styled HTML; elsewhere it is plain text."""
    view = frame if rows is None else frame.head(rows)
    if title:
        print(f"\n{title}")
    if not _RICH or not isinstance(view, pd.DataFrame):
        print(view.to_string(index=False))
        if caption:
            print(f"  ({caption})")
        return
    styler = (view.style
              .hide(axis="index")
              .format(precision=4, na_rep="-")
              .set_table_styles([
                  {"selector": "th", "props": [("background-color", "#EEF2F5"),
                                               ("color", "#1F2933"), ("font-weight", "600"),
                                               ("text-align", "left"),
                                               ("border-bottom", "2px solid #C3CCD4"),
                                               ("padding", "4px 8px")]},
                  {"selector": "td", "props": [("padding", "3px 8px"),
                                               ("border-bottom", "1px solid #EDF1F4")]},
                  {"selector": "caption", "props": [("caption-side", "bottom"),
                                                    ("font-size", "85%"),
                                                    ("color", "#5A6570"),
                                                    ("padding-top", "6px")]}]))
    if caption:
        styler = styler.set_caption(caption)
    display(styler)
    if rows is not None and len(frame) > rows:
        print(f"  ({rows} of {len(frame)} rows shown)")


def note(text: str, kind: str = "info") -> None:
    """A boxed statement, so a gate result or a limitation is not lost in the log."""
    palette = {"info": ("#EEF4F8", "#245B83"), "good": ("#EDF5F0", "#447C61"),
               "warn": ("#FBF3E9", "#A85D21"), "bad": ("#F9EDED", "#8E3B3B")}
    background, edge = palette.get(kind, palette["info"])
    if _RICH:
        display(HTML(f"<div style='background:{background};border-left:4px solid {edge};"
                     f"padding:8px 12px;margin:6px 0;font-size:90%;line-height:1.45'>"
                     f"{text}</div>"))
    else:
        print(f"[{kind.upper()}] {text}")

## 0.4, Control panel

Every setting of the run is here, and nothing elsewhere in the notebook needs editing. The
defaults are the reportable configuration.

| Setting | What it decides |
|---|---|
| `NUTS_BACKEND` | In Part 0.2 rather than here, because it has to be installed. `pymc` is the reference implementation; `nutpie` is a Rust one that is several times faster on the same CPU. Both are NUTS on the same posterior with the same priors and the same draws, so the choice is one of throughput. |
| `MODE` | `full` is the reportable run. `smoke` exercises the whole chain on tiny grids and can never produce a verdict. `preflight` collects the same tiny grids and runs everything up to and including the parity gates and parameter recovery, which are themselves fits, then stops before the reportable ones. |
| `RECOVERY_DENOMINATOR` | What the amplitude recovery is measured against. `injected` is the amplitude the tone was injected at, which is how Appendix B defines it, and is the default. `true_continuation` is the amplitude the true future actually carries at that frequency, which is what the shared estimator has always returned. Both are computed from the same collection; Part 2.4 shows what the choice does to the response. |
| `STAGE` | Leave it on `auto`. It inspects the run folder: with no collection it collects and then analyses, with a complete one it analyses only, and with neither a collection nor a usable runtime it stops and says which. |
| `TONE_SNR` | Tone amplitude over a unit-variance background. The one design constant the report does not fix, so it is set here and enters the run fingerprint. |
| `STANDALONE_FIGURES` | Regenerate Part 6 from the saved artifacts alone. Collection and sampling are refused, so a missing artifact is reported by name instead of being recomputed over hours. |
| `FIT_MODEL_*` | Which families this session fits. A model already finished in any session reloads from its checkpoint regardless, so these split the work across sessions rather than changing what is analysed. |

**The sampler settings are deliberately not on this panel.** Appendix B fixes four chains, 2000
warm-up draws, 2000 retained draws and a target acceptance rate of 0.9, and gives the reason: no
model may receive a more or less careful exploration than another. Every fit here uses them, the
auxiliary ones included. They are printed below so the run records what it used.

In [ ]:
#@title 0.4  Control panel { display-mode: "form" }

#@markdown ### Run
MODE = "full"  #@param ["preflight", "smoke", "full"]
STAGE = "auto"  #@param ["auto", "collect", "analyse", "all"]
STANDALONE_FIGURES = False  #@param {type:"boolean"}

#@markdown ### Design constants the report does not fix
TONE_SNR = 1.5  #@param {type:"number"}
RECOVERY_DENOMINATOR = "injected"  #@param ["injected", "true_continuation"]
FIT_ALTERNATIVE_RECOVERY = False  #@param {type:"boolean"}

#@markdown ### Where results are written
USE_DRIVE = True  #@param {type:"boolean"}
RUN_FOLDER = "final_implementation"  #@param {type:"string"}

#@markdown ### Which families this session fits
FIT_MODEL_A = True  #@param {type:"boolean"}
FIT_MODEL_B = True  #@param {type:"boolean"}
FIT_MODEL_C = True  #@param {type:"boolean"}
FIT_MODEL_D1 = True  #@param {type:"boolean"}
FIT_MODEL_D2 = True  #@param {type:"boolean"}

#@markdown ### Collection (Part 2 only)
COLLECTION_DEVICE = "auto"  #@param ["auto", "cpu", "cuda"]
COLLECTION_BATCH_SIZE = 16  #@param {type:"integer"}
SITES_PER_BLOCK = 4  #@param {type:"integer"}
CALIBRATE_COST = True  #@param {type:"boolean"}

# ---------------------------------------------------------------------------------------
if MODE not in ("preflight", "smoke", "full"):
    raise ValueError("MODE must be preflight, smoke or full")
if STAGE not in ("auto", "collect", "analyse", "all"):
    raise ValueError("STAGE must be auto, collect, analyse or all")
if not TONE_SNR > 0:
    raise ValueError("TONE_SNR must be positive")
if RECOVERY_DENOMINATOR not in ("injected", "true_continuation"):
    raise ValueError("RECOVERY_DENOMINATOR must be injected or true_continuation")
IS_FULL = MODE == "full"

# Derived permissions. Nothing expensive can start unless the panel above allows it, so a
# missing artifact in a figures-only session surfaces as a named file rather than as an
# unattended multi-hour job.
ALLOW_COLLECTION = not STANDALONE_FIGURES
# Preflight samples: the log-likelihood parity gate and parameter recovery are fits, and they are
# the plumbing it exists to validate. What it does not do is fit anything reportable, and Part 4.0
# is where it stops.
ALLOW_SAMPLING = not STANDALONE_FIGURES

# -- sampling, exactly as Appendix B prescribes ------------------------------------------
CHAINS, BLOCK_CHAINS = 4, 2
TARGET_ACCEPT = 0.9
DRAWS, TUNE = (2000, 2000) if IS_FULL else (150, 150)
SAMPLE_CORES = max(1, min(BLOCK_CHAINS, os.cpu_count() or 1))

# -- analysis constants, from the document ----------------------------------------------
FS, CTX, PRED, BAND = 512.0, 480, 64, (2.0, 250.0)
NU = 4                        # Student-t degrees of freedom, as written in the .tex
EPS = 0.01                    # the additive stabiliser of Eq. (8)
N_PHASE_BINS = 8              # equal slices of the phase circle, Eq. (11)
PRIOR_SCALE = 0.5             # the primary scale of tab:bayesPriors
PRIOR_FACTORS = (0.5, 1.0, 2.0)   # the sensitivity ladder, as a factor on every prior scale
LOG08, LOG11, LOG12 = math.log(0.8), math.log(1.1), math.log(1.2)
ROPE_SLOPE = 0.1              # |kappa_F - 1| < 0.1
PROB = 0.95                   # the support / refute cutoff
RHAT_MAX, ESS_MIN, MAX_DIVERGENCES = 1.01, 1000, 0
DELTA_F = 1.0                 # the sweep step; the D2 resolution floor
MIN_D2_SITES = 10             # the identification bar of tab:bayesDecisions, per branch
GRID_TOL_HZ = 1e-6            # D1 label tolerance: exact grid membership
GRID_TOL_ALT_HZ = 1.0         # the one-hertz alternative, fitted as a robustness reading
COMB_FLOOR = 1e-2             # pure-tone reference only; the two fitted modes need no floor

# -- operational thresholds, declared here and not attributed to the document ------------
PPC_COVERAGE_MIN = 0.90       # share of strata whose observed statistic must be covered
SENSITIVITY_SPREAD_MAX = 0.10 # admissible excursion of a rule probability over the ladder
RECOVERY_COVERAGE_MIN = 0.80  # share of recovery repeats that must cover the truth
RECOVERY_ROWS = 12000 if IS_FULL else 400
RECOVERY_REPEATS = 2 if IS_FULL else 1
PPC_DRAWS = 400 if IS_FULL else 40
PPC_BATCH = 25
LOO_TARGET_BYTES = 192 * 1024 ** 2   # resident budget for one observation slice

if NUTS_BACKEND != "pymc" and importlib.util.find_spec(NUTS_BACKEND) is None:
    raise ImportError(
        f"NUTS_BACKEND is {NUTS_BACKEND!r} and it is not installed. Re-run Part 0.2, which "
        "installs it, and let the runtime restart once.")

print(f"mode              : {MODE}" + ("" if IS_FULL else "   (NON-REPORTABLE)"))
print(f"sampler           : {NUTS_BACKEND}")
print(f"stage requested   : {STAGE}")
print(f"tone SNR          : {TONE_SNR}")
print(f"collection allowed: {ALLOW_COLLECTION}      sampling allowed: {ALLOW_SAMPLING}")
print(f"recovery ratio    : a_pred / "
      + ("the injected amplitude, as Appendix B defines it"
         if RECOVERY_DENOMINATOR == "injected"
         else "the true continuation's own amplitude at that frequency"))
print(f"sampling          : {CHAINS} chains in blocks of {BLOCK_CHAINS}, {TUNE} warm-up + "
      f"{DRAWS} retained, target_accept={TARGET_ACCEPT}, cores={SAMPLE_CORES}")
print(f"families this run : "
      + ", ".join(n for n, f in (("A", FIT_MODEL_A), ("B", FIT_MODEL_B), ("C", FIT_MODEL_C),
                                 ("D1", FIT_MODEL_D1), ("D2", FIT_MODEL_D2)) if f))

## 0.5, Run namespace

The run writes into one folder named after the mode and the tone amplitude, so two runs that
differ in either cannot share a directory. On Colab that folder is on Drive, which is what makes a
run survive a disconnect; off Colab it is under `chronos/bayesian/_run/`.

In [ ]:
#@title 0.5  Run namespace and output folders
RUN_ID = f"{MODE}_snr{TONE_SNR:g}".replace(".", "p")


def _run_root() -> Path:
    if _on_colab() and USE_DRIVE:
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive
            drive.mount("/content/drive")
        root = Path("/content/drive/MyDrive/patchAliasing") / RUN_FOLDER
    else:
        root = BAYES_DIR / "_run" / RUN_FOLDER
    root.mkdir(parents=True, exist_ok=True)
    return root


RUN_ROOT = _run_root()
CKPT_DIR = RUN_ROOT / RUN_ID
DATA_DIR = CKPT_DIR / "data"
FIG_DIR = CKPT_DIR / "figures"
TAB_DIR = CKPT_DIR / "tables"
for _directory in (CKPT_DIR, DATA_DIR, FIG_DIR, TAB_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

print("run id     :", RUN_ID)
print("checkpoints:", CKPT_DIR)
print("data       :", DATA_DIR)
print("figures    :", FIG_DIR)
print("tables     :", TAB_DIR)

## 0.6, The checkpoint contract

Every artifact is written atomically and recorded in `analysis_manifest.json` with its SHA-256 and
its byte count. `have(name)` is the only question the rest of the notebook asks about a checkpoint,
and it answers by hash rather than by the file's presence, refusing three situations: a file with
no manifest entry, a manifest entry with no file, and a file whose content has moved under a name
already recorded. None of the three is resumed silently.

The run fingerprint covers the seed, the sampler settings, the tone amplitude, every analysis
constant, the support-script hashes and the hash of the code objects of the functions that build
models and datasets. Editing a cell in place therefore invalidates exactly the checkpoints that
depended on it, and a run directory written under a different specification is refused rather than
mixed.

Hashing over Drive's FUSE mount is slow enough to dominate a cell on its own, so hashes are cached
on (size, modification time) beside the manifest.

In [ ]:
#@title 0.6  Atomic writes, hashes and the run manifest
def banner(text: str) -> None:
    print("\n" + "=" * 78)
    print(text)
    print("=" * 78)


def sha256_file(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_hash(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def atomic_json(path, value) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def atomic_parquet(path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def atomic_netcdf(path, idata) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp.nc")
    idata.to_netcdf(tmp, engine="h5netcdf")
    os.replace(tmp, path)


def _code_record(code: types.CodeType) -> dict:
    def constant(value):
        if isinstance(value, types.CodeType):
            return _code_record(value)
        if isinstance(value, (tuple, list)):
            return [constant(item) for item in value]
        return repr(value)
    return {"bytecode": code.co_code.hex(), "names": list(code.co_names),
            "variables": list(code.co_varnames), "free": list(code.co_freevars),
            "constants": [constant(item) for item in code.co_consts]}


def code_fingerprint(functions) -> str:
    """Hash of the code objects currently loaded, so an in-place cell edit is detected.

    Filenames and cell numbers are excluded on purpose: re-running an unchanged cell must not
    invalidate a checkpoint whose inputs and logic are the same.
    """
    return json_hash({f.__name__: _code_record(f.__code__) for f in functions})


HASH_CACHE_PATH = CKPT_DIR / "_hash_cache.json"
_HASH_CACHE = json.loads(HASH_CACHE_PATH.read_text()) if HASH_CACHE_PATH.is_file() else {}
_HASH_CACHE_DIRTY = False


def cached_hash(path) -> str:
    global _HASH_CACHE_DIRTY
    path = Path(path)
    stat = path.stat()
    key = str(path)
    entry = _HASH_CACHE.get(key)
    if entry and entry[0] == stat.st_size and abs(entry[1] - stat.st_mtime) < 1e-6:
        return entry[2]
    digest = sha256_file(path)
    _HASH_CACHE[key] = [stat.st_size, stat.st_mtime, digest]
    _HASH_CACHE_DIRTY = True
    return digest


def flush_hash_cache() -> None:
    global _HASH_CACHE_DIRTY
    if _HASH_CACHE_DIRTY:
        atomic_json(HASH_CACHE_PATH, _HASH_CACHE)
        _HASH_CACHE_DIRTY = False


SOURCE_SHA256 = {name: sha256_file(SUPPORT / name)
                 for name in ("collect.py", "probe_lib.py", "model_loader.py",
                              "checkpointing.py", "bayesian_checks.py")}

ANALYSIS_SPEC = {
    "run_id": RUN_ID, "mode": MODE,
    "models_from": "coursework/deliverable2_v0, stated in full in deliverable3 appendix B",
    "repository_commit": REPO_COMMIT, "seed": SEED, "tone_snr": TONE_SNR,
    "recovery_denominator": RECOVERY_DENOMINATOR,
    "sampling": {"chains": CHAINS, "block_chains": BLOCK_CHAINS, "draws": DRAWS, "tune": TUNE,
                 "target_accept": TARGET_ACCEPT, "backend": NUTS_BACKEND},
    "constants": {"fs": FS, "ctx": CTX, "pred": PRED, "band": list(BAND), "nu": NU, "eps": EPS,
                  "n_phase_bins": N_PHASE_BINS, "prior_scale": PRIOR_SCALE,
                  "prior_factors": list(PRIOR_FACTORS), "grid_tol_hz": GRID_TOL_HZ,
                  "grid_tol_alt_hz": GRID_TOL_ALT_HZ, "delta_f": DELTA_F,
                  "min_d2_sites": MIN_D2_SITES, "comb_floor": COMB_FLOOR},
    "thresholds": {"rhat_max": RHAT_MAX, "ess_min": ESS_MIN,
                   "max_divergences": MAX_DIVERGENCES, "prob": PROB,
                   "ppc_coverage_min": PPC_COVERAGE_MIN,
                   "sensitivity_spread_max": SENSITIVITY_SPREAD_MAX,
                   "recovery_coverage_min": RECOVERY_COVERAGE_MIN,
                   "recovery_rows": RECOVERY_ROWS, "recovery_repeats": RECOVERY_REPEATS,
                   "ppc_draws": PPC_DRAWS},
    "source_sha256": SOURCE_SHA256, "versions": VERSIONS,
}
ANALYSIS_FINGERPRINT = json_hash(ANALYSIS_SPEC)
MANIFEST_PATH = CKPT_DIR / "analysis_manifest.json"

if MANIFEST_PATH.is_file():
    MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if MANIFEST.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError(
            "this run folder was written under a different analysis specification. Change the "
            "tone SNR back, or change RUN_FOLDER, rather than mixing artifacts; the specification "
            f"on disk is in {MANIFEST_PATH.name} under 'analysis_spec'.")
else:
    MANIFEST = {"schema_version": 1, "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "analysis_spec": ANALYSIS_SPEC, "artifacts": {}}
    atomic_json(MANIFEST_PATH, MANIFEST)


def ckpt(name: str) -> Path:
    return CKPT_DIR / name


def _record(name: str) -> None:
    """Record one artifact, merging with whatever is on disk at this instant.

    Two sessions may write different models into one run folder. Re-reading the file immediately
    before merging this single entry means neither session's save erases the other's.
    """
    path = ckpt(name)
    entry = {"sha256": cached_hash(path), "bytes": path.stat().st_size,
             "written": time.strftime("%Y-%m-%dT%H:%M:%S")}
    if MANIFEST_PATH.is_file():
        on_disk = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if on_disk.get("analysis_fingerprint") == ANALYSIS_FINGERPRINT:
            MANIFEST["artifacts"] = {**on_disk.get("artifacts", {}), **MANIFEST["artifacts"]}
    MANIFEST["artifacts"][name] = entry
    atomic_json(MANIFEST_PATH, MANIFEST)
    flush_hash_cache()


def have(name: str) -> bool:
    path = ckpt(name)
    entry = MANIFEST["artifacts"].get(name)
    if path.exists() != (entry is not None):
        raise ValueError(
            f"untracked or missing artifact {name!r}: the file "
            f"{'exists but is not in the manifest' if path.exists() else 'is recorded but absent'}. "
            "Delete the stray file, or use a new RUN_FOLDER.")
    if not path.exists():
        return False
    if cached_hash(path) != entry["sha256"]:
        raise ValueError(f"artifact {name!r} has changed on disk; refusing to resume altered data")
    return True


def save_df(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    atomic_parquet(ckpt(name), frame)
    _record(name)
    print("  checkpoint ->", name)
    return frame


def load_df(name: str) -> pd.DataFrame:
    if not have(name):
        raise FileNotFoundError(missing_artifact_message(name))
    print("  checkpoint <-", name)
    return pd.read_parquet(ckpt(name))


def save_json(value, name: str):
    atomic_json(ckpt(name), value)
    _record(name)
    print("  checkpoint ->", name)
    return value


def load_json(name: str):
    if not have(name):
        raise FileNotFoundError(missing_artifact_message(name))
    return json.loads(ckpt(name).read_text(encoding="utf-8"))


def save_idata(idata, name: str):
    atomic_netcdf(ckpt(name), idata)
    _record(name)
    print("  checkpoint ->", name)
    return idata


def load_idata(name: str):
    if not have(name):
        raise FileNotFoundError(missing_artifact_message(name))
    loaded = az.from_netcdf(ckpt(name), engine="h5netcdf")
    for group in loaded.groups():
        getattr(loaded, group).load()
    print("  checkpoint <-", name)
    return loaded


def missing_artifact_message(name: str) -> str:
    if STANDALONE_FIGURES:
        return (f"{name} is not in {CKPT_DIR}. STANDALONE_FIGURES is on, so nothing is "
                "recomputed. Either point RUN_FOLDER at a finished run, or turn it off.")
    return f"{name} is not in {CKPT_DIR}"


def save_table(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    """A result table: Parquet for the notebook, CSV beside it for reading by eye."""
    save_df(frame, name)
    frame.to_csv(TAB_DIR / (Path(name).stem + ".csv"), index=False)
    return frame


def savefig(fig, name: str) -> None:
    fig.savefig(FIG_DIR / name, bbox_inches="tight")
    print("  figure     ->", name)


print("manifest   :", MANIFEST_PATH.name,
      f"({len(MANIFEST['artifacts'])} artifacts already recorded)")

## 0.7, The frozen fifteen-geometry design

`probe_lib.DELIVERABLE3_MODELS` is the registry of `tab:hfModels`. It is read, never written.

Three properties of the document are recomputed here from the arithmetic rather than assumed, so
that a disagreement between this implementation and the submitted report shows up before any
observation exists:

* fifteen geometries, each with a stride that divides the 480-sample context, which is the
  property that made `p32-s28` ineligible;
* the exclusive-site counts of `tab:branchSites`, 36 stride-only against 102 patch-only
  frequencies, which is the identification budget the whole stride branch rests on;
* the correlation of the two centred configuration covariates, which Appendix B reports as 0.43
  and uses to argue that $\delta_O$ and $\delta_P$ are separately estimable.

Any of the three failing stops the notebook.

In [ ]:
#@title 0.7  The fifteen geometries, checked against the report
import probe_lib as pl

# The tone amplitude is NOT set here. It travels in the collection's own configuration object
# (`collect.Config.tone_snr`, Part 2.1), which is what puts it inside the collection's design
# fingerprint; a module global would not be, and two collections differing only in amplitude
# would then be indistinguishable to the manifest.

FULL_MODELS = list(pl.DELIVERABLE3_MODELS)
TAGS = [pl.model_tag(P, S) for P, S in FULL_MODELS]
GENERATORS = tuple(pl.GENERATORS)

if len(FULL_MODELS) != 15:
    raise ValueError(f"expected the fifteen geometries of tab:hfModels, found {len(FULL_MODELS)}")
if not all(CTX % S == 0 for _, S in FULL_MODELS):
    raise ValueError("a geometry's stride does not divide the 480-sample analysis context")
if (pl.FS, pl.CTX, pl.PRED, tuple(pl.BAND)) != (FS, CTX, PRED, BAND):
    raise ValueError("probe_lib and this notebook disagree on fs, context, horizon or band")


def _exclusive(members, other_spacing) -> list[float]:
    """Members of one comb that are not also members of the other."""
    return [f for f in members
            if pl.comb_distance(np.array([f], float), other_spacing)[0] > 1e-6]


rows = []
for P, S in FULL_MODELS:
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    usable = [f for f, delta in offsets.items() if np.isfinite(delta)]
    phase_sum = sum(len(pl.phases_Sf(f, 10)) for f in usable)
    rows.append({
        "model": pl.model_tag(P, S), "P": P, "S": S, "overlap": (P - S) / P,
        "L_tok": P + S * ((CTX - P) // S),
        "n_lock": len(offsets), "n_usable": len(usable),
        "stride_only": len(_exclusive(pl.stride_locks(S), FS / P)),
        "patch_only": len(_exclusive(pl.patch_nulls(P), FS / S)),
        "phases": phase_sum,
        "triplets": phase_sum * 100 * len(GENERATORS),
        "delta_min": min((offsets[f] for f in usable), default=np.nan),
        "delta_max": max((offsets[f] for f in usable), default=np.nan),
    })
DESIGN = pd.DataFrame(rows)
show(DESIGN, caption="L_tok is the part of the 480-sample context the patch grid spans; "
                     "stride_only and patch_only are the frequencies one grid predicts and "
                     "the other does not; triplets is what the collection will produce.")

BRANCH_TOTALS = {"stride_only": int(DESIGN.stride_only.sum()),
                 "patch_only": int(DESIGN.patch_only.sum())}
if BRANCH_TOTALS != {"stride_only": 36, "patch_only": 102}:
    raise ValueError(f"exclusive-site counts {BRANCH_TOTALS} do not reproduce tab:branchSites "
                     "(36 stride-only, 102 patch-only)")

# Centred configuration covariates of Eq. (9), averaged over the fifteen runs as Appendix B
# states, so beta_bar is the baseline at the average configuration rather than an extrapolation.
GEOM = DESIGN[["model", "P", "S", "overlap"]].copy()
GEOM["x_overlap"] = (GEOM.overlap - GEOM.overlap.mean()) / 0.5
GEOM["x_logP"] = np.log(GEOM.P) - np.log(GEOM.P).mean()
COVARIATE_CORR = float(np.corrcoef(GEOM.x_overlap, GEOM.x_logP)[0, 1])
if abs(COVARIATE_CORR - 0.43) > 0.01:
    raise ValueError(f"covariate correlation {COVARIATE_CORR:.3f} does not reproduce the 0.43 "
                     "Appendix B reports; the centring or the population differs")

note(f"<b>Design checks against the report.</b> Fifteen geometries, every stride dividing 480. "
     f"Exclusive sites {BRANCH_TOTALS['stride_only']} stride-only and "
     f"{BRANCH_TOTALS['patch_only']} patch-only, reproducing <code>tab:branchSites</code>. "
     f"Centred overlap spans [{GEOM.x_overlap.min():+.2f}, {GEOM.x_overlap.max():+.2f}] and "
     f"corr(x_overlap, x_logP) = {COVARIATE_CORR:+.3f}, reproducing the 0.43 of Appendix B.",
     "good")
show(GEOM, caption="the two centred configuration covariates of Eq. (9)")

TOTAL_TRIPLETS = int(DESIGN.triplets.sum())
print(f"\nexact full design: {TOTAL_TRIPLETS:,} triplets = {TOTAL_TRIPLETS * 3:,} arms "
      f"at 100 backgrounds per generator and {len(GENERATORS)} generators")
note("Nine of the fifteen geometries yield no stride-only frequency, because there every stride "
     "site is also a patch site. The stride branch is carried by the six runs with S not "
     "dividing P, and those same six are the geometries whose analysis context is truncated. "
     "That is fixed by the choice of (P, S) before anything is measured, and it bounds what "
     "theta_S and kappa_S can be asked to settle.", "warn")

## 0.8, The preregistered priors and decision rules

Both tables are transcribed from the document into code, so the analysis and the report cannot
drift apart. Every probabilistic rule's prior probability is then recomputed analytically and
checked against the **Prior** column of `tab:bayesDecisions`. That column is the calibration
reference the document itself offers, so reproducing it tests whether the priors implemented here
are the ones that were preregistered. A mismatch stops the notebook.

One difference from Deliverable 2 is an erratum rather than a change of plan. Its
`tab:bayesDecisions` gives the mitigation rule as $\Pr(\delta_O<0\mid D)\ge 0.95$; Deliverable 3
gives $\Pr(\delta_O>0\mid D)\ge 0.95$, and Appendix B states why: the response is the contrast of
Eq. (8), which rises towards zero as recovery at a candidate approaches recovery at its controls,
so an overlap that mitigates raises it. The positive sign is what is implemented, and the
correction is recorded in the table below rather than applied silently.

In [ ]:
#@title 0.8  Priors and decision rules, checked against tab:bayesDecisions
PRIOR_SPEC = pd.DataFrame([
    ("A, C", "beta_bar",     f"StudentT(4, 0, {PRIOR_SCALE})",  "population phase-lock effect, on the log-recovery scale"),
    ("A",    "delta_O",      f"StudentT(4, 0, {PRIOR_SCALE})",  "slope in the centred overlap; one unit is 0.5 of O"),
    ("A",    "delta_P",      f"StudentT(4, 0, {PRIOR_SCALE})",  "slope in the centred log patch size"),
    ("A, C", "tau",          f"HalfStudentT(4, {PRIOR_SCALE})", "spread of the configuration effects"),
    ("A, C", "sigma_harm",   f"HalfStudentT(4, {PRIOR_SCALE})", "spread across lock harmonics (sum-to-zero)"),
    ("A, C", "sigma_bg",     f"HalfStudentT(4, {PRIOR_SCALE})", "spread across background realisations (sum-to-zero)"),
    ("A, C", "sigma",        f"HalfStudentT(4, {PRIOR_SCALE})", "residual scale of the contrast"),
    ("C",    "sigma_phase",  "HalfNormal(0.25)",                "spread of the eight per-phase offsets (sum-to-zero)"),
    ("B",    "alpha_0",      "Normal(log mean L, 1)",           "baseline log codelength"),
    ("B",    "theta_lock",   "Normal(0, 0.5)",                  "log codelength expansion at a lock"),
    ("B",    "r",            "Gamma(2, 0.1)",                   "Gamma shape, carrying the dispersion"),
    ("B",    "sigma_st",     "HalfNormal(1)",                   "spread of the probe-stage offsets (sum-to-zero)"),
    ("B",    "sigma_geo",    "HalfNormal(1)",                   "spread of the geometry offsets (sum-to-zero)"),
    ("D1",   "alpha_g",      "Normal(0, 1)",                    "per-geometry off-grid level of log z"),
    ("D1",   "theta_S",      "Normal(0, 1)",                    "dip depth on the stride grid"),
    ("D1",   "theta_P",      "Normal(0, 1)",                    "dip depth on the patch grid"),
    ("D1",   "sigma",        "HalfNormal(1)",                   "residual scale of log z"),
    ("D2",   "kappa",        "Normal(0, 1)",                    "measured spacing per unit of predicted spacing"),
    ("D2",   "sigma_F",      "HalfNormal(5) Hz",                "scatter beyond the sweep-resolution floor"),
], columns=["model", "parameter", "prior", "role"])
show(PRIOR_SPEC, caption="tab:bayesPriors, transcribed. Every effect prior is centred on no "
                         "effect; the sum-to-zero marks are the constraint of Appendix B.")

DECISION_SPEC = pd.DataFrame([
    ("H1_behavioural", "supported", "A", "Pr(beta_bar < log 0.8) >= 0.95",
     float(stats.t.cdf(LOG08, df=NU, scale=PRIOR_SCALE)), 0.34,
     "recovery at a candidate falls to at most four fifths of its controls"),
    ("H1_behavioural", "refuted", "A", "Pr(|beta_bar| < log 1.1) >= 0.95",
     float(stats.t.cdf(LOG11, df=NU, scale=PRIOR_SCALE)
           - stats.t.cdf(-LOG11, df=NU, scale=PRIOR_SCALE)), 0.14,
     "any effect on that recovery is under 10 per cent"),
    ("H1_representational", "supported", "B", "Pr(theta_lock > log 1.2) >= 0.95",
     float(stats.norm.sf(LOG12, scale=0.5)), 0.36, "at least 20 per cent more code"),
    ("H1_representational", "refuted", "B", "Pr(|theta_lock| < log 1.1) >= 0.95",
     float(2 * stats.norm.cdf(LOG11, scale=0.5) - 1), 0.15,
     "any expansion is under 10 per cent"),
    ("H2", "supported", "C", "Pr(sigma_phase < log 1.1) >= 0.95",
     float(stats.halfnorm.cdf(LOG11, scale=0.25)), 0.30,
     "phase moves the deficit by under 10 per cent"),
    ("H3a_H3b", "supported", "D2", "Pr(|kappa_F - 1| < 0.1) >= 0.95",
     float(stats.norm.cdf(1 + ROPE_SLOPE) - stats.norm.cdf(1 - ROPE_SLOPE)), 0.05,
     "an identified branch tracks its own parameter"),
    ("M1", "supported", "A'", "Pr(delta_O > 0) >= 0.95 and the overlap term wins LOO",
     0.5, 0.50, "overlap is associated with a smaller loss"),
], columns=["claim", "leg", "model", "rule", "prior_probability", "reported_in_tex", "reads_as"])
DECISION_SPEC["reproduces_tex"] = ((DECISION_SPEC.prior_probability
                                    - DECISION_SPEC.reported_in_tex).abs() < 0.01)
show(DECISION_SPEC.drop(columns=["reads_as"]),
     caption="prior_probability is recomputed here; reported_in_tex is the Prior column of "
             "tab:bayesDecisions.")
if not DECISION_SPEC.reproduces_tex.all():
    failed = DECISION_SPEC.loc[~DECISION_SPEC.reproduces_tex, "claim"].tolist()
    raise ValueError(f"the prior probability of {failed} does not reproduce tab:bayesDecisions; "
                     "the priors implemented here are not the ones preregistered")

RULES_WITHOUT_PRIOR = pd.DataFrame([
    ("H3_location", "D1", "the two-label fit wins LOO and theta_S, theta_P < 0",
     "the table records no prior probability for a rule that is partly a model comparison"),
    ("D2 identification", "D2", f"at least {MIN_D2_SITES} unambiguous sites for branch F",
     "a design property, checked before either branch rule is read"),
], columns=["claim", "model", "rule", "why no prior probability"])
show(RULES_WITHOUT_PRIOR)

note("<b>Prior calibration.</b> Every probabilistic rule reproduces the Prior column of "
     "<code>tab:bayesDecisions</code> to better than 0.01, so the priors fitted below are the "
     "ones that were preregistered. A posterior counts as evidence only in so far as it has "
     "moved away from these values.", "good")
note("<b>Erratum carried.</b> Deliverable 2 prints the M1 rule with a negative sign. The "
     "response of Eq. (8) rises as recovery at a candidate approaches its controls, so a "
     "mitigation raises it: the rule is Pr(delta_O &gt; 0) and that is what is implemented, as "
     "Deliverable 3 and Appendix B have it.", "warn")

save_table(PRIOR_SPEC, "00_prior_spec.parquet")
save_table(DECISION_SPEC, "00_decision_spec.parquet")

# Every place this implementation interprets a clause, corrects one, or departs from one, in one
# list. It is printed here rather than left to be discovered, saved with the run, and repeated in
# the companion report. A reader who disagrees with any row can find the control that changes it.
SPECIFICATION_NOTES = pd.DataFrame([
    ("erratum", "M1 sign",
     "Deliverable 2 prints Pr(delta_O < 0) >= 0.95",
     "Pr(delta_O > 0) >= 0.95, as Deliverable 3 and Appendix B have it: the contrast rises as the "
     "loss shrinks, so a mitigation raises it"),
    ("interpretation", "sum-to-zero",
     "Appendix B: each set of offsets is constrained to sum to zero",
     "applied to the offset sets u_harm, u_bg, u_phase, u_stage and u_geometry, and not to "
     "beta_c, which is the configuration level itself and carries the M1 regression"),
    ("choice", "the denominator of R",
     "Appendix B defines R against the amplitude INJECTED; the shared estimator returns the "
     "amplitude of the true continuation at that frequency",
     f"both are collected and {RECOVERY_DENOMINATOR!r} is fitted; the control is "
     "RECOVERY_DENOMINATOR and it enters the run fingerprint"),
    ("choice", "D1 grid tolerance",
     "Appendix B: on a grid when the distance is within tolerance, with no number given",
     f"exact membership to {GRID_TOL_HZ:g} Hz as primary, because the sweep grid contains the "
     f"exact sites; {GRID_TOL_ALT_HZ:g} Hz fitted as a declared robustness reading"),
    ("departure", "D1 signal modes",
     "Appendix B: all three modes are fitted",
     "all three are fitted and reported; the verdict is read from the two generator modes, "
     "because on a pure sinusoid the response depends on the floor and a stratum of identical "
     "values has no dispersion for a predictive check"),
    ("generalisation", "the sensitivity ladder",
     "Deliverable 3: every effect prior is varied over {0.25, 0.5, 1.0}",
     "applied as a factor of 0.5, 1 and 2 on every prior scale, which reproduces those three for "
     "the priors tabulated at 0.5 and extends the rule to the families tabulated elsewhere"),
    ("addition", "D2 readings must agree",
     "tab:bayesDecisions gates D2 on ten unambiguous sites only",
     "an operational gate is added: the preregistered fit and the fit restricted to geometries "
     "whose lowest detected site is the fundamental must reach the same decision"),
    ("addition", "operational thresholds",
     "the report names the checks but not their thresholds",
     f"predictive coverage {PPC_COVERAGE_MIN:.0%}, sensitivity band {SENSITIVITY_SPREAD_MAX}, "
     f"recovery coverage and discrimination {RECOVERY_COVERAGE_MIN:.0%}, all declared here and "
     "recorded in the manifest"),
], columns=["kind", "subject", "what the document says", "what is implemented"])
save_table(SPECIFICATION_NOTES, "00_specification_notes.parquet")
show(SPECIFICATION_NOTES, caption="every interpretation, correction and departure, declared in "
                                  "one place and saved with the run")

## 0.9, Progress, stage resolution and the run plan

A full run is long enough that a silent cell cannot be told apart from a hung one, so every
expensive stage is a timed task. It announces when it starts, how long it took, how much of the
declared plan is finished and a projection of what remains. The timings persist, so a resumed
session reports the cumulative cost of the whole run rather than of the current session, and a
stage that reloads from a checkpoint records a near-zero time, which is itself the signal that it
was reused.

The weights are relative costs declared before the run; the elapsed figures are measured. The
remaining-time figure is therefore a projection of measured tasks onto declared weights, and it
is labelled as such rather than presented as a measurement.

The run plan below is the inventory the notebook starts from: one row per stage, `done` when its
artifact is present and hashes correctly, `todo` when it will be computed.

In [ ]:
#@title 0.9  Timed tasks, stage resolution and the run plan
TIMINGS_FILE = "00_timings.json"
_TIMINGS = load_json(TIMINGS_FILE) if have(TIMINGS_FILE) else {}

PLAN_WEIGHTS = {
    "part1_prior_predictive": 2,
    "part2_collection": 120,
    "part3_parity": 3,
    "part3_recovery": 30,
    "fit_A_both": 12, "fit_A_overlap": 12, "fit_A_patch": 12, "fit_A_none": 12,
    "fit_B_adjusted": 2, "fit_B_unadjusted": 2,
    "fit_C_phase": 14, "fit_C_nophase": 14,
    "fit_D1": 8, "fit_D1_alt_tol": 3, "fit_D2": 2,
    "part5_loo": 20, "part5_ppc": 12, "part5_sensitivity": 60,
    "part6_figures": 4,
}


class Progress:
    """A declared plan of timed tasks, with measured elapsed time and a projected remainder."""

    def __init__(self, weights: dict[str, float]):
        self.weights = dict(weights)

    def report(self) -> str:
        total = sum(self.weights.values())
        done = sum(self.weights[name] for name in _TIMINGS if name in self.weights)
        spent = sum(seconds for name, seconds in _TIMINGS.items() if name in self.weights)
        if done <= 0 or spent <= 0:
            return f"{len(_TIMINGS)}/{len(self.weights)} tasks recorded, no projection yet"
        remaining = max(0.0, (total - done) * (spent / done))
        return (f"{100 * done / total:.0f}% of the declared plan, {spent / 60:.0f} min spent, "
                f"~{remaining / 60:.0f} min projected")

    def task(self, name: str):
        if name not in self.weights:
            raise KeyError(f"timed task {name!r} is not in PLAN_WEIGHTS")
        return _Task(self, name)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame([{"task": name, "weight": weight,
                              "seconds": _TIMINGS.get(name, np.nan),
                              "state": "done" if name in _TIMINGS else "todo"}
                             for name, weight in self.weights.items()])


class _Task:
    def __init__(self, progress: Progress, name: str):
        self.progress, self.name = progress, name

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n>>> {self.name}: start   ({self.progress.report()})")
        sys.stdout.flush()
        return self

    def __exit__(self, exc_type, exc, tb):
        seconds = time.time() - self.t0
        if exc_type is None:
            _TIMINGS[self.name] = seconds
            atomic_json(ckpt(TIMINGS_FILE), _TIMINGS)
            _record(TIMINGS_FILE)
            print(f"<<< {self.name}: done in {seconds:.0f}s   ({self.progress.report()})")
        else:
            print(f"<<< {self.name}: FAILED after {seconds:.0f}s")
        sys.stdout.flush()
        return False


PROGRESS = Progress(PLAN_WEIGHTS)


def collection_status() -> tuple[str, list[str]]:
    """Whether DATA_DIR already holds a complete fifteen-geometry collection."""
    manifest = DATA_DIR / "collection_manifest.json"
    if not manifest.is_file():
        return "absent", []
    payload = json.loads(manifest.read_text(encoding="utf-8"))
    done = sorted({key.split("__", 1)[1] for key in payload.get("shards", {})})
    return ("complete" if payload.get("status") == "complete" else "partial"), done


COLLECTION_STATE, COLLECTED_TAGS = collection_status()
if STAGE == "auto":
    if COLLECTION_STATE == "complete":
        STAGE = "analyse"
    elif STANDALONE_FIGURES:
        STAGE = "analyse"
    elif HAVE_TORCH:
        STAGE = "all"
    else:
        raise RuntimeError(
            f"the collection in {DATA_DIR} is {COLLECTION_STATE} "
            f"({len(COLLECTED_TAGS)}/15 geometries) and this runtime cannot load Chronos, so it "
            "can neither finish the collection nor analyse it. Run the notebook once in a "
            "runtime where Part 0.2 reports that Chronos can be loaded.")

ARTIFACT_PLAN = [
    ("0  priors", "01_prior_predictive.parquet"),
    ("2  contrasts derived", "02_contrasts.parquet"),
    ("3  log-likelihood parity", "03_parity.parquet"),
    ("3  parameter recovery", "03_recovery.parquet"),
    ("4  fit A (both covariates)", "04_A_both.nc"),
    ("4  fit B (adjusted)", "04_B_adjusted.nc"),
    ("4  fit C (with phase)", "04_C_phase.nc"),
    ("4  fit D1 (tsmixup, both grids)", "04_D1_tsmixup_both.nc"),
    ("4  fit D2 (stride, f1)", "04_D2_stride_f1.nc"),
    ("5  convergence", "05_convergence.parquet"),
    ("5  LOO comparisons", "05_loo.parquet"),
    ("5  posterior predictive", "05_ppc.parquet"),
    ("5  prior sensitivity", "05_sensitivity.parquet"),
    ("5  verdicts", "05_verdicts.parquet"),
]


def run_plan() -> pd.DataFrame:
    state, tags = collection_status()
    rows = [{"stage": "2  collection", "artifact": "data/collection_manifest.json",
             "state": f"{state} ({len(tags)}/15)"}]
    rows += [{"stage": stage, "artifact": name, "state": "done" if have(name) else "todo"}
             for stage, name in ARTIFACT_PLAN]
    return pd.DataFrame(rows)


banner(f"RUN PLAN, stage resolved to {STAGE!r}")
show(run_plan(), caption=f"run folder {CKPT_DIR}")
show(PROGRESS.frame(), caption="declared relative weights; seconds are measured and persist "
                               "across sessions in 00_timings.json")
if STANDALONE_FIGURES:
    note("<b>Figures-only session.</b> Collection and sampling are refused. Every stage will "
         "reload from its checkpoint, and a missing one is reported by name.", "info")

---
# Part 1, Priors, before any observation exists

A prior is not weakly informative because it is labelled weakly informative. It is weakly
informative if the data it predicts are plausible and if it does not quietly assert the
conclusion. Part 0.8 has already checked that these priors reproduce the prior probabilities the
document publishes; this part asks the complementary question, what they imply about observable
quantities, by pushing each one through its own likelihood.

Nothing here needs Chronos. Which geometries, which candidate frequencies, which phases and which
backgrounds the experiment will visit is fixed by the design in advance, and only the responses are
unknown. That is exactly what makes a genuine prior predictive check possible rather than a
retrospective one, and it is why this part runs before Part 2 rather than after it.

The offsets are simulated under the **constrained** prior, drawn and then centred, because that is
the prior Part 3 fits. Simulating unconstrained offsets would put a group mean of order
$\sigma_u/\sqrt{n}$ into data that the fitted model has nowhere to absorb except in the intercept,
and the check would then be of a model nobody fits.

## 1.1, The design skeleton

One row per triplet the collection will produce, at a reduced number of backgrounds. The full
design uses 100 realisations per generator; six are enough to carry every geometry, candidate
frequency and phase, which are the axes the prior predictive needs.

In [ ]:
#@title 1.1  The rows the collection will produce
banner("PART 1, PRIORS")

SKELETON_BG = 6          # the full design uses 100 per generator; the axes are what matter here


def design_skeleton(n_bg: int = SKELETON_BG, n_phase: int = 10) -> pd.DataFrame:
    rows = []
    for P, S in FULL_MODELS:
        offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
        for f_lock in [f for f, delta in offsets.items() if np.isfinite(delta)]:
            for generator in GENERATORS:
                for bg_id in range(n_bg):
                    for phase_index, phase in enumerate(pl.phases_Sf(f_lock, n_phase)):
                        rows.append({"model": pl.model_tag(P, S), "P": P, "S": S,
                                     "overlap": (P - S) / P, "f_lock": round(float(f_lock), 6),
                                     "family": pl.lock_family(f_lock, P, S),
                                     "generator": generator, "bg_id": bg_id,
                                     "phase_idx": phase_index, "phase": float(phase)})
    return pd.DataFrame(rows)


SKELETON = design_skeleton()
print(f"prior-predictive skeleton: {len(SKELETON):,} triplets at {SKELETON_BG} backgrounds "
      f"per generator")
print(f"the full design will be  : {TOTAL_TRIPLETS:,} triplets at 100 per generator")
show(SKELETON.groupby("model", as_index=False).agg(
        triplets=("f_lock", "size"), candidates=("f_lock", "nunique"),
        phases=("phase_idx", "nunique")),
     caption="candidate frequencies and distinct phases per geometry")
print("\ndistinct harmonics over the whole design:", SKELETON.f_lock.nunique())

## 1.2, Models A and C, the paired contrast

The contrast $d$ lives on a log-ratio scale, so $\bar\beta$ is read as $e^{\bar\beta}$: the
multiplicative change in forecast amplitude recovery at a candidate relative to its controls.
Three things are checked.

* The prior is **symmetric**, giving attenuation and amplification the same mass, so finding
  attenuation cannot be an artefact of the prior.
* It **reaches far enough**. A scale of 0.5 puts $e^{\pm 0.5}\approx 1.65$ at one scale unit and
  the heavy $t_4$ tails admit much larger effects, so a genuinely strong effect will not be shrunk
  away.
* The simulated contrasts are of the order contrasts could physically take, recovery ratios being
  bounded below by zero and rarely much above a few.

In [ ]:
#@title 1.2  Prior predictive for the contrast (Models A and C)
def _zero_sum(draws: np.ndarray, axis: int = -1) -> np.ndarray:
    """Centre a set of group offsets, which is the projection ZeroSumNormal applies."""
    return draws - draws.mean(axis=axis, keepdims=True)


def simulate_contrast_prior(skeleton: pd.DataFrame, scale: float, n_draw: int = 20000,
                            seed: int = SEED) -> dict:
    """Draw Model A's parameters from the prior and push them through its likelihood."""
    rng = np.random.default_rng(seed)
    cfg_codes, cfg_levels = pd.factorize(skeleton["model"])
    harm_codes, harm_levels = pd.factorize(skeleton["f_lock"])
    bg_codes, bg_levels = pd.factorize(skeleton["generator"] + "#"
                                       + skeleton["bg_id"].astype(str))
    x_overlap = GEOM.set_index("model").loc[list(cfg_levels), "x_overlap"].to_numpy()
    x_logP = GEOM.set_index("model").loc[list(cfg_levels), "x_logP"].to_numpy()

    t4 = lambda size: stats.t.rvs(NU, scale=scale, size=size,
                                  random_state=rng.integers(1 << 31))
    beta_bar, delta_O, delta_P = t4(n_draw), t4(n_draw), t4(n_draw)
    tau, sigma_h, sigma_b, sigma = (np.abs(t4(n_draw)) for _ in range(4))

    # one random design row per draw gives the marginal prior predictive of a single observation
    row = rng.integers(0, len(skeleton), n_draw)
    harm = _zero_sum(rng.normal(size=(n_draw, len(harm_levels)))) * sigma_h[:, None]
    bg = _zero_sum(rng.normal(size=(n_draw, len(bg_levels)))) * sigma_b[:, None]
    beta_c = (beta_bar + delta_O * x_overlap[cfg_codes[row]]
              + delta_P * x_logP[cfg_codes[row]] + tau * rng.normal(size=n_draw))
    mu = (beta_c + harm[np.arange(n_draw), harm_codes[row]]
          + bg[np.arange(n_draw), bg_codes[row]])
    d = mu + sigma * stats.t.rvs(NU, size=n_draw, random_state=rng.integers(1 << 31))
    return {"beta_bar": beta_bar, "delta_O": delta_O, "delta_P": delta_P, "d": d}


PRIOR_SIMS = {}
figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
for factor in PRIOR_FACTORS:
    scale = PRIOR_SCALE * factor
    sim = simulate_contrast_prior(SKELETON, scale)
    PRIOR_SIMS[scale] = sim
    label = f"scale {scale:g}" + ("  (primary)" if factor == 1.0 else "")
    style = dict(lw=2.0 if factor == 1.0 else 1.2, histtype="step", density=True,
                 color=INK["primary"] if factor == 1.0 else INK["muted"])
    axes[0].hist(sim["beta_bar"], bins=90, range=(-3, 3), label=label, **style)
    axes[1].hist(np.clip(np.exp(sim["beta_bar"]), 0, 4), bins=90, label=label, **style)
    axes[2].hist(np.clip(sim["d"], -6, 6), bins=90, label=label, **style)

axes[0].axvline(LOG08, color=INK["bad"], ls="--", lw=1, label="log 0.8, the H1 threshold")
axes[0].axvspan(-LOG11, LOG11, color=INK["good"], alpha=0.12, label="the refutation ROPE")
axes[0].set(title=r"prior on $\bar\beta$", xlabel=r"$\bar\beta$  (log recovery ratio)")
axes[1].axvline(1.0, color=INK["line"], lw=0.9)
axes[1].axvline(0.8, color=INK["bad"], ls="--", lw=1)
axes[1].set(title=r"implied recovery ratio $e^{\bar\beta}$", xlabel="ratio")
axes[2].set(title="prior predictive contrast $d$ on the real design", xlabel="$d$")
for axis in axes:
    axis.legend(fontsize=7)
figure.tight_layout()
savefig(figure, "P1_prior_contrast.png")
plt.show()

primary = PRIOR_SIMS[PRIOR_SCALE]
PRIOR_A_SUMMARY = {
    "P(beta_bar < 0)": float(np.mean(primary["beta_bar"] < 0)),
    "P(attenuation >= 20%)": float(np.mean(primary["beta_bar"] < LOG08)),
    "P(attenuation >= 50%)": float(np.mean(primary["beta_bar"] < math.log(0.5))),
    "P(delta_O > 0)": float(np.mean(primary["delta_O"] > 0)),
    "ratio 2.5%": float(np.exp(np.quantile(primary["beta_bar"], 0.025))),
    "ratio 97.5%": float(np.exp(np.quantile(primary["beta_bar"], 0.975))),
    "d 2.5%": float(np.quantile(primary["d"], 0.025)),
    "d 97.5%": float(np.quantile(primary["d"], 0.975)),
}
show(pd.DataFrame([PRIOR_A_SUMMARY]).T.rename(columns={0: "value"}).reset_index()
     .rename(columns={"index": "quantity"}),
     caption=f"under the primary prior, scale {PRIOR_SCALE}")
note("A prior probability of 0.5 that the effect is negative is what makes attenuation and "
     "amplification equally available, and a reachable 50 per cent attenuation is what stops a "
     "strong real effect from being shrunk away. Both hold.", "good")

## 1.3, The phase term of Model C

Model C cuts the phase circle into eight equal slices and gives each its own offset. The estimand
$\sigma_\phi$ is the spread of those offsets: how far the deficit moves as the signal slides
through its cycle. Its $\mathrm{Half}\mathcal N(0,0.25)$ prior puts mass on both sides of the
equivalence boundary $\log 1.1$, so the H2 verdict is decided by the data rather than by the prior,
and Part 0.8 has already confirmed that the mass below the boundary is the 0.30 the document
reports.

The offsets are drawn sum-to-zero, which is how they are fitted. A set of eight offsets that did
not sum to zero would be partly absorbable into the intercept, and the spread being estimated
would not be the spread the model reports.

In [ ]:
#@title 1.3  Prior predictive for the phase spread (Model C)
SIGMA_PHASE_PRIOR = np.abs(RNG.normal(0, 0.25, 40000))

figure, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(SIGMA_PHASE_PRIOR, bins=90, density=True, color=INK["primary"], alpha=0.85)
axes[0].axvline(LOG11, color=INK["bad"], ls="--",
                label=f"equivalence edge log 1.1 = {LOG11:.3f}")
axes[0].set(title=r"prior on the phase spread $\sigma_\phi$", xlabel=r"$\sigma_\phi$")
axes[0].legend(fontsize=8)

for _ in range(30):
    offsets = _zero_sum(RNG.normal(size=N_PHASE_BINS)) * RNG.choice(SIGMA_PHASE_PRIOR)
    axes[1].plot(np.arange(N_PHASE_BINS), offsets, marker="o", ms=3, alpha=0.35, lw=1,
                 color=INK["primary"])
axes[1].axhline(0, color=INK["line"], lw=0.8)
axes[1].set(title="prior-plausible per-phase offsets (sum-to-zero)", xlabel="phase slice",
            ylabel="offset")
figure.tight_layout()
savefig(figure, "P1_prior_phase.png")
plt.show()

P_SIGMA_PHASE_IN_ROPE = float(np.mean(SIGMA_PHASE_PRIOR < LOG11))
print(f"P(sigma_phase < log 1.1) a priori = {P_SIGMA_PHASE_IN_ROPE:.3f}")
note("Mass on both sides of the boundary is what makes H2 decidable. A prior concentrated below "
     "it would support H2 before any data were seen, and the notebook would be reporting its own "
     "assumption.", "good")

## 1.4, Model B, the codelength

The response is a prequential codelength in bits: strictly positive, right-skewed, with a variance
that grows with its mean. That is what the Gamma likelihood with a log link is for, and it is why
$\theta_{\mathrm{lock}}$ is read multiplicatively. The absolute bit scale is supplied by the
observed table through $\log \bar L$, so the prior predictive is drawn relative to that baseline
rather than inventing one.

In [ ]:
#@title 1.4  Prior predictive for the codelength (Model B)
_n = 20000
alpha_offset = RNG.normal(0.0, 1.0, _n)          # alpha_0 ~ Normal(log mean L, 1), in units of L
theta_prior = RNG.normal(0.0, 0.5, _n)
r_prior = RNG.gamma(2.0, 1 / 0.1, _n)
relative_mean = np.exp(alpha_offset + theta_prior)
relative_draw = RNG.gamma(r_prior, relative_mean / r_prior)

figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))
axes[0].hist(np.exp(theta_prior), bins=90, range=(0, 4), density=True,
             color=INK["accent"], alpha=0.85)
axes[0].axvline(1, color=INK["line"], lw=0.9)
axes[0].axvline(1.2, color=INK["bad"], ls="--", label="+20%, the H1 threshold")
axes[0].axvspan(1 / 1.1, 1.1, color=INK["good"], alpha=0.12, label="the refutation ROPE")
axes[0].set(title=r"implied codelength ratio $e^{\theta_{lock}}$", xlabel="ratio")
axes[0].legend(fontsize=7)
axes[1].hist(r_prior, bins=90, density=True, color=INK["accent"], alpha=0.85)
axes[1].set(title="prior on the Gamma shape $r$", xlabel="$r$")
axes[2].hist(np.clip(relative_draw, 0, 20), bins=90, density=True,
             color=INK["accent"], alpha=0.85)
axes[2].set(title=r"prior predictive locked codelength $L/\bar L$", xlabel="ratio to the baseline")
figure.tight_layout()
savefig(figure, "P1_prior_codelength.png")
plt.show()

PRIOR_B_SUMMARY = {"P(theta_lock > 0)": float(np.mean(theta_prior > 0)),
                   "P(expansion >= 20%)": float(np.mean(theta_prior > LOG12)),
                   "P(|theta| < log 1.1)": float(np.mean(np.abs(theta_prior) < LOG11))}
print({k: round(v, 3) for k, v in PRIOR_B_SUMMARY.items()})
note("A Gamma shape prior with mean 20 and a long right tail admits both a nearly deterministic "
     "codelength and a highly dispersed one, so the dispersion is estimated rather than assumed.",
     "info")

## 1.5, Models D1 and D2

D1 asks whether the token dispersion is lower on a predicted grid than off it. Its
$\mathcal N(0,1)$ priors on $\theta_S$ and $\theta_P$ are symmetric, so a dip is not assumed.

D2 asks whether the measured spacing follows the predicted one, and its prior on $\kappa_F$ is
centred on **zero**, that is on no relationship at all, so the hypothesis $\kappa_F=1$ has to be
earned from the data. That choice has a consequence worth showing rather than hiding: a symmetric
Normal prior on a slope multiplying a positive spacing puts prior mass on negative frequencies.
The third panel draws it. The prior is kept as the document fixes it, and the unphysical region is
reported as a property of that choice.

In [ ]:
#@title 1.5  Prior predictive for the two H3 models
figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))

theta_S_prior = RNG.normal(0, 1, 20000)
axes[0].hist(theta_S_prior, bins=80, density=True, color=INK["primary"], alpha=0.85)
axes[0].axvline(0, color=INK["line"], lw=0.9)
axes[0].fill_betweenx([0, 0.42], -4, 0, color=INK["good"], alpha=0.10)
axes[0].set(title=r"D1 prior on $\theta_S$ (H3 predicts $<0$)", xlabel=r"$\theta_S$")

kappa_prior = RNG.normal(0, 1, 20000)
axes[1].hist(kappa_prior, bins=80, density=True, color=INK["good"], alpha=0.85)
axes[1].axvline(1.0, color=INK["bad"], ls="--", label=r"$\kappa_F=1$, the claim")
axes[1].axvspan(1 - ROPE_SLOPE, 1 + ROPE_SLOPE, color=INK["bad"], alpha=0.12)
axes[1].axvline(0.0, color=INK["line"], lw=0.8, label="prior centre: no relationship")
axes[1].set(title=r"D2 prior on $\kappa_F$", xlabel=r"$\kappa_F$")
axes[1].legend(fontsize=7)

predicted = RNG.choice([FS / S for _, S in FULL_MODELS] + [FS / P for P, _ in FULL_MODELS],
                       size=20000)
sigma_F_prior = np.abs(RNG.normal(0, 5, 20000))
f1_prior = RNG.normal(kappa_prior * predicted,
                      np.sqrt(sigma_F_prior ** 2 + DELTA_F ** 2))
axes[2].hist(np.clip(f1_prior, -120, 260), bins=100, density=True,
             color=INK["accent"], alpha=0.8)
axes[2].axvspan(*BAND, color=INK["muted"], alpha=0.16, label="the analysed band")
axes[2].axvline(0, color=INK["bad"], ls="--", lw=1)
axes[2].set(title=r"D2 prior predictive $\hat f_1$ [Hz]", xlabel="Hz")
axes[2].legend(fontsize=7)

figure.tight_layout()
savefig(figure, "P1_prior_H3.png")
plt.show()

P_KAPPA_IN_ROPE = float(np.mean(np.abs(kappa_prior - 1) < ROPE_SLOPE))
P_F1_NEGATIVE = float(np.mean(f1_prior < 0))
print(f"P(|kappa_F - 1| < {ROPE_SLOPE}) a priori = {P_KAPPA_IN_ROPE:.3f}")
print(f"P(prior-predictive f1 < 0)              = {P_F1_NEGATIVE:.3f}")
note(f"About {100 * P_F1_NEGATIVE:.0f} per cent of the D2 prior predictive falls on negative "
     "frequencies, which no measurement can produce. The prior is the one the document fixes and "
     "it is kept; this is stated so the reader knows the support is wider than the measurement, "
     "not narrower.", "warn")

## 1.6, Checkpoint

The prior specification and its predictive summaries are stored, so the reported analysis can be
traced back to priors fixed before any observation existed.

In [ ]:
#@title 1.6  Checkpoint the prior predictive
with PROGRESS.task("part1_prior_predictive"):
    PRIOR_PREDICTIVE = pd.DataFrame([
        {"quantity": key, "value": value, "model": "A"}
        for key, value in PRIOR_A_SUMMARY.items()
    ] + [
        {"quantity": key, "value": value, "model": "B"}
        for key, value in PRIOR_B_SUMMARY.items()
    ] + [
        {"quantity": "P(sigma_phase < log 1.1)", "value": P_SIGMA_PHASE_IN_ROPE, "model": "C"},
        {"quantity": "P(|kappa - 1| < 0.1)", "value": P_KAPPA_IN_ROPE, "model": "D2"},
        {"quantity": "P(prior predictive f1 < 0)", "value": P_F1_NEGATIVE, "model": "D2"},
        {"quantity": "skeleton triplets", "value": float(len(SKELETON)), "model": "design"},
        {"quantity": "full design triplets", "value": float(TOTAL_TRIPLETS), "model": "design"},
        {"quantity": "distinct harmonics", "value": float(SKELETON.f_lock.nunique()),
         "model": "design"},
    ])
    save_table(PRIOR_PREDICTIVE, "01_prior_predictive.parquet")
show(PRIOR_PREDICTIVE)

---
# Part 2, Observation: Chronos produces the data

This is the only part that loads a model. It runs `support_scripts/collect.py`, which visits the
fifteen geometries and writes five tidy tables; everything after this point treats them as plain
data.

| Table | Measurement | Why it exists |
|---|---|---|
| `contrasts` | forecast amplitude recovery $R=A_{\mathrm{pred}}/A_{\mathrm{true}}$ at each candidate $f_k$ and at both controls $f_k\pm\delta$, sharing the background realisation and the phase | the behavioural endpoint of Eq. (8); the phase index is kept, which is what makes H2 testable at all |
| `mdl_cells` | prequential codelength of a probe separating $f_c-1$ Hz from $f_c+1$ Hz, per probe stage | the representational endpoint of Eq. (10) |
| `mdl_bandtasks` | the seven hierarchical band tasks, with shuffled-label and random-initialisation controls | descriptive cross-check of overall decodability; fitted by no model |
| `collapse` | across-patch token dispersion $z_g(f)$ on the union grid, in three signal modes | the location endpoint of Eq. (12) |
| `sites` | detected dips, assigned to a branch, with the branch fundamental and its median gap | the movement endpoint of Eq. (13) |

**Signals.** The tone rides on a unit-variance Light TSMixup or KernelSynth background at the
amplitude ratio set in Part 0.4. The collapse sweep additionally records the pure sinusoid,
because only there is the degeneracy exact, $z=0$ when consecutive patches coincide; the two
background modes show the same comb as a deep dip and are what the verdict is read from.

**Resumption.** Shards are written per table per geometry and skipped on the next run, and a
geometry whose shards are all present never loads its model at all. An interrupted session
therefore costs only the table it was writing.

## 2.1, What the collection costs, before it starts

The counts below are exact rather than estimated: they are read from the same `probe_lib`
functions the collectors loop over. Only the seconds per forward pass are measured, by timing one
batch on the smallest geometry.

The estimate covers Chronos forward passes alone. It excludes parameter recovery, the posterior
fits, the sensitivity ladder, LOO and posterior-predictive simulation, so it is not a total wall
time. Part 0.9's projection is the figure that covers those.

In [ ]:
#@title 2.1  Exact forward-pass counts and a measured cost estimate
banner("PART 2, OBSERVATION")

import collect

# Both generators even in smoke. The shipped smoke configuration keeps one, which is cheaper but
# makes the design gate of Part 2.5 pass a criterion the full run would fail: u_bg is indexed by
# (generator, realisation), and a rehearsal that never exercises the second generator is not a
# rehearsal of this design.
_cfg_kwargs = dict(tone_snr=float(TONE_SNR), sites_per_block=int(SITES_PER_BLOCK),
                   batch_size=int(COLLECTION_BATCH_SIZE), generators=tuple(pl.GENERATORS))
if COLLECTION_DEVICE != "auto":
    _cfg_kwargs["device"] = COLLECTION_DEVICE
CFG = collect.Config(**_cfg_kwargs) if IS_FULL else collect.Config.smoke_cfg(**_cfg_kwargs)
if CFG.smoke != (not IS_FULL):
    raise ValueError("the collection config and the notebook mode disagree about smoke")
if CFG.tone_snr != TONE_SNR:
    raise ValueError("the collection config did not take the tone amplitude")
if tuple(CFG.generators) != GENERATORS and IS_FULL:
    raise ValueError("the collection config does not carry both generators")

SESSION_HOURS = 3.0        # the length of one Colab session, for the split printed below
SEC_FORECAST = 0.035       # fallback seconds per forecast, overwritten by the calibration
SEC_CAPTURE = 0.020        # a forecast decodes the horizon; a capture does not


def forward_pass_counts(P: int, S: int, cfg) -> tuple[int, int]:
    """Exact forward-pass counts for one geometry, split into forecasts and state captures."""
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    sites = [f for f, delta in offsets.items() if np.isfinite(delta)]
    n_generators = len(cfg.generators)

    forecasts = sum(len(pl.phases_Sf(f, cfg.n_phase_contrast))
                    for f in sites) * cfg.n_bg * 3 * n_generators

    centres = {round(f, 6) for fk in sites
               for f in (fk, fk - offsets[fk], fk + offsets[fk])}
    captures = len(centres) * 2 * cfg.mdl_n_per_class

    if cfg.band_tasks:
        band = np.arange(BAND[0], BAND[1] + 1e-9, cfg.bt_step)
        captures += sum(len(pl.phases_Sf(f, cfg.bt_n_phase)) for f in band)
        if cfg.bt_random_init:
            coarse = np.arange(BAND[0], BAND[1] + 1e-9, cfg.bt_step * 4)
            captures += sum(len(pl.phases_Sf(f, max(2, cfg.bt_n_phase // 2))) for f in coarse)

    captures += (len(pl.union_grid(FULL_MODELS, cfg.collapse_step))
                 * cfg.collapse_reps * len(cfg.collapse_modes))
    return forecasts, captures


if CALIBRATE_COST and HAVE_TORCH and ALLOW_COLLECTION:
    try:
        probe = pl.Probe(8, 8, device=CFG.device, batch_size=CFG.batch_size)
        try:
            n = min(16, CFG.batch_size)
            full = np.stack([pl.build_context(None, 40.0, 0.0, CTX + PRED)] * n)
            t0 = time.perf_counter()
            probe.recovery(full[:, :CTX], full[:, CTX:], np.full(n, 40.0))
            SEC_FORECAST = (time.perf_counter() - t0) / n
            t0 = time.perf_counter()
            probe.capture_reg(full[:, :CTX])
            SEC_CAPTURE = (time.perf_counter() - t0) / n
            print(f"calibrated on p8-s8, device={probe.device}: "
                  f"{SEC_FORECAST * 1000:.1f} ms per forecast, "
                  f"{SEC_CAPTURE * 1000:.1f} ms per capture")
        finally:
            probe.close()
            gc.collect()
    except Exception as exc:
        print(f"calibration skipped ({type(exc).__name__}: {exc}); using the fallback rates")
else:
    print("calibration off, Chronos unavailable or collection disabled; using the fallback rates")

COST = pd.DataFrame([
    {"model": pl.model_tag(P, S), "forecasts": f, "captures": c,
     "hours": (f * SEC_FORECAST + c * SEC_CAPTURE) / 3600}
    for P, S in FULL_MODELS for f, c in [forward_pass_counts(P, S, CFG)]
]).sort_values("hours").reset_index(drop=True)
COST["cumulative_h"] = COST.hours.cumsum()
show(COST, caption="ordered smallest first, which is also the order a partial session should "
                   "collect in")

TOTAL_HOURS = float(COST.hours.sum())
print(f"\ntotal: {int(COST.forecasts.sum()):,} forecasts + {int(COST.captures.sum()):,} captures "
      f"= {TOTAL_HOURS:.1f} h at the rates above")

group, accumulated, sessions = [], 0.0, []
for row in COST.itertuples():
    if accumulated + row.hours > SESSION_HOURS and group:
        sessions.append((group, accumulated))
        group, accumulated = [], 0.0
    group.append(row.model)
    accumulated += row.hours
if group:
    sessions.append((group, accumulated))
print(f"\nif the runtime is limited to ~{SESSION_HOURS:g} h, this is {len(sessions)} session(s):")
for index, (models, hours) in enumerate(sessions, 1):
    print(f"  session {index} ({hours:.1f} h): {models}")
note("Nothing has to be done to act on that split. The notebook collects in this order and skips "
     "any geometry whose shards are already present, so re-running it after a disconnect "
     "continues where it stopped.", "info")

## 2.2, Collecting

`collect.py` does the work and this cell is a single call into it. Three of its settings carry the
choices this analysis makes, and all three are recorded in the collection's own manifest.

**`tone_snr`.** The amplitude of the tone over the unit-variance background. It lives in the
configuration object rather than in a module constant, which is what puts it inside the design
fingerprint: a directory collected at one amplitude refuses a run that asks for another, instead of
merging two sets of observations that differ in the one design constant the report does not fix.

**`sites_per_block`.** How many candidate frequencies are forwarded at once. The whole geometry at
once is what an unblocked pass does, and at 100 backgrounds per generator that reaches 130,000
contexts of 544 samples, twice over for the true continuations, which is enough to exhaust a hosted
runtime before the first shard is written. Blocking changes the working set and nothing else: the
blocks partition the candidate list in order and each row's readings depend on that row alone, so
the table is identical to an unblocked one, row for row.

**`response="contrast"`.** Which criteria the design gate applies. The gate's coverage and schema
checks always run, and so does the requirement that both arm levels and both generators be present
in every geometry. The rest depend on what is being fitted: under `localisation` the hit indicator
must not be constant and each geometry must retain candidate sites under the instrument ceiling,
neither of which is defined for a paired contrast; under `contrast` every geometry must instead
yield complete triplets with finite, non-negative recoveries, which is what Eq. (8) needs. Selecting
the right one is not a way of avoiding a gate: it is the difference between a criterion and a
criterion that does not apply.

**Resumption.** Shards are written per table per geometry and skipped on the next run, and a
geometry whose shards are all present never loads its model at all. An interrupted session costs
only the table it was writing, and the collector reports its resident memory as it goes.

In [ ]:
#@title 2.2  Collect the five tables
if STAGE in ("collect", "all") and ALLOW_COLLECTION:
    if not HAVE_TORCH:
        raise RuntimeError("this stage needs Chronos and Part 0.2 reported it unavailable")
    by_tag = {pl.model_tag(P, S): (P, S) for P, S in FULL_MODELS}
    ordered_models = [by_tag[tag] for tag in COST.model]     # cheapest geometry first
    with PROGRESS.task("part2_collection"):
        collect.collect_all(DATA_DIR, models=ordered_models, cfg=CFG,
                            planned_models=FULL_MODELS, response="contrast")
else:
    print(f"stage {STAGE!r}: no collection in this session "
          f"(collection allowed: {ALLOW_COLLECTION})")

COLLECTION_STATE, COLLECTED_TAGS = collection_status()
print(f"collection: {COLLECTION_STATE} ({len(COLLECTED_TAGS)}/15 geometries)")
if COLLECTION_STATE != "complete":
    raise RuntimeError(
        f"the collection is {COLLECTION_STATE}: {len(COLLECTED_TAGS)}/15 geometries have a "
        "complete shard set. It is safely checkpointed, so re-run this notebook to continue from "
        "the geometry it stopped on. Do not go on to the fits on a partial design.")

## 2.3, Loading, with the manifest verified

`collect.load_collection` is the single verified entry point: it refuses a manifest whose status is
not complete, refuses a merged file whose SHA-256 has moved, checks each table's required columns
and finiteness, and applies the design gate for the response asked for.

The amplitude is then read back out of the manifest the collection wrote and compared with the one
this run asked for. That comparison is cheap and it is the one that matters: the manifest's design
fingerprint already refuses a mismatch, and reading the value lets the notebook say which amplitude
the data in front of it were measured at rather than only that something differs.

In [ ]:
#@title 2.3  Load the five tables and verify their provenance
DATA = collect.load_collection(DATA_DIR, cfg=CFG, planned_models=FULL_MODELS,
                               require_complete=True, response="contrast")
CONTRAST_ARMS = DATA["contrasts"]
MDL = DATA["mdl_cells"].reset_index(drop=True)
COLLAPSE = DATA["collapse"].reset_index(drop=True)
SITES = DATA["sites"].reset_index(drop=True)

COLLECTION_MANIFEST = json.loads((DATA_DIR / "collection_manifest.json").read_text())
SOURCE_HASHES = {name: entry["sha256"]
                 for name, entry in COLLECTION_MANIFEST["merged"].items()}
SOURCE_HASHES["collection_manifest"] = sha256_file(DATA_DIR / "collection_manifest.json")

COLLECTED_SNR = COLLECTION_MANIFEST["design"]["config"].get("tone_snr")
if COLLECTED_SNR is None:
    raise RuntimeError(
        f"the collection in {DATA_DIR} predates the amplitude being part of the design, so what "
        "these observations were measured at cannot be established. Collect into a fresh "
        "RUN_FOLDER.")
if abs(float(COLLECTED_SNR) - TONE_SNR) > 1e-12:
    raise RuntimeError(
        f"these observations were measured at tone SNR {COLLECTED_SNR} and this run asks for "
        f"{TONE_SNR}. Change TONE_SNR back, or change RUN_FOLDER for a fresh collection.")

show(pd.DataFrame([{"table": name, "rows": len(frame),
                    "geometries": frame.model.nunique(),
                    "sha256": SOURCE_HASHES.get(name, "-")[:16]}
                   for name, frame in DATA.items()]),
     caption=f"loaded from {DATA_DIR}, measured at tone SNR {COLLECTED_SNR}. mdl_bandtasks is the "
             f"descriptive band-probe sweep; no model is fitted to it, and it is collected "
             f"because the report promises it as a cross-check of overall decodability.")

recorded_check = COLLECTION_MANIFEST.get("design_check", {})
print("design gate recorded by the collection:",
      f"{recorded_check.get('response', '?')}, "
      f"{'PASS' if recorded_check.get('ok') else 'FAIL'}")

## 2.4, From arms to the paired contrast

The collected table carries one row per **arm**, so the three members of a triplet are three rows.
Models A and C are fitted to one number per **triplet**, the contrast of Eq. (8),

$$d=\log(R_k+0.01)-\tfrac12\left[\log(R_-+0.01)+\log(R_++0.01)\right],$$

negative when the candidate recovers less of its tone than its neighbours recover of theirs. It is
formed here rather than in the collection, which keeps the two sides independent: the collection
records what was measured and the notebook records what is fitted.

### What $R$ is measured against

Appendix B defines $R$ as "the ratio of the amplitude the forecast places at the frequency the arm
carries to the amplitude **injected** there". The estimator in the shared library has always
returned something else: the amplitude of the **true continuation** at that frequency, which is the
injected tone plus whatever the background happens to contribute there. The two are not the same
quantity and they do not agree. A forecast that reproduces the tone exactly and none of the
background scores 1 under the first and less than 1 under the second.

Neither is obviously wrong. The injected amplitude is a constant shared by all three arms of a
triplet, so it cancels from the contrast entirely and $d$ becomes a comparison of forecast
amplitudes alone. The continuation's amplitude differs between the three arms, so dividing by it
also controls for the background's differing energy at the three frequencies, at the cost of
putting that background's noise into the response.

The collection records `a_pred`, `a_true` and `amp_injected` per arm, so **both ratios are formed
from the same data** and the choice is not fixed at collection time. `RECOVERY_DENOMINATOR` in
Part 0.4 selects the one that is fitted, it enters the run fingerprint, and it defaults to
`injected`, which is what the document says. The other is computed alongside and the two are
compared below, so the size of the difference is reported rather than assumed small.

Four properties are asserted while pivoting, because each one silently corrupts the pairing if it
fails: no duplicated arm, so no two measurements are averaged without saying so; the three arms of
a triplet share one phase and one geometry, so the difference cannot be produced by either; each
arm's frequency is consistent with its role; and all three recoveries are finite and non-negative.

**No response filter is applied.** Section 4 of Deliverable 3 states the rule: the arms where the
forecast rebuilds nothing, which is where the loss should be largest, are kept rather than dropped.
Earlier notebooks fitted `contrasts[contrasts["live"]]` instead; the count that filter would have
removed is reported below as an audit figure, and it changes nothing here.

`phase_idx` is an index into a per-frequency list of non-redundant offsets, so the same index is a
**different physical angle at different frequencies**. Model C's eight slices are cut on
`phase mod 2π`, the angle, and the mapping is written out so the substitution can be checked.

In [ ]:
#@title 2.4  Derive the triplet contrast, with the pairing asserted
_ROLE_MAP = {"lock": "lock", "lo": "lo", "hi": "hi"}
_TRIPLET_KEYS = ["model", "generator", "bg_id", "f_lock", "phase_idx"]


def canonical_contrasts(arms: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """One row per triplet, carrying d and the deficit y = -d, with the pairing asserted."""
    required = {"model", "generator", "bg_id", "f_lock", "phase_idx", "phase", "P", "S",
                "overlap", "role", "f", "a_pred", "a_true", "amp_injected"}
    missing = sorted(required - set(arms.columns))
    if missing:
        raise ValueError(f"the contrasts table is missing {missing}")
    frame = arms[arms.model.isin(TAGS)].copy()
    if frame[sorted(required)].isna().any().any():
        raise ValueError("missing metadata in the contrasts table; triplets cannot be rebuilt")

    frame["_role"] = frame.role.astype(str).str.lower().map(_ROLE_MAP)
    if frame._role.isna().any():
        raise ValueError("unrecognised arm roles; lock, lo and hi are expected")
    if frame.duplicated(_TRIPLET_KEYS + ["_role"]).any():
        raise ValueError("duplicated arms: two measurements of one arm would be averaged silently")

    invariant = frame.groupby(_TRIPLET_KEYS, observed=True)[["phase", "P", "S", "overlap"]].nunique()
    if (invariant > 1).any().any():
        raise ValueError("the arms of a triplet do not share phase and geometry: not paired")
    consistent = ((frame._role.eq("lock") & np.isclose(frame.f, frame.f_lock, atol=1e-6))
                  | (frame._role.eq("lo") & (frame.f < frame.f_lock))
                  | (frame._role.eq("hi") & (frame.f > frame.f_lock)))
    if not consistent.all():
        raise ValueError("an arm's frequency is inconsistent with its role")

    # Both ratios, from the same forward pass. `primary` is the one that is fitted.
    frame["_R_injected"] = frame.a_pred / frame.amp_injected
    frame["_R_continuation"] = frame.a_pred / np.maximum(frame.a_true, 1e-9)
    primary = "_R_injected" if RECOVERY_DENOMINATOR == "injected" else "_R_continuation"
    alternative = "_R_continuation" if primary == "_R_injected" else "_R_injected"

    def pivot(column: str, prefix: str) -> pd.DataFrame:
        return (frame.pivot(index=_TRIPLET_KEYS, columns="_role", values=column)
                .reindex(columns=["lock", "lo", "hi"])
                .rename(columns={"lock": f"{prefix}_lock", "lo": f"{prefix}_lo",
                                 "hi": f"{prefix}_hi"}))

    meta = (frame.groupby(_TRIPLET_KEYS, observed=True)
            [["phase", "P", "S", "overlap", "family", "cpp", "delta"]].first())
    out = meta.join(pivot(primary, "R")).join(pivot(alternative, "Ralt")).reset_index()

    recoveries = out[["R_lock", "R_lo", "R_hi"]].to_numpy(float)
    complete = np.isfinite(recoveries).all(axis=1) & (recoveries >= 0).all(axis=1)
    audit = {
        "arms": int(len(frame)),
        "triplets_formed": int(len(out)),
        "triplets_incomplete": int((~complete).sum()),
        "legacy_live_rows": (int(arms["live"].sum()) if "live" in arms else None),
        "recovery_denominator": RECOVERY_DENOMINATOR,
        "rule": "every complete triplet with finite, non-negative recoveries; no response filter",
    }
    out = out.loc[complete].reset_index(drop=True)

    def contrast(prefix: str) -> np.ndarray:
        return (np.log(out[f"{prefix}_lock"] + EPS)
                - 0.5 * (np.log(out[f"{prefix}_lo"] + EPS)
                         + np.log(out[f"{prefix}_hi"] + EPS)))

    out["d"] = contrast("R")
    out["d_alt"] = contrast("Ralt")          # the same contrast under the other denominator
    out["y_deficit"] = -out["d"]

    angle = np.mod(out.phase.to_numpy(float), 2 * np.pi)
    angle[np.isclose(angle, 2 * np.pi, atol=1e-10)] = 0.0
    out["phase_angle"] = angle
    out["phase_bin"] = np.floor(angle / (2 * np.pi / N_PHASE_BINS)).astype(int).clip(
        0, N_PHASE_BINS - 1)
    out["background"] = out.generator + "#" + out.bg_id.astype(str)
    out["harmonic"] = out.f_lock.round(6).astype(str)
    return out.sort_values(_TRIPLET_KEYS).reset_index(drop=True), audit


CONTRASTS, CONTRAST_AUDIT = canonical_contrasts(CONTRAST_ARMS)
CONTRAST_AUDIT.update(
    backgrounds=int(CONTRASTS.background.nunique()),
    harmonics=int(CONTRASTS.harmonic.nunique()),
    phase_bins=int(CONTRASTS.phase_bin.nunique()),
    geometries=int(CONTRASTS.model.nunique()),
)
show(pd.DataFrame([CONTRAST_AUDIT]).T.rename(columns={0: "value"}).reset_index()
     .rename(columns={"index": "quantity"}), caption="how the fitted population was formed")
if CONTRAST_AUDIT["legacy_live_rows"] is not None:
    note(f"The <code>live</code> column is present and marks "
         f"{CONTRAST_AUDIT['legacy_live_rows']:,} of {CONTRAST_AUDIT['arms']:,} arms. It is "
         "recorded and not used: Deliverable 3 states that no response filter is applied after "
         "collection, so every complete triplet is fitted.", "info")

PHASE_MAP = (CONTRASTS[["f_lock", "phase_idx", "phase_angle", "phase_bin"]]
             .drop_duplicates().sort_values(["f_lock", "phase_idx"]).reset_index(drop=True))
PHASE_MAP["degrees"] = np.rad2deg(PHASE_MAP.phase_angle)
save_table(PHASE_MAP, "02_phase_map.parquet")
show(PHASE_MAP.head(16), caption="the same phase_idx is a different physical angle at different "
                                 "frequencies, which is why Model C bins the angle")

save_table(CONTRASTS.drop(columns=["family"], errors="ignore"), "02_contrasts.parquet")
save_json(CONTRAST_AUDIT, "02_contrast_audit.json")
print(f"\nfitted population: {len(CONTRASTS):,} triplets "
      f"(the design predicted {TOTAL_TRIPLETS:,})")

# What the choice of denominator does to the response, before any model sees it.
_delta = CONTRASTS.d - CONTRASTS.d_alt
DENOMINATOR_COMPARISON = pd.DataFrame([
    {"reading": f"fitted: {RECOVERY_DENOMINATOR}", "mean": float(CONTRASTS.d.mean()),
     "median": float(CONTRASTS.d.median()), "sd": float(CONTRASTS.d.std()),
     "share negative": float((CONTRASTS.d < 0).mean())},
    {"reading": "the other denominator", "mean": float(CONTRASTS.d_alt.mean()),
     "median": float(CONTRASTS.d_alt.median()), "sd": float(CONTRASTS.d_alt.std()),
     "share negative": float((CONTRASTS.d_alt < 0).mean())},
    {"reading": "difference", "mean": float(_delta.mean()), "median": float(_delta.median()),
     "sd": float(_delta.std()), "share negative": float((_delta < 0).mean())},
])
save_table(DENOMINATOR_COMPARISON, "02_denominator_comparison.parquet")
show(DENOMINATOR_COMPARISON, caption="the same triplets under both definitions of R; a large "
                                     "difference here is a reason to read Part 5.3's robustness "
                                     "row rather than to prefer one silently")

figure, axes = plt.subplots(1, 2, figsize=(12, 3.4))
_bins = np.linspace(*np.quantile(np.r_[CONTRASTS.d, CONTRASTS.d_alt], [0.002, 0.998]), 90)
axes[0].hist(CONTRASTS.d, bins=_bins, histtype="step", lw=1.8, color=INK["primary"],
             label=f"fitted: {RECOVERY_DENOMINATOR}")
axes[0].hist(CONTRASTS.d_alt, bins=_bins, histtype="step", lw=1.4, color=INK["muted"],
             label="the other denominator")
axes[0].axvline(0, color=INK["line"], lw=0.9)
axes[0].set(title="the contrast under both definitions of $R$", xlabel="$d$")
axes[0].legend(fontsize=7)
_sample = CONTRASTS.sample(min(len(CONTRASTS), 30000), random_state=SEED)
axes[1].scatter(_sample.d_alt, _sample.d, s=3, alpha=0.15, color=INK["primary"], linewidths=0)
_lim = np.quantile(np.r_[_sample.d, _sample.d_alt], [0.002, 0.998])
axes[1].plot(_lim, _lim, ls="--", color=INK["line"], lw=1.1, label="equal")
axes[1].set(title="one against the other, per triplet", xlabel="the other denominator",
            ylabel="fitted", xlim=_lim, ylim=_lim)
axes[1].legend(fontsize=7)
figure.tight_layout()
savefig(figure, "P2_recovery_denominator.png")
plt.show()

## 2.5, Does the collected design support these five inferences?

This is the gate. It is written for the five models fitted here, and it asks of each one the
question that would make its estimand meaningless if the answer were no.

| Model | What must hold | Why |
|---|---|---|
| A, C | every geometry contributes triplets, both generators are present, and all eight phase slices are populated | $\beta_c$ is one number per geometry and $\sigma_\phi$ is the spread of eight offsets; an empty slice is an offset with no data |
| A$'$ | both centred covariates vary and are not collinear | $\delta_O$ and $\delta_P$ are separable only because the design moves $P$ and $S$ independently |
| B | both levels of the lock indicator appear, in every geometry, and every codelength is positive and finite | $\theta_{\mathrm{lock}}$ is the contrast between the two levels, and a Gamma response has positive support |
| D1 | the geometry dummies and the two grid labels are jointly of full rank, and $z>0$ in the two fitted modes | on the $P=S$ diagonal the two combs coincide, so the labels are identified only by the off-diagonal geometries |
| D2 | at least ten unambiguous sites per branch, on the replicate-averaged curves of the two generator modes | the identification bar of `tab:bayesDecisions`; below it the branch is reported as not identified, which is not a null result |

The grid labels are built here, once, from the same `probe_lib` helper that defines the combs
everywhere else. Membership is required **exactly**, to a tolerance of $10^{-6}$ Hz, because the
sweep grid is the union of a uniform one-hertz grid with the exact predicted sites of every
geometry, so an exact match is available. A one-hertz tolerance would also label the integer
neighbours of a non-integer site, $42$ and $43$ Hz for the $42.\overline{6}$ Hz stride site of
$S=12$, which are not on any comb; that dilutes $\theta_S$ towards zero. The one-hertz labelling is
fitted in Part 4.4 as a declared robustness reading rather than discarded.

Each curve is normalised by its **own off-grid median**, as Appendix B states. Because $\alpha_g$ is
a free level per geometry, $\theta_S$ and $\theta_P$ are invariant to any per-geometry constant
rescaling; the normalisation matters only in that it also removes differences between replicates,
which $\alpha_g$ does not absorb.

In [ ]:
#@title 2.5  The design gate for these five models
def grid_labels(frame: pd.DataFrame, tolerance: float) -> tuple[np.ndarray, np.ndarray]:
    """Exact-or-within-tolerance membership of each swept frequency in the two combs."""
    on_stride = np.zeros(len(frame))
    on_patch = np.zeros(len(frame))
    for (P, S), index in frame.groupby(["P", "S"], observed=True).indices.items():
        frequencies = frame.f.to_numpy(float)[index]
        on_stride[index] = pl.comb_distance(frequencies, FS / float(S)) <= tolerance
        on_patch[index] = pl.comb_distance(frequencies, FS / float(P)) <= tolerance
    return on_stride, on_patch


COLLAPSE = COLLAPSE.copy()
COLLAPSE["on_stride"], COLLAPSE["on_patch"] = grid_labels(COLLAPSE, GRID_TOL_HZ)
COLLAPSE["on_stride_alt"], COLLAPSE["on_patch_alt"] = grid_labels(COLLAPSE, GRID_TOL_ALT_HZ)
COLLAPSE["grid_class"] = np.select(
    [(COLLAPSE.on_stride == 1) & (COLLAPSE.on_patch == 0),
     (COLLAPSE.on_patch == 1) & (COLLAPSE.on_stride == 0),
     (COLLAPSE.on_stride == 1) & (COLLAPSE.on_patch == 1)],
    ["stride", "patch", "both"], default="neither")

# each curve divided by its own OFF-GRID median, per Appendix B
_offgrid = COLLAPSE.grid_class.eq("neither")  # noqa: E501
_median = (COLLAPSE[_offgrid].groupby(["model", "mode", "rep"], observed=True)["z"]
           .median().rename("z_offgrid_median"))
COLLAPSE = COLLAPSE.merge(_median, on=["model", "mode", "rep"], how="left")
if COLLAPSE.z_offgrid_median.isna().any() or (COLLAPSE.z_offgrid_median <= 0).any():
    raise ValueError("a collapse curve has no positive off-grid median; the normalisation of "
                     "Appendix B cannot be formed")
COLLAPSE["z_rel"] = COLLAPSE.z / COLLAPSE.z_offgrid_median
# One floor, applied uniformly, because the log of the ratio is what D1 is fitted to. On a pure
# sinusoid the dispersion is exactly zero at a lock and the log is undefined; on either generator
# the dip bottoms out far above the floor, so it binds nowhere and the two cases need no separate
# treatment. How often it binds is reported rather than assumed.
COLLAPSE["z_floored"] = np.maximum(COLLAPSE.z_rel, COMB_FLOOR)
FLOOR_BINDING = (COLLAPSE.assign(bound=COLLAPSE.z_rel < COMB_FLOOR)
                 .groupby("mode", as_index=False)
                 .agg(rows=("bound", "size"), floored=("bound", "sum"),
                      min_z_rel=("z_rel", "min")))
show(FLOOR_BINDING, caption=f"where the floor of {COMB_FLOOR:g} binds; it should bind only on the "
                            f"pure sinusoid, where the degeneracy is exact")

# Appendix B sweeps three signal modes and says that fitting all three is what shows the
# conclusion is not an artefact of noise-free inputs, so all three are fitted. The verdict is
# read from the two generator modes: on a pure sinusoid the dispersion is exactly zero at a lock,
# so its response depends on the floor below, and a stratum whose observed values are all equal
# has no dispersion for a predictive check to reproduce. That restriction is a declared departure
# from the appendix and the pure fit is reported in full beside the others.
D1_MODES = tuple(sorted(set(COLLAPSE["mode"])))
VERDICT_MODES = tuple(m for m in GENERATORS if m in D1_MODES)
REFERENCE_MODES = tuple(m for m in D1_MODES if m not in VERDICT_MODES)

failures: list[str] = []
checks: list[dict] = []


def record(name: str, ok: bool, detail: str) -> None:
    checks.append({"check": name, "ok": bool(ok), "detail": detail})
    if not ok:
        failures.append(f"{name}: {detail}")


# -- coverage ----------------------------------------------------------------------------
for name, frame in (("contrasts", CONTRASTS), ("mdl_cells", MDL), ("collapse", COLLAPSE),
                    ("sites", SITES)):
    absent = sorted(set(TAGS) - set(frame.model.astype(str)))
    record(f"coverage:{name}", not absent,
           "all fifteen geometries present" if not absent else f"missing {absent}")

# -- Models A and C ----------------------------------------------------------------------
per_geometry = CONTRASTS.groupby("model", observed=True).agg(
    triplets=("d", "size"), candidates=("f_lock", "nunique"),
    generators=("generator", "nunique"), phase_bins=("phase_bin", "nunique"),
    backgrounds=("background", "nunique"))
record("A/C:triplets per geometry", bool((per_geometry.triplets > 0).all()),
       f"minimum {int(per_geometry.triplets.min()):,} triplets in one geometry")
record("A/C:every configured generator everywhere",
       bool((per_geometry.generators == len(CFG.generators)).all()),
       f"{len(CFG.generators)} configured; per geometry "
       f"{sorted(per_geometry.generators.unique())}")
record("A/C:eight phase slices populated",
       int(CONTRASTS.phase_bin.nunique()) == N_PHASE_BINS,
       f"{int(CONTRASTS.phase_bin.nunique())} of {N_PHASE_BINS} slices carry data")
record("A/C:contrast finite", bool(np.isfinite(CONTRASTS.d).all()),
       "every d is finite")
record("A':covariates vary and are not collinear",
       bool(GEOM.x_overlap.std() > 0 and GEOM.x_logP.std() > 0 and abs(COVARIATE_CORR) < 0.9),
       f"corr = {COVARIATE_CORR:+.3f}")
show(per_geometry.reset_index(), caption="Models A and C: what each geometry contributes")

# -- Model B -----------------------------------------------------------------------------
mdl_levels = MDL.groupby("model", observed=True).is_locked.nunique()
record("B:both lock levels in every geometry", bool((mdl_levels == 2).all()),
       f"geometries with one level only: {sorted(mdl_levels[mdl_levels < 2].index)}")
record("B:codelength positive and finite",
       bool(np.isfinite(MDL.L_bits).all() and (MDL.L_bits > 0).all()),
       f"L_bits in [{MDL.L_bits.min():.1f}, {MDL.L_bits.max():.1f}] bits")
record("B:probe stages present", MDL.stage.nunique() >= 2,
       f"{MDL.stage.nunique()} stages: {sorted(MDL.stage.unique())[:4]} ...")

# -- Model D1 ----------------------------------------------------------------------------
for mode in D1_MODES:
    subset = COLLAPSE[COLLAPSE["mode"] == mode]
    geometry_codes, geometries = pd.factorize(subset.model, sort=True)
    matrix = np.column_stack([np.eye(len(geometries))[geometry_codes],
                              subset.on_stride.to_numpy(), subset.on_patch.to_numpy()])
    full_rank = np.linalg.matrix_rank(matrix) == matrix.shape[1]
    record(f"D1:{mode} labels separable from geometry", full_rank,
           f"rank {np.linalg.matrix_rank(matrix)} of {matrix.shape[1]} columns")
    record(f"D1:{mode} dispersion finite and non-negative",
           bool(np.isfinite(subset.z).all() and (subset.z >= 0).all()),
           f"z in [{subset.z.min():.3e}, {subset.z.max():.3e}], "
           f"{int((subset.z_rel < COMB_FLOOR).sum())} row(s) at the floor")
show(COLLAPSE.groupby(["mode", "grid_class"], observed=True).size().unstack(fill_value=0)
     .reset_index(), caption="rows per signal mode and grid class, on the union sweep grid")

# -- Model D2 ----------------------------------------------------------------------------
mean_curves = SITES[(SITES.rep == -1) & SITES["mode"].isin(VERDICT_MODES)]
site_totals = mean_curves.groupby("branch").n_sites.sum()
D2_IDENTIFICATION = {}
for branch in ("stride", "patch"):
    total = int(site_totals.get(branch, 0))
    identified = total >= MIN_D2_SITES
    D2_IDENTIFICATION[branch] = {"unambiguous_sites": total, "identified": bool(identified)}
    record(f"D2:{branch} identified", identified,
           f"{total} unambiguous sites against a bar of {MIN_D2_SITES}")
show(pd.DataFrame(D2_IDENTIFICATION).T.reset_index().rename(columns={"index": "branch"}),
     caption="the identification gate of tab:bayesDecisions, on replicate-averaged curves of the "
             "two generator modes")

DESIGN_CHECK = save_table(pd.DataFrame(checks), "02_design_check.parquet")
show(DESIGN_CHECK)
if failures:
    for line in failures:
        print("  FAIL", line)
    blocking = [f for f in failures if not f.startswith("D2:")]
    if blocking:
        raise ValueError("the collected design cannot support these models: "
                         + "; ".join(blocking))
    note("The D2 identification bar is not met for at least one branch. That is a property of the "
         "design, not a failure of the collection: the branch will be reported as "
         "<b>NOT IDENTIFIED</b>, which the document distinguishes from a null result. Everything "
         "else passes.", "warn")
else:
    note("<b>Design gate: PASS.</b> Every one of the five models is identified on these data.",
         "good")

## 2.6, The raw observations

Four readings of the collected data, before any model is fitted. The point of showing them first
is that a posterior is only ever a summary of what is already here: a claim that is not visible at
all in these panels and yet emerges from a fit is a claim to be suspicious of, and one that is
plainly visible here should survive the fit.

In [ ]:
#@title 2.6  What the data look like before any fit
figure, axes = plt.subplots(1, 3, figsize=(15, 3.8))

# (a) the contrast against cycles per patch: d < 0 is the localised loss H1 predicts
sample = CONTRASTS.sample(min(len(CONTRASTS), 40000), random_state=SEED)
axes[0].scatter(sample.cpp, sample.d, s=3, alpha=0.18, color=INK["primary"], linewidths=0)
binned = (sample.assign(bin=pd.cut(sample.cpp, 24))
          .groupby("bin", observed=True).agg(cpp=("cpp", "mean"), d=("d", "median")))
axes[0].plot(binned.cpp, binned.d, color=INK["accent"], lw=2, label="binned median")
axes[0].axhline(0, color=INK["line"], lw=1)
axes[0].axhline(LOG08, color=INK["bad"], ls="--", lw=1, label="log 0.8")
axes[0].set(title="Eq. (8) contrast against cycles per patch",
            xlabel=r"cycles per patch  $f_k P / f_s$",
            ylabel="$d$   (negative = the candidate recovers less)")
axes[0].legend(fontsize=7)

# (b) the same contrast by geometry, ordered by overlap: this is what M1 reads across
order = GEOM.sort_values("overlap").model.tolist()
positions = np.arange(len(order))
data = [CONTRASTS.loc[CONTRASTS.model == tag, "d"].to_numpy() for tag in order]
parts = axes[1].violinplot(data, positions=positions, widths=0.85, showextrema=False,
                           showmedians=True)
for body in parts["bodies"]:
    body.set_facecolor(INK["primary"])
    body.set_alpha(0.45)
parts["cmedians"].set_color(INK["accent"])
axes[1].axhline(0, color=INK["line"], lw=1)
axes[1].set_xticks(positions)
axes[1].set_xticklabels([f"{tag}\nO={GEOM.set_index('model').overlap[tag]:.2f}" for tag in order],
                        fontsize=6, rotation=90)
axes[1].set(title="the contrast by geometry, ordered by overlap", ylabel="$d$")

# (c) codelength by probe stage and lock status: the response of Model B
stages = sorted(MDL.stage.unique())
for level, colour, label in ((0, INK["primary"], "not a candidate"),
                             (1, INK["bad"], "candidate frequency")):
    means = [MDL.loc[(MDL.stage == s) & (MDL.is_locked == level), "L_bits"].mean()
             for s in stages]
    axes[2].plot(range(len(stages)), means, marker="o", ms=4, color=colour, label=label)
axes[2].set_xticks(range(len(stages)))
axes[2].set_xticklabels(stages, rotation=90, fontsize=6)
axes[2].set(title="Eq. (10) codelength by probe stage", ylabel="mean $L$ [bits]")
axes[2].legend(fontsize=7)

figure.tight_layout()
savefig(figure, "P2_raw_contrast.png")
plt.show()

In [ ]:
#@title 2.6b  The collapse profiles and the detected sites
example = "p16-s12" if "p16-s12" in TAGS else TAGS[0]
figure, axes = plt.subplots(1, 2, figsize=(15, 3.8))

subset = COLLAPSE[(COLLAPSE.model == example) & (COLLAPSE["mode"] == VERDICT_MODES[0])]
profile = subset.groupby("f", as_index=False).z_rel.mean().sort_values("f")
P0, S0 = int(subset.P.iloc[0]), int(subset.S.iloc[0])
axes[0].plot(profile.f, profile.z_rel, color=INK["bad"], lw=1.2)
for index, f in enumerate(pl.stride_locks(S0)):
    axes[0].axvline(f, color=INK["grid_s"], ls="--", lw=1, alpha=0.75,
                    label=f"stride grid $c f_s/S$, S={S0}" if index == 0 else None)
for index, f in enumerate(pl.patch_nulls(P0)):
    axes[0].axvline(f, color=INK["grid_p"], ls=":", lw=1, alpha=0.6,
                    label=f"patch grid $k f_s/P$, P={P0}" if index == 0 else None)
axes[0].axhline(1.0, color=INK["line"], lw=0.8)
axes[0].set(title=f"Eq. (12) collapse profile, {example}, {VERDICT_MODES[0]}",
            xlabel="frequency [Hz]", ylabel=r"$z$ / off-grid median")
axes[0].legend(fontsize=7)

# the normalised dispersion by grid class, pooled: the difference D1 estimates
classes = ["neither", "stride", "patch", "both"]
pooled = [np.log(COLLAPSE.loc[(COLLAPSE.grid_class == c)
                              & COLLAPSE["mode"].isin(VERDICT_MODES), "z_rel"].to_numpy())
          for c in classes]
present = [(c, values) for c, values in zip(classes, pooled) if len(values)]
box = axes[1].boxplot([values for _, values in present], showfliers=False,
                      patch_artist=True, widths=0.6)
axes[1].set_xticks(np.arange(1, len(present) + 1))
axes[1].set_xticklabels([c for c, _ in present])
for patch, (name, _) in zip(box["boxes"], present):
    patch.set_facecolor({"neither": INK["muted"], "stride": INK["grid_s"],
                         "patch": INK["grid_p"], "both": INK["accent"]}[name])
    patch.set_alpha(0.45)
axes[1].axhline(0, color=INK["line"], lw=0.9)
axes[1].set(title="log normalised dispersion by grid class, both generators pooled",
            ylabel=r"$\log(z / $off-grid median$)$")

figure.tight_layout()
savefig(figure, "P2_raw_collapse.png")
plt.show()

show(SITES[(SITES.rep == -1) & SITES["mode"].isin(VERDICT_MODES)]
     [["model", "P", "S", "mode", "branch", "n_sites", "f1", "delta_hat",
       "predicted_spacing", "n_ambiguous"]],
     caption="detected sites on the replicate-averaged curves: the input to Model D2")

---
# Part 3, The likelihood, the models, and whether they work

Part 1 fixed the priors and Part 2 produced the observations. This part supplies the third
ingredient and, more importantly, checks that it works before anything is trusted to it.

1. **3.1** The Student-$t_4$ density Eq. (9) prescribes, written out and checked against SciPy, with
   the picture of why $t_4$ rather than a Gaussian.
2. **3.2** The five models, one builder, with the parameterisation Appendix B requires and an
   explicit statement of which offset sets are constrained to sum to zero and which is not.
3. **3.3** The five dataset specifications, built from the collected tables.
4. **3.4** The sampler, in chain blocks, each block checkpointed.
5. **3.5** The reconstructed log likelihood, **proved** against `pm.compute_log_likelihood`.
6. **3.6** PSIS-LOO on that reconstruction, **proved** against `az.loo`.
7. **3.7** Parameter recovery: every family refitted on data simulated with a known effect, in two
   scenarios, on the real design axes.

Sections 3.5 and 3.6 exist because Model A has 371,000 observations. Its `log_likelihood` at four
chains and 2000 retained draws would be 23.7 GiB, and this runtime has 10 GB. Nothing is
subsampled to avoid that: the array is never built, and the two parity gates are what make its
absence safe rather than merely convenient.

## 3.1, The Student-$t_4$ contrast likelihood

For observation $i$ with linear predictor $\mu_i$ and scale $\sigma$,

$$\log p(d_i \mid \mu_i, \sigma, \nu) = \log\Gamma\!\Big(\tfrac{\nu+1}{2}\Big)
- \log\Gamma\!\Big(\tfrac{\nu}{2}\Big) - \tfrac12\log(\pi\nu) - \log\sigma
- \tfrac{\nu+1}{2}\log\!\Big(1 + \tfrac{(d_i-\mu_i)^2}{\nu\sigma^2}\Big).$$

The last term is the whole point. A Gaussian charges a residual quadratically, so one contrast ten
scale units out costs 50 units of log density and the fit contorts itself to reduce it. The $t_4$
charges it logarithmically, about 5 units, so an outlier is **tolerated rather than obeyed**. With
recovery ratios that can collapse towards zero at a frequency the model rebuilds nothing of, that
robustness is not a stylistic preference: it is what lets every collected row stay in the fit,
including the arms where the predicted loss should be largest.

In [ ]:
#@title 3.1  The likelihood, written out and checked against SciPy
banner("PART 3, LIKELIHOOD AND METHOD VALIDATION")


def student_t_logpdf(y, mu, sigma, nu: int = NU):
    """log Student-t(nu, mu, sigma), written out rather than imported."""
    z2 = ((np.asarray(y) - mu) / sigma) ** 2
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * nu)
            - np.log(sigma) - (nu + 1) / 2 * np.log1p(z2 / nu))


_grid = np.linspace(-6, 6, 501)
if not np.allclose(student_t_logpdf(_grid, 0.3, 0.7),
                   stats.t.logpdf(_grid, NU, loc=0.3, scale=0.7)):
    raise ValueError("the hand-written Student-t density disagrees with SciPy")
print("student_t_logpdf agrees with scipy.stats.t.logpdf to machine precision")

figure, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(_grid, np.exp(student_t_logpdf(_grid, 0, 1)), lw=1.9, color=INK["primary"],
             label=r"Student-$t_4$")
axes[0].plot(_grid, stats.norm.pdf(_grid), lw=1.4, ls="--", color=INK["accent"], label="Normal")
axes[0].set(title="density", xlabel="residual / scale")
axes[0].legend(fontsize=8)
axes[1].plot(_grid, -student_t_logpdf(_grid, 0, 1), lw=1.9, color=INK["primary"],
             label=r"Student-$t_4$")
axes[1].plot(_grid, -stats.norm.logpdf(_grid), lw=1.4, ls="--", color=INK["accent"],
             label="Normal")
axes[1].set(title=r"what a residual costs, $-\log p$", xlabel="residual / scale")
axes[1].legend(fontsize=8)
figure.tight_layout()
savefig(figure, "P3_likelihood.png")
plt.show()

# What the data say about beta_bar with the nuisance parameters held fixed. Part 1 showed the
# prior was nearly flat over a factor of three; this shows the likelihood is not.
_d = CONTRASTS.d.to_numpy(float)
_grid_beta = np.linspace(-2.5, 2.5, 400)
_profile = np.array([student_t_logpdf(_d, b, np.std(_d)).sum() for b in _grid_beta])
figure, axis = plt.subplots(figsize=(7.5, 3.0))
axis.plot(_grid_beta, _profile - _profile.max(), lw=1.9, color=INK["primary"],
          label="log-likelihood profile (data)")
_prior_curve = stats.t.logpdf(_grid_beta, NU, scale=PRIOR_SCALE)
axis.plot(_grid_beta, _prior_curve - _prior_curve.max(), lw=1.4, ls="--", color=INK["muted"],
          label="log prior")
axis.axvline(_grid_beta[_profile.argmax()], color=INK["bad"], lw=1,
             label=f"profile maximum {_grid_beta[_profile.argmax()]:+.3f}")
axis.set(ylim=(-30, 1), xlabel=r"$\bar\beta$", ylabel="relative log density",
         title="the data, not the prior, locate the effect")
axis.legend(fontsize=8)
figure.tight_layout()
savefig(figure, "P3_profile.png")
plt.show()

## 3.2, The five models

Each model is one dataset specification plus one builder. Keeping them in a single builder is what
lets parameter recovery, the posterior predictive, LOO and the sensitivity ladder run uniformly over
all five instead of once per family, and it is what makes the reconstruction of Part 3.4 a mirror of
the builder rather than an independent second implementation of it.

### The parameterisation Appendix B requires

Appendix B states two rules that no earlier notebook implemented, and both matter here.

**Sum to zero.** "Because the offsets enter the same linear predictor additively, a constant added
to one set and subtracted from another leaves the likelihood unchanged, so each set is constrained
to sum to zero; without that constraint the levels are identified only through their priors and the
sampler is left to explore a ridge along which they trade off exactly." That ridge is exactly what
the earlier runs found: $\hat R$ up to 2.23 at an effective sample size of 5 **with zero
divergences**. Zero divergences with a terrible effective sample size is a ridge, not a funnel, so
no amount of re-parameterising a funnel could have fixed it.

The constraint is applied to the **offset** sets: `u_harm` and `u_bg` in Models A and C, `u_phase` in
Model C, `u_stage` and `u_geometry` in Model B. It is deliberately **not** applied to $\beta_c$.
$\beta_c$ is not an offset around a separate intercept: it is the configuration level itself, and
it carries $\bar\beta$, $\delta_O$ and $\delta_P$ through its prior mean. Constraining it to sum to
zero would delete the regression that M1 is read from. With the offset sets constrained the additive
constant has nowhere left to go, which is what identifies the model.

**Centred or non-centred, per group.** "The configuration, harmonic and background effects of
Models A and C are informed by hundreds to thousands of observations each and are written in the
first form; the eight phase offsets of Model C are written in the second, since H2 predicts their
scale near zero, which is the regime the second form exists for." So $\beta_c$, `u_harm` and `u_bg`
are centred, and `u_phase` is written $u_p=\sigma_\phi z_p$ with $z_p$ standard normal and
sum-to-zero. Choosing the form per group rather than once per model is the appendix's own rule.

One consequence of the constraint is worth stating. A sum-to-zero normal of length $n$ and scale
$\sigma$ has component standard deviation $\sigma\sqrt{1-1/n}$, so `sigma_phase` is the scale
parameter of the constrained distribution rather than the realised spread of the eight offsets. The
decision rule of `tab:bayesDecisions` is written on the parameter and is applied to the parameter;
the realised spread is reported alongside as `sd_u_phase` so the two can be compared and neither is
mistaken for the other.

In [ ]:
#@title 3.2  Dataset specifications and the one model builder
@dataclass
class Spec:
    """One fitted dataset: its frame, its arrays, its latent dimensions and its PPC strata."""
    name: str
    family: str                      # A | C | B | D1 | D2
    frame: pd.DataFrame
    arrays: dict
    coords: dict
    constants: dict
    strata: tuple
    identified: bool = True
    reason: str = ""


def subset_spec(spec: Spec, index) -> Spec:
    """A row subset. The latent dimensions and the covariate centring stay those of the full
    design, so a fit on a subset still carries every level the full model has."""
    index = np.asarray(index)
    return replace(spec, frame=spec.frame.iloc[index].reset_index(drop=True),
                   arrays={key: np.asarray(value)[index] for key, value in spec.arrays.items()})


def with_response(spec: Spec, y) -> Spec:
    return replace(spec, arrays={**spec.arrays, "y": np.asarray(y, float)})


def stratified_index(frame: pd.DataFrame, limit: int, strata, seed: int = SEED) -> np.ndarray:
    """A quota from every stratum, then a random top-up, so no level is lost to subsampling."""
    if len(frame) <= limit:
        return np.arange(len(frame))
    rng = np.random.default_rng(seed)
    groups = list(frame.groupby(list(strata), observed=True, dropna=False, sort=True)
                  .indices.values())
    if len(groups) > limit:
        raise ValueError(f"{len(groups)} strata need at least as many rows; limit is {limit}")
    quota = max(1, limit // len(groups))
    chosen = np.concatenate([rng.choice(g, min(len(g), quota), replace=False) for g in groups])
    if len(chosen) < limit:
        rest = np.setdiff1d(np.arange(len(frame)), chosen)
        chosen = np.concatenate([chosen, rng.choice(rest, limit - len(chosen), replace=False)])
    return np.sort(chosen)


def make_model(spec: Spec, factor: float = 1.0, overlap: bool = True, patch: bool = True,
               phase: bool = True, use_s: bool = True, use_p: bool = True,
               adjusted: bool = True, likelihood: str = "student",
               alt_tolerance: bool = False) -> pm.Model:
    """The five families of Deliverable 2, as Appendix B states them.

    `factor` multiplies every prior scale, which is the sensitivity ladder of Part 5.3. The other
    switches select the nested alternatives the LOO comparisons need: the configuration covariates
    of Model A, the phase term of Model C, the two grid labels of Model D1, and Model B's stage and
    geometry offsets. `likelihood="normal"` swaps in a Gaussian for Model A's robustness check, and
    `alt_tolerance` swaps Model D1's exact grid labels for the one-hertz ones.
    """
    arrays, coords, constants = spec.arrays, spec.coords, spec.constants
    y = np.asarray(arrays["y"], float)
    model_coords = {**coords, "obs": np.arange(len(y))}

    with pm.Model(coords=model_coords) as model:
        if spec.family in ("A", "C"):
            scale = PRIOR_SCALE * factor
            beta_bar = pm.StudentT("beta_bar", nu=NU, mu=0.0, sigma=scale)
            tau = pm.HalfStudentT("tau", nu=NU, sigma=scale)
            sigma_harm = pm.HalfStudentT("sigma_harm", nu=NU, sigma=scale)
            sigma_bg = pm.HalfStudentT("sigma_bg", nu=NU, sigma=scale)
            sigma = pm.HalfStudentT("sigma", nu=NU, sigma=scale)

            configuration_mean = beta_bar
            if spec.family == "A":
                # Eq. (9)'s configuration level; delta_O is the estimand of M1
                if overlap:
                    delta_O = pm.StudentT("delta_O", nu=NU, mu=0.0, sigma=scale)
                    configuration_mean = configuration_mean + delta_O * constants["x_overlap"]
                if patch:
                    delta_P = pm.StudentT("delta_P", nu=NU, mu=0.0, sigma=scale)
                    configuration_mean = configuration_mean + delta_P * constants["x_logP"]

            # centred: each level is informed by hundreds to thousands of observations
            beta = pm.Normal("beta", mu=configuration_mean, sigma=tau, dims="config")
            # sum-to-zero: without it these two absorb an additive constant from beta
            u_harm = pm.ZeroSumNormal("u_harm", sigma=sigma_harm, dims="harmonic")
            u_bg = pm.ZeroSumNormal("u_bg", sigma=sigma_bg, dims="background")
            mu = beta[arrays["ci"]] + u_harm[arrays["ki"]] + u_bg[arrays["bi"]]

            if spec.family == "C" and phase:
                # non-centred AND sum-to-zero: sigma_phase is the estimand and H2 predicts it
                # near zero, which is the one place the funnel is genuinely at risk
                sigma_phase = pm.HalfNormal("sigma_phase", sigma=0.25 * factor)
                z_phase = pm.ZeroSumNormal("z_phase", sigma=1.0, dims="phasebin")
                u_phase = pm.Deterministic("u_phase", sigma_phase * z_phase, dims="phasebin")
                mu = mu + u_phase[arrays["pi"]]
                pm.Deterministic("sd_u_phase",
                                 pt.sqrt(pt.sum(u_phase ** 2) / (N_PHASE_BINS - 1)))

            if likelihood == "normal":
                pm.Normal("obs", mu=mu, sigma=sigma, observed=y, dims="obs")
            else:
                pm.StudentT("obs", nu=NU, mu=mu, sigma=sigma, observed=y, dims="obs")
            if spec.family == "A":
                pm.Deterministic("recovery_ratio", pm.math.exp(beta_bar))

        elif spec.family == "B":
            alpha_0 = pm.Normal("alpha_0", mu=constants["alpha_center"], sigma=1.0)
            theta_lock = pm.Normal("theta_lock", mu=0.0, sigma=0.5 * factor)
            r = pm.Gamma("r", alpha=2.0, beta=0.1)
            log_mu = alpha_0 + theta_lock * arrays["lock"]
            if adjusted:
                sigma_st = pm.HalfNormal("sigma_st", sigma=1.0 * factor)
                sigma_geo = pm.HalfNormal("sigma_geo", sigma=1.0 * factor)
                u_stage = pm.ZeroSumNormal("u_stage", sigma=sigma_st, dims="stage")
                u_geometry = pm.ZeroSumNormal("u_geometry", sigma=sigma_geo, dims="geometry")
                log_mu = log_mu + u_stage[arrays["si"]] + u_geometry[arrays["gi"]]
            mu = pm.math.exp(log_mu)
            pm.Gamma("obs", alpha=r, beta=r / mu, observed=y, dims="obs")
            pm.Deterministic("codelength_ratio", pm.math.exp(theta_lock))

        elif spec.family == "D1":
            suffix = "_alt" if alt_tolerance else ""
            alpha_g = pm.Normal("alpha_g", mu=0.0, sigma=1.0 * factor, dims="geometry")
            mu = alpha_g[arrays["gi"]]
            if use_s:
                theta_S = pm.Normal("theta_S", mu=0.0, sigma=1.0 * factor)
                mu = mu + theta_S * arrays["on_s" + suffix]
            if use_p:
                theta_P = pm.Normal("theta_P", mu=0.0, sigma=1.0 * factor)
                mu = mu + theta_P * arrays["on_p" + suffix]
            sigma = pm.HalfNormal("sigma", sigma=1.0 * factor)
            pm.Normal("obs", mu=mu, sigma=sigma, observed=y, dims="obs")

        elif spec.family == "D2":
            kappa = pm.Normal("kappa", mu=0.0, sigma=1.0 * factor)
            sigma_F = pm.HalfNormal("sigma_F", sigma=5.0 * factor)
            # the resolution floor enters in quadrature: a spacing read off a discrete grid is no
            # more precise than one step, and without the floor sigma_F is driven to zero
            pm.Normal("obs", mu=kappa * arrays["x"],
                      sigma=pm.math.sqrt(sigma_F ** 2 + DELTA_F ** 2), observed=y, dims="obs")

        else:
            raise ValueError(f"unknown family {spec.family!r}")
    return model


print("one builder, five families: A (H1 behavioural + M1), B (H1 representational), C (H2), "
      "D1 (H3 location), D2 (H3a and H3b movement)")

## 3.3, The specifications, built from the collected tables

`u_bg` is indexed by the pair (generator, realisation) rather than by the realisation number alone.
The two generators draw their realisation 0 independently, so indexing on the number would put two
unrelated signals in one level and the scale $\sigma_b$, which Appendix B says is what separates
generator-specific variation from the phenomenon, would be measuring something else.

Model D1 is fitted on the two generator modes. The pure sinusoid reaches $z=0$ exactly at a locked
frequency, so its log is undefined; it is a reference for the exactness of the degeneracy, which is
what Appendix B calls it, and it is fitted separately under a declared floor rather than pooled
into the verdict.

In [ ]:
#@title 3.3  Build the five specifications
def integer_codes(frame: pd.DataFrame, columns) -> tuple[np.ndarray, list]:
    values = pd.MultiIndex.from_frame(frame[list(columns)])
    codes, levels = pd.factorize(values, sort=True)
    if (codes < 0).any():
        raise ValueError(f"unindexable rows in {columns}")
    return codes.astype("int32"), [str(level) for level in levels]


def build_specs() -> dict[str, Spec]:
    specs: dict[str, Spec] = {}

    # -- A and C, on the paired contrast ---------------------------------------------------
    configuration_codes = pd.Categorical(CONTRASTS.model, categories=GEOM.model).codes.astype("int32")
    if (configuration_codes < 0).any():
        raise ValueError("a contrast row names a geometry outside the frozen design")
    harmonic_codes, harmonic_levels = integer_codes(CONTRASTS, ["harmonic"])
    background_codes, background_levels = integer_codes(CONTRASTS, ["generator", "bg_id"])
    contrast_arrays = {
        "y": CONTRASTS.d.to_numpy(float), "ci": configuration_codes,
        "ki": harmonic_codes, "bi": background_codes,
        "pi": CONTRASTS.phase_bin.to_numpy("int32"),
    }
    contrast_coords = {"config": GEOM.model.tolist(), "harmonic": harmonic_levels,
                       "background": background_levels,
                       "phasebin": [f"{j * 360 // N_PHASE_BINS}-"
                                    f"{(j + 1) * 360 // N_PHASE_BINS} deg"
                                    for j in range(N_PHASE_BINS)]}
    contrast_constants = {"x_overlap": GEOM.x_overlap.to_numpy(float),
                          "x_logP": GEOM.x_logP.to_numpy(float)}
    strata_AC = ("model", "generator", "harmonic", "phase_bin", "P", "S")
    specs["A"] = Spec("A", "A", CONTRASTS, contrast_arrays, contrast_coords,
                      contrast_constants, strata_AC)
    specs["C"] = Spec("C", "C", CONTRASTS,
                      {**contrast_arrays, "y": CONTRASTS.y_deficit.to_numpy(float)},
                      contrast_coords, contrast_constants, strata_AC)

    # -- B, on the probe codelength -------------------------------------------------------
    stage_codes, stage_levels = integer_codes(MDL, ["stage"])
    geometry_codes, geometry_levels = integer_codes(MDL, ["model"])
    specs["B"] = Spec(
        "B", "B", MDL,
        {"y": MDL.L_bits.to_numpy(float), "lock": MDL.is_locked.to_numpy(float),
         "si": stage_codes, "gi": geometry_codes},
        {"stage": stage_levels, "geometry": geometry_levels},
        {"alpha_center": float(np.log(MDL.L_bits.mean()))},
        tuple(column for column in ("model", "stage", "is_locked", "family") if column in MDL))

    # -- D1, on the token collapse, one specification per signal mode ----------------------
    for mode in D1_MODES:
        subset = COLLAPSE[COLLAPSE["mode"] == mode].reset_index(drop=True)
        codes, levels = integer_codes(subset, ["model"])
        specs[f"D1_{mode}"] = Spec(
            f"D1_{mode}", "D1", subset,
            {"y": np.log(subset.z_floored.to_numpy(float)), "gi": codes,
             "on_s": subset.on_stride.to_numpy(float), "on_p": subset.on_patch.to_numpy(float),
             "on_s_alt": subset.on_stride_alt.to_numpy(float),
             "on_p_alt": subset.on_patch_alt.to_numpy(float)},
            {"geometry": levels}, {}, ("model", "grid_class", "P", "S"))

    # -- D2, on the detected sites --------------------------------------------------------
    replicate_sites = SITES[(SITES.rep >= 0) & SITES["mode"].isin(VERDICT_MODES)].copy()
    for branch in ("stride", "patch"):
        for response, minimum in (("f1", 1), ("delta_hat", 2)):
            frame = replicate_sites[(replicate_sites.branch == branch)
                                    & (replicate_sites.n_sites >= minimum)
                                    & replicate_sites[response].notna()].reset_index(drop=True)
            if frame.empty:
                continue
            identified = D2_IDENTIFICATION[branch]["identified"]
            specs[f"D2_{branch}_{response}"] = Spec(
                f"D2_{branch}_{response}", "D2", frame,
                {"y": frame[response].to_numpy(float),
                 "x": frame.predicted_spacing.to_numpy(float)},
                {}, {}, ("model", "mode"), identified,
                "" if identified else
                f"fewer than {MIN_D2_SITES} unambiguous sites on the averaged curves")
    return specs


SPECS = build_specs()
if MODE != "full":
    for name, spec in list(SPECS.items()):
        strata = {"A": ["model", "generator"], "C": ["model", "phase_bin"],
                  "B": ["model", "is_locked"], "D1": ["model", "grid_class"],
                  "D2": ["model"]}[spec.family]
        strata = [column for column in strata if column in spec.frame]
        SPECS[name] = subset_spec(spec, stratified_index(spec.frame, 400, strata))

show(pd.DataFrame([{"spec": name, "family": spec.family, "observations": len(spec.frame),
                    "latent dims": ", ".join(f"{k}={len(v)}" for k, v in spec.coords.items())
                                   or "none",
                    "identified": spec.identified}
                   for name, spec in SPECS.items()]),
     caption="A and C share their rows and their row order exactly; only the response differs")

if REFERENCE_MODES:
    note("All three signal modes are fitted, as Appendix B asks. The verdict is read from "
         f"{', '.join(VERDICT_MODES)}; {', '.join(REFERENCE_MODES)} is reported in full beside "
         "them but does not gate the claim, because there the degeneracy is exact, the response "
         f"depends on the floor of {COMB_FLOOR:g}, and a stratum whose observed values are all "
         "equal has no dispersion for a posterior predictive check to reproduce. That restriction "
         "is a departure from the appendix and is declared here rather than left implicit.",
         "warn")

## 3.4, The sampler

Four chains, 2000 warm-up draws, 2000 retained draws, target acceptance 0.9, for every fit, under
whichever implementation of NUTS Part 0.2 selected. The
chains are drawn in blocks of two and each block is checkpointed on its own, so a disconnect costs
at most one block. Blocks are given different seeds, which is what makes them independent chains and
lets $\hat R$ be computed across the concatenation.

No fit computes `log_likelihood`. PyMC builds that array **after** sampling finishes, when memory is
already at its peak, and for Model A it is 23.7 GiB. The posterior is bit-identical either way,
since the log likelihood is computed from the draws and does not influence them. Part 3.5 rebuilds
it where it is needed and proves that rebuild against PyMC before anything depends on it.

In [ ]:
#@title 3.4  Chain-blocked sampling and the convergence diagnostic
FIT_IDENTITY = {
    "analysis_fingerprint": ANALYSIS_FINGERPRINT,
    "collection": SOURCE_HASHES,
    "builders": code_fingerprint([make_model, build_specs, canonical_contrasts, grid_labels,
                                 integer_codes, stratified_index]),
}
IDENTITY_FILE = "03_fit_identity.json"
if have(IDENTITY_FILE):
    recorded_identity = load_json(IDENTITY_FILE)
    differing = [key for key in FIT_IDENTITY
                 if recorded_identity.get(key) != FIT_IDENTITY[key]]
    if differing:
        raise ValueError(
            f"the fits already in {CKPT_DIR} were produced under a different {differing}. "
            "Reusing them would mix results: change RUN_FOLDER, or restore the earlier state.")
else:
    save_json(FIT_IDENTITY, IDENTITY_FILE)
print("fit identity verified; checkpoints in this folder belong to this data and this code")

FIT_LOG: list[dict] = []


def diagnostics(idata, label: str = "") -> dict:
    """The three preregistered diagnostics, over every posterior variable.

    The variables are iterated rather than stacked into one array, because a Dataset of variables
    with different dimensions broadcasts into their cartesian product and the minimum ESS would
    then be read off cells that do not exist.
    """
    result = {"fit": label, "max_rhat": np.nan, "min_ess_bulk": np.nan, "min_ess_tail": np.nan,
              "divergences": np.nan, "chains": np.nan, "diagnostics_ok": False, "reason": ""}
    try:
        names = list(idata.posterior.data_vars)
        rhat = az.rhat(idata, var_names=names)
        bulk = az.ess(idata, var_names=names, method="bulk")
        tail = az.ess(idata, var_names=names, method="tail")
        flattened = [np.concatenate([np.asarray(dataset[name]).ravel() for name in names])
                     for dataset in (rhat, bulk, tail)]
        if not all(len(values) and np.isfinite(values).all() for values in flattened):
            result["reason"] = "a diagnostic is not finite"
            return result
        groups = idata.groups() if callable(idata.groups) else idata.groups
        if "sample_stats" not in {str(group).lstrip("/") for group in groups} \
                or "diverging" not in idata.sample_stats:
            result["reason"] = "the sampler reported no divergence statistic"
            return result
        result.update(max_rhat=float(flattened[0].max()),
                      min_ess_bulk=float(flattened[1].min()),
                      min_ess_tail=float(flattened[2].min()),
                      divergences=int(np.asarray(idata.sample_stats.diverging).sum()),
                      chains=int(idata.posterior.sizes["chain"]))
        result["diagnostics_ok"] = bool(
            result["chains"] >= CHAINS
            and result["max_rhat"] < RHAT_MAX
            and min(result["min_ess_bulk"], result["min_ess_tail"]) > ESS_MIN
            and result["divergences"] <= MAX_DIVERGENCES)
    except (ValueError, TypeError, KeyError, AttributeError) as exception:
        result["reason"] = f"{type(exception).__name__}: {exception}"
    return result


def sample_blocked(spec: Spec, name: str, options: dict | None = None, draws: int | None = None,
                   tune: int | None = None, seed: int = SEED, record_diagnostics: bool = True):
    """Fit `spec` under `options`, in blocks of BLOCK_CHAINS chains, resuming any block on disk.

    `NUTS_BACKEND` chooses who implements the sampler. The model, the priors, the number of draws
    and the target acceptance rate are identical either way, so the posterior is the same target
    distribution; only the implementation of NUTS, and therefore the throughput and the random
    stream, differ. Every diagnostic in Part 5.1 applies unchanged.
    """
    options = dict(options or {})
    draws = DRAWS if draws is None else draws
    tune = TUNE if tune is None else tune
    final = f"{name}.nc"
    if have(final):
        idata = load_idata(final)
    else:
        if not ALLOW_SAMPLING:
            raise RuntimeError(
                f"{final} is not in {CKPT_DIR} and sampling is disabled "
                f"(MODE={MODE!r}, STANDALONE_FIGURES={STANDALONE_FIGURES}). "
                "Set MODE to smoke or full, and STANDALONE_FIGURES to False, to fit it here.")
        blocks = []
        for start in range(0, CHAINS, BLOCK_CHAINS):
            count = min(BLOCK_CHAINS, CHAINS - start)
            block_name = f"{name}__chains{start:02d}.nc"
            if have(block_name):
                blocks.append(load_idata(block_name))
                continue
            print(f"  {name}: chains {start + 1}-{start + count}, {tune} warm-up + {draws} "
                  f"retained, n={len(spec.frame):,}", flush=True)
            extra = {} if NUTS_BACKEND == "pymc" else {"nuts_sampler": NUTS_BACKEND}
            with make_model(spec, **options):
                block = pm.sample(draws=draws, tune=tune, chains=count,
                                  cores=min(count, SAMPLE_CORES), random_seed=SEED + start,
                                  target_accept=TARGET_ACCEPT, progressbar=True,
                                  idata_kwargs={"log_likelihood": False}, **extra)
            save_idata(block, block_name)
            blocks.append(block)
            gc.collect()
        idata = (blocks[0] if len(blocks) == 1
                 else az.concat(*blocks, dim="chain", reset_dim=True))
        save_idata(idata, final)
        del blocks
        gc.collect()
    if record_diagnostics:
        entry = diagnostics(idata, name)
        entry["observations"] = len(spec.frame)
        entry["options"] = json.dumps(options, sort_keys=True)
        FIT_LOG.append(entry)
    return idata

## 3.5, The reconstructed log likelihood, and the proof that it is right

Every likelihood here is a closed-form density of a linear predictor built from the posterior and
the design index arrays, so the log likelihood of observation $i$ under draw $s$ can be evaluated
directly. Doing it in slices along the observation axis bounds the working set by the slice, not by
the design: Model A's 23.7 GiB becomes a few hundred megabytes.

Nothing about that is safe on its own. The reconstruction is a second implementation of what
`make_model` already expresses, and two implementations of one formula are two chances to be wrong.
So it is checked, for every one of the five families, against `pm.compute_log_likelihood` on the
same posterior and the same rows, and the notebook stops unless they agree to better than
$10^{-8}$. The check is cheap because it runs on a few hundred rows and a short fit; what it buys
is that every LOO comparison and every posterior-predictive check below rests on a formula that has
been shown to be the model's own.

In [ ]:
#@title 3.5  Reconstruct the log likelihood, and prove it against PyMC
_T_CONST = float(gammaln((NU + 1) / 2) - gammaln(NU / 2) - 0.5 * np.log(np.pi * NU))


def posterior_draws(idata) -> dict:
    """Every posterior variable flattened to [n_samples, ...], chain-major.

    Chain-major is the order ArviZ's own `__sample__` stacking produces, so per-observation
    quantities computed here line up with anything ArviZ computes from the same checkpoint.
    """
    out = {}
    for name, array in idata.posterior.data_vars.items():
        values = array.transpose("chain", "draw", ...).values
        out[name] = values.reshape(values.shape[0] * values.shape[1], *values.shape[2:])
    return out


def _linear_predictor(spec: Spec, draws: dict, options: dict, sl: slice) -> np.ndarray:
    """mu (or log mu, for Model B) for one observation slice: [n_samples, block]."""
    arrays = spec.arrays
    if spec.family in ("A", "C"):
        mu = draws["beta"][:, arrays["ci"][sl]]        # advanced indexing: already a copy
        mu += draws["u_harm"][:, arrays["ki"][sl]]
        mu += draws["u_bg"][:, arrays["bi"][sl]]
        if spec.family == "C" and options.get("phase", True):
            mu += draws["u_phase"][:, arrays["pi"][sl]]
        return mu
    if spec.family == "B":
        log_mu = (draws["alpha_0"][:, None]
                  + draws["theta_lock"][:, None] * arrays["lock"][sl])
        if options.get("adjusted", True):
            log_mu = log_mu + draws["u_stage"][:, arrays["si"][sl]]
            log_mu += draws["u_geometry"][:, arrays["gi"][sl]]
        return log_mu
    if spec.family == "D1":
        suffix = "_alt" if options.get("alt_tolerance") else ""
        mu = draws["alpha_g"][:, arrays["gi"][sl]]     # advanced indexing: already a copy
        if options.get("use_s", True):
            mu += draws["theta_S"][:, None] * arrays["on_s" + suffix][sl]
        if options.get("use_p", True):
            mu += draws["theta_P"][:, None] * arrays["on_p" + suffix][sl]
        return mu
    if spec.family == "D2":
        return draws["kappa"][:, None] * arrays["x"][sl]
    raise ValueError(spec.family)


def _scale(spec: Spec, draws: dict) -> np.ndarray:
    """The observation scale as a column, for the families that have one."""
    if spec.family in ("A", "C", "D1"):
        return draws["sigma"][:, None]
    if spec.family == "D2":
        return np.sqrt(draws["sigma_F"][:, None] ** 2 + DELTA_F ** 2)
    raise ValueError(spec.family)


def loglik_block(spec: Spec, draws: dict, options: dict, start: int, stop: int) -> np.ndarray:
    """log p(y_i | theta_s) for observations [start, stop): [n_samples, stop - start].

    Written with in-place arithmetic on one array, because the alternative is three temporaries of
    the slice's size and the slice is sized to the memory budget.
    """
    sl = slice(start, stop)
    y = np.asarray(spec.arrays["y"], float)[sl]
    if spec.family == "B":
        log_mu = _linear_predictor(spec, draws, options, sl)
        r = draws["r"][:, None]
        out = r * (np.log(r) - log_mu) - gammaln(r)
        out += (r - 1.0) * np.log(y)
        out -= r * np.exp(-log_mu) * y
        return out

    mu = _linear_predictor(spec, draws, options, sl)
    sigma = _scale(spec, draws)
    mu -= y                                  # residual
    mu /= sigma
    gaussian = (spec.family in ("D1", "D2")
                or options.get("likelihood", "student") == "normal")
    if gaussian:
        np.square(mu, out=mu)
        mu *= -0.5
        mu -= 0.5 * np.log(2.0 * np.pi)
        mu -= np.log(sigma)
        return mu
    np.square(mu, out=mu)
    mu /= NU
    np.log1p(mu, out=mu)
    mu *= -(NU + 1) / 2.0
    mu += _T_CONST
    mu -= np.log(sigma)
    return mu


def simulate_replicates(spec: Spec, draws: dict, options: dict, rows: np.ndarray,
                        draw_index: np.ndarray, rng) -> np.ndarray:
    """Posterior-predictive replicates for the given draws and rows: [n_draw, n_rows].

    Built from the same linear predictor as loglik_block, so a replicate is drawn from the model
    the log likelihood scores rather than from a second reading of it.
    """
    sub = {name: value[draw_index] for name, value in draws.items()}
    sl = slice(0, len(rows))
    picked = replace(spec, arrays={key: np.asarray(value)[rows]
                                  for key, value in spec.arrays.items()})
    if spec.family == "B":
        mean = np.exp(_linear_predictor(picked, sub, options, sl))
        r = sub["r"][:, None]
        return rng.gamma(shape=np.broadcast_to(r, mean.shape), scale=mean / r)
    mu = _linear_predictor(picked, sub, options, sl)
    sigma = _scale(picked, sub)
    gaussian = (spec.family in ("D1", "D2")
                or options.get("likelihood", "student") == "normal")
    if gaussian:
        return mu + sigma * rng.standard_normal(mu.shape)
    return mu + sigma * rng.standard_t(NU, mu.shape)


def pymc_log_likelihood(spec: Spec, idata, options: dict) -> np.ndarray:
    """PyMC's own log likelihood for the same fit, flattened chain-major."""
    with make_model(spec, **options):
        computed = pm.compute_log_likelihood(idata, extend_inferencedata=False,
                                             progressbar=False)
    dataset = computed.log_likelihood if hasattr(computed, "log_likelihood") else computed
    array = dataset["obs"].transpose("chain", "draw", ...).values
    return array.reshape(array.shape[0] * array.shape[1], -1)


PARITY_ROWS = 384
PARITY_DRAWS, PARITY_TUNE = (250, 250) if IS_FULL else (80, 80)
PARITY_TOLERANCE = 1e-8
PARITY_SPECS = [
    ("A", {}), ("A_normal", {"likelihood": "normal"}), ("C", {}), ("B", {}),
    (f"D1_{VERDICT_MODES[0]}", {}),
] + [(name, {}) for name in SPECS if name.startswith("D2_") and name.endswith("_f1")][:1]

PARITY_FILE = "03_parity.parquet"
if have(PARITY_FILE):
    PARITY = load_df(PARITY_FILE)
else:
    with PROGRESS.task("part3_parity"):
        rows = []
        for label, options in PARITY_SPECS:
            base = SPECS["A"] if label == "A_normal" else SPECS[label]
            strata = [column for column in ("model", "grid_class", "is_locked")
                      if column in base.frame]
            index = stratified_index(base.frame, min(PARITY_ROWS, len(base.frame)),
                                     strata or ["model"], seed=SEED + 3)
            small = subset_spec(base, index)
            short = sample_blocked(small, f"03_parity_{label}", options,
                                   draws=PARITY_DRAWS, tune=PARITY_TUNE,
                                   record_diagnostics=False)
            draws = posterior_draws(short)
            mine = loglik_block(small, draws, options, 0, len(small.frame))
            theirs = pymc_log_likelihood(small, short, options)
            difference = float(np.nanmax(np.abs(mine - theirs)))
            rows.append({"family": label, "observations": len(small.frame),
                         "samples": mine.shape[0], "max_abs_difference": difference,
                         "tolerance": PARITY_TOLERANCE,
                         "agrees": bool(difference < PARITY_TOLERANCE)})
            print(f"  {label:<14s} max |reconstructed - pymc| = {difference:.3e}")
            del draws, mine, theirs, short
            gc.collect()
        PARITY = save_table(pd.DataFrame(rows), PARITY_FILE)

show(PARITY, caption="the reconstructed log likelihood against pm.compute_log_likelihood, on the "
                     "same posterior and the same rows")
if not PARITY.agrees.all():
    raise ValueError("the reconstructed log likelihood does not match PyMC's for "
                     f"{PARITY.loc[~PARITY.agrees, 'family'].tolist()}; every LOO and PPC result "
                     "below would inherit that error")
note("<b>Reconstruction parity: PASS.</b> Every family's log likelihood agrees with PyMC's to "
     f"better than {PARITY_TOLERANCE:g}. The full 23.7 GiB array is therefore never needed, and "
     "LOO below runs on the whole design rather than on a subsample of it.", "good")

## 3.6, PSIS-LOO on the reconstruction, and the proof that it is right

The Pareto-smoothed importance sampling of `az.loo` is computed **independently per observation**,
so the log likelihood never has to be resident either. What follows is ArviZ's own algorithm, ported
so it can consume one slice at a time: the generalised-Pareto fit, its inverse, the smoothing of the
importance-sampling weights, and the aggregation into $\mathrm{elpd}$, its standard error and
$p_{\mathrm{loo}}$.

Ported, and therefore checked. Every quantity is compared against `az.loo(idata, pointwise=True)`
on the same short fits as above, including every per-observation $\mathrm{elpd}_i$ and every Pareto
$k$, and a disagreement stops the notebook. The comparison itself, `elpd_diff` and its standard
error, is then computed on paired per-observation differences exactly as ArviZ defines it.

In [ ]:
#@title 3.6  Streaming PSIS-LOO, and prove it against az.loo
def _logsumexp(values: np.ndarray) -> float:
    largest = np.max(values)
    if not np.isfinite(largest):
        largest = 0.0
    return float(np.log(np.sum(np.exp(values - largest))) + largest)


def _gpdfit(ordered: np.ndarray) -> tuple[float, float]:
    """Empirical-Bayes generalised-Pareto fit, as arviz.stats.stats._gpdfit."""
    prior_bs, prior_k = 3, 10
    n = len(ordered)
    m_est = 30 + int(n ** 0.5)
    b_ary = 1 - np.sqrt(m_est / (np.arange(1, m_est + 1, dtype=float) - 0.5))
    b_ary /= prior_bs * ordered[int(n / 4 + 0.5) - 1]
    b_ary += 1 / ordered[-1]
    k_ary = np.log1p(-b_ary[:, None] * ordered).mean(axis=1)
    length = n * (np.log(-(b_ary / k_ary)) - k_ary - 1)
    weights = 1 / np.exp(length - length[:, None]).sum(axis=1)
    real = weights >= 10 * np.finfo(float).eps
    if not np.all(real):
        weights, b_ary = weights[real], b_ary[real]
    weights /= weights.sum()
    b_post = np.sum(b_ary * weights)
    k_post = np.log1p(-b_post * ordered).mean()
    sigma = -k_post / b_post
    k_post = (n * k_post + prior_k * 0.5) / (n + prior_k)
    return k_post, sigma


def _gpinv(probabilities: np.ndarray, kappa: float, sigma: float) -> np.ndarray:
    """Inverse generalised-Pareto CDF, as arviz.stats.stats._gpinv."""
    x = np.full_like(probabilities, np.nan)
    if sigma <= 0:
        return x
    ok = (probabilities > 0) & (probabilities < 1)
    if np.all(ok):
        if np.abs(kappa) < np.finfo(float).eps:
            x = -np.log1p(-probabilities)
        else:
            x = np.expm1(-kappa * np.log1p(-probabilities)) / kappa
        x *= sigma
    else:
        if np.abs(kappa) < np.finfo(float).eps:
            x[ok] = -np.log1p(-probabilities[ok])
        else:
            x[ok] = np.expm1(-kappa * np.log1p(-probabilities[ok])) / kappa
        x *= sigma
        x[probabilities == 0] = 0
        x[probabilities == 1] = np.inf if kappa >= 0 else -sigma / kappa
    return x


def _psislw_one(log_weights: np.ndarray, cutoff_index: int, cutoff_min: float):
    """Smooth one observation's importance weights, as arviz.stats.stats._psislw."""
    x = np.asarray(log_weights, float)
    x = x - np.max(x)
    order = np.argsort(x)
    cutoff = max(x[order[cutoff_index]], cutoff_min)
    expcutoff = np.exp(cutoff)
    (tail_index,) = np.where(x > cutoff)
    tail = x[tail_index]
    if len(tail) <= 4:
        k = np.inf
    else:
        tail_order = np.argsort(tail)
        tail = np.exp(tail) - expcutoff
        k, sigma = _gpdfit(tail[tail_order])
        if np.isfinite(k):
            quantiles = np.arange(0.5, len(tail_index)) / len(tail_index)
            x[tail_index[tail_order]] = np.log(_gpinv(quantiles, k, sigma) + expcutoff)
            x[x > 0] = 0
    x -= _logsumexp(x)
    return x, k


def _relative_ess(idata) -> float:
    """Relative effective sample size, exactly as az.loo computes it when reff is not given."""
    posterior = idata.posterior
    n_chains = int(posterior.sizes["chain"])
    n_samples = n_chains * int(posterior.sizes["draw"])
    if n_chains == 1:
        return 1.0
    ess = az.ess(posterior, method="mean")
    values = np.hstack([np.asarray(ess[name]).ravel() for name in ess.data_vars])
    return float(values.mean() / n_samples)


def loo_reconstructed(spec: Spec, idata, options: dict, label: str = "",
                      target_bytes: int = LOO_TARGET_BYTES, report_every: int = 20) -> dict:
    """PSIS-LOO over the whole design, reading nothing larger than one observation slice."""
    draws = posterior_draws(idata)
    n_samples = int(idata.posterior.sizes["chain"] * idata.posterior.sizes["draw"])
    n_obs = len(spec.frame)
    reff = _relative_ess(idata)
    cutoff_index = -int(np.ceil(min(n_samples / 5.0, 3 * (n_samples / reff) ** 0.5))) - 1
    cutoff_min = float(np.log(np.finfo(float).tiny))

    block = max(1, min(n_obs, int(target_bytes // (n_samples * 8))))
    n_blocks = math.ceil(n_obs / block)
    print(f"  LOO {label or spec.name}: {n_obs:,} observations x {n_samples:,} draws, "
          f"{n_blocks} slices of at most {block:,} "
          f"(~{block * n_samples * 8 / 2 ** 20:.0f} MiB resident)", flush=True)

    elpd_i = np.empty(n_obs)
    pareto_k = np.empty(n_obs)
    lppd_i = np.empty(n_obs)
    log_n = float(np.log(n_samples))
    started = time.time()
    for index, start in enumerate(range(0, n_obs, block)):
        stop = min(start + block, n_obs)
        chunk = loglik_block(spec, draws, options, start, stop)
        # PSIS reads one observation's draws at a time. Transposing once makes each of those a
        # contiguous row instead of a strided column, which is the difference between a sort over
        # scattered memory and one over a cache line, repeated 371,000 times.
        rows_of_draws = np.ascontiguousarray(chunk.T)
        del chunk
        for position in range(stop - start):
            draws_i = rows_of_draws[position]
            log_weights, k = _psislw_one(-draws_i, cutoff_index, cutoff_min)
            elpd_i[start + position] = _logsumexp(log_weights + draws_i)
            pareto_k[start + position] = k
            lppd_i[start + position] = _logsumexp(draws_i) - log_n
        del rows_of_draws
        if (index + 1) % report_every == 0 or index + 1 == n_blocks:
            print(f"    slice {index + 1}/{n_blocks}, {time.time() - started:.0f}s", flush=True)
            gc.collect()

    elpd_loo = float(elpd_i.sum())
    good_k = float(min(1 - 1 / np.log10(n_samples), 0.7))
    return {"label": label or spec.name, "elpd_loo": elpd_loo,
            "se": float((n_obs * np.var(elpd_i)) ** 0.5),
            "p_loo": float(lppd_i.sum() - elpd_loo), "n_samples": n_samples, "n_obs": n_obs,
            "good_k": good_k, "max_pareto_k": float(np.nanmax(pareto_k)),
            "warning": bool(np.any(pareto_k > good_k)),
            "reliable": bool(np.isfinite(pareto_k).all() and np.nanmax(pareto_k) <= good_k),
            "elpd_i": elpd_i, "pareto_k": pareto_k}


def _elpd_field(loo, *names):
    for name in names:
        if hasattr(loo, name):
            return np.asarray(getattr(loo, name))
    raise AttributeError(f"the ArviZ LOO object exposes none of {names}")


PSIS_PARITY_FILE = "03_psis_parity.parquet"
if have(PSIS_PARITY_FILE):
    PSIS_PARITY = load_df(PSIS_PARITY_FILE)
else:
    rows = []
    for label, options in PARITY_SPECS:
        base = SPECS["A"] if label == "A_normal" else SPECS[label]
        strata = [column for column in ("model", "grid_class", "is_locked")
                  if column in base.frame]
        index = stratified_index(base.frame, min(PARITY_ROWS, len(base.frame)),
                                 strata or ["model"], seed=SEED + 3)
        small = subset_spec(base, index)
        short = load_idata(f"03_parity_{label}.nc")
        mine = loo_reconstructed(small, short, options, label=label, report_every=10 ** 6)
        with make_model(small, **options):
            computed = pm.compute_log_likelihood(short, extend_inferencedata=False,
                                                 progressbar=False)
        dataset = computed.log_likelihood if hasattr(computed, "log_likelihood") else computed
        reference = az.loo(az.InferenceData(posterior=short.posterior, log_likelihood=dataset),
                           pointwise=True)
        rows.append({
            "family": label,
            "d_elpd_loo": abs(mine["elpd_loo"] - float(_elpd_field(reference, "elpd_loo", "elpd"))),
            "d_se": abs(mine["se"] - float(_elpd_field(reference, "se"))),
            "d_p_loo": abs(mine["p_loo"] - float(_elpd_field(reference, "p_loo", "p"))),
            "max_d_elpd_i": float(np.nanmax(np.abs(
                mine["elpd_i"] - _elpd_field(reference, "loo_i", "elpd_i").ravel()))),
            "max_d_pareto_k": float(np.nanmax(np.abs(
                mine["pareto_k"] - _elpd_field(reference, "pareto_k").ravel()))),
        })
        del mine, computed, dataset, reference, short
        gc.collect()
    PSIS_PARITY = pd.DataFrame(rows)
    PSIS_PARITY["agrees"] = (PSIS_PARITY[[c for c in PSIS_PARITY.columns
                                          if c.startswith(("d_", "max_d_"))]].max(axis=1) < 1e-6)
    PSIS_PARITY = save_table(PSIS_PARITY, PSIS_PARITY_FILE)

show(PSIS_PARITY, caption="absolute differences against az.loo(pointwise=True) on the same fits")
if not PSIS_PARITY.agrees.all():
    raise ValueError("the ported PSIS-LOO disagrees with ArviZ for "
                     f"{PSIS_PARITY.loc[~PSIS_PARITY.agrees, 'family'].tolist()}")
note("<b>PSIS-LOO parity: PASS.</b> Every scalar, every per-observation elpd and every Pareto k "
     "agrees with ArviZ to better than 1e-6, so the comparisons in Part 5.4 are ArviZ's own "
     "numbers computed over the whole design instead of over a subsample.", "good")

## 3.7, Parameter recovery

Deliverable 2 lists parameter recovery among its validation steps and states what it is for: "data
simulated with a known effect, to confirm the model finds it. This is what separates *no effect*
from *a design that cannot see one*."

Two scenarios are run per family, because one would answer only half the question. The first
simulates the effect the claim is about, and asks whether the design can see it. The second simulates
the state the claim denies, and asks whether the model returns a posterior on **that** instead, which
is what a reported null has to rest on. A family that passed only the first could still be
manufacturing effects; one that passed only the second could be blind.

What "the state the claim denies" is depends on the claim, and it is not always zero. For A, B, C and
D1 it is no effect. For D2 it is a slope that is **not one**, since H3a and H3b claim that a branch
tracks its own parameter and their negation is that it tracks it imperfectly: the near-null scenario
is therefore $\kappa_F=0.65$, and the question is whether the posterior can tell that from one.

The simulation uses the **real design**: the same index arrays, the same number of levels, the same
centred covariates. For Models A and C that is a stratified subsample, declared here, because a
recovery run at the full 371,000 rows would cost as much as the reported fit and buys nothing: a
model that recovers a truth at reduced $n$ recovers it at full $n$ with a narrower interval. For B,
D1 and D2 the whole table is used, being small enough.

The group offsets are simulated **sum-to-zero**, matching the prior they are recovered under. An
independent draw has a group mean of order $\sigma_u/\sqrt{n}$ which the constrained model has
nowhere to put except the intercept, and the result would be a bias in the test rather than in the
estimator.

Coverage on its own would be too weak a gate to say what a PASS here is taken to say. An interval
wide enough to contain both scenarios' truths covers each of them and demonstrates nothing: it shows
that the estimator is not biased, not that the design can tell an effect from its absence. The gate
therefore has a second half, **discrimination**: the interval from each scenario must **exclude the
other scenario's truth**, in at least the same share of repeats. Coverage says the estimate is in
the right place; discrimination says the design can resolve the two places apart. The interval width
is reported beside both. These numbers validate the method; they are never evidence about
Chronos, and the verdict table never reads them as such.

In [ ]:
#@title 3.7  Simulate a known effect and refit, in two scenarios per family
def _zero_sum_draw(rng, n: int, scale: float) -> np.ndarray:
    draw = rng.normal(0.0, scale, n)
    return draw - draw.mean()


def simulate_known(spec: Spec, with_effect: bool, seed: int) -> tuple[Spec, dict]:
    """Simulate a response from `spec`'s own design, with the truths returned alongside."""
    rng = np.random.default_rng(seed)
    arrays, coords, constants = spec.arrays, spec.coords, spec.constants

    if spec.family in ("A", "C"):
        beta_bar = -0.30 if with_effect else 0.0
        configuration = beta_bar + _zero_sum_draw(rng, len(coords["config"]), 0.12)
        truth = {"beta_bar": beta_bar}
        if spec.family == "A":
            delta_O, delta_P = (0.30, -0.15) if with_effect else (0.0, 0.0)
            configuration = (configuration + delta_O * constants["x_overlap"]
                             + delta_P * constants["x_logP"])
            truth.update(delta_O=delta_O, delta_P=delta_P)
        mu = (configuration[arrays["ci"]]
              + _zero_sum_draw(rng, len(coords["harmonic"]), 0.12)[arrays["ki"]]
              + _zero_sum_draw(rng, len(coords["background"]), 0.10)[arrays["bi"]])
        if spec.family == "C":
            sigma_phase = 0.25 if with_effect else 0.02
            mu = mu + _zero_sum_draw(rng, N_PHASE_BINS, sigma_phase)[arrays["pi"]]
            truth = {"sigma_phase": sigma_phase}
        y = mu + 0.40 * stats.t.rvs(NU, size=len(mu), random_state=rng.integers(1 << 31))

    elif spec.family == "B":
        theta_lock = 0.35 if with_effect else 0.0
        log_mu = (constants["alpha_center"] + theta_lock * arrays["lock"]
                  + _zero_sum_draw(rng, len(coords["stage"]), 0.15)[arrays["si"]]
                  + _zero_sum_draw(rng, len(coords["geometry"]), 0.15)[arrays["gi"]])
        y = rng.gamma(shape=20.0, scale=np.exp(log_mu) / 20.0)
        truth = {"theta_lock": theta_lock}

    elif spec.family == "D1":
        theta_S, theta_P = (-0.70, -0.50) if with_effect else (0.0, 0.0)
        mu = (_zero_sum_draw(rng, len(coords["geometry"]), 0.30)[arrays["gi"]]
              + theta_S * arrays["on_s"] + theta_P * arrays["on_p"])
        y = mu + 0.40 * rng.normal(size=len(mu))
        truth = {"theta_S": theta_S, "theta_P": theta_P}

    elif spec.family == "D2":
        kappa = 1.0 if with_effect else 0.65
        y = (kappa * arrays["x"]
             + np.sqrt(2.0 ** 2 + DELTA_F ** 2) * rng.normal(size=len(arrays["x"])))
        truth = {"kappa": kappa}

    else:
        raise ValueError(spec.family)
    return with_response(spec, y), truth


RECOVERY_FAMILIES = ["A", "C", "B"] + [f"D1_{VERDICT_MODES[0]}"] \
    + [name for name in SPECS if name.startswith("D2_") and name.endswith("_f1")]
RECOVERY_FILE = "03_recovery.parquet"

if have(RECOVERY_FILE):
    RECOVERY = load_df(RECOVERY_FILE)
else:
    with PROGRESS.task("part3_recovery"):
        rows = []
        for name in RECOVERY_FAMILIES:
            spec = SPECS[name]
            strata = [column for column in ("model", "phase_bin", "grid_class", "is_locked")
                      if column in spec.frame]
            design = subset_spec(spec, stratified_index(
                spec.frame, min(RECOVERY_ROWS, len(spec.frame)),
                strata or ["model"], seed=SEED + 7))
            for with_effect in (False, True):
                scenario = "effect" if with_effect else "near_null"
                for repeat in range(RECOVERY_REPEATS):
                    seed = SEED + 1000 + 100 * repeat + int(with_effect)
                    simulated, truth = simulate_known(design, with_effect, seed)
                    fitted = sample_blocked(
                        simulated, f"03_recovery_{name}_{scenario}_{repeat}",
                        seed=seed, record_diagnostics=False)
                    diagnostic = diagnostics(fitted, f"recovery:{name}:{scenario}:{repeat}")
                    for parameter, value in truth.items():
                        posterior = fitted.posterior[parameter].values.ravel()
                        low, high = np.quantile(posterior, [0.025, 0.975])
                        rows.append({
                            "spec": name, "family": spec.family, "scenario": scenario,
                            "repeat": repeat, "parameter": parameter, "truth": float(value),
                            "median": float(np.median(posterior)), "low": float(low),
                            "high": float(high), "width": float(high - low),
                            "covered": bool(low <= value <= high),
                            "converged": bool(diagnostic["diagnostics_ok"]),
                            "max_rhat": diagnostic["max_rhat"],
                            "min_ess": min(diagnostic["min_ess_bulk"],
                                           diagnostic["min_ess_tail"]),
                            "divergences": diagnostic["divergences"],
                            "observations": len(design.frame)})
                    del fitted
                    gc.collect()
        RECOVERY = save_table(pd.DataFrame(rows), RECOVERY_FILE)

show(RECOVERY, caption="every recovery fit is SYNTHETIC and NON-REPORTABLE; these numbers "
                       "validate the estimators and say nothing about Chronos")

# The truth the OTHER scenario used, for the discrimination half of the gate.
_other_truth = (RECOVERY.pivot_table(index=["spec", "parameter"], columns="scenario",
                                     values="truth", aggfunc="first")
                .rename(columns={"effect": "truth_effect", "near_null": "truth_near_null"}))
RECOVERY = RECOVERY.merge(_other_truth, on=["spec", "parameter"], how="left")
RECOVERY["truth_other"] = np.where(RECOVERY.scenario == "effect",
                                   RECOVERY.truth_near_null, RECOVERY.truth_effect)
RECOVERY["separation"] = (RECOVERY.truth_effect - RECOVERY.truth_near_null).abs()
RECOVERY["excludes_other"] = ~((RECOVERY.low <= RECOVERY.truth_other)
                               & (RECOVERY.truth_other <= RECOVERY.high))

RECOVERY_OK: dict[str, bool] = {}
recovery_summary = []
for spec_name, group in RECOVERY.groupby("spec"):
    coverage = group.groupby(["scenario", "parameter"]).covered.mean()
    discrimination = group.groupby(["scenario", "parameter"]).excludes_other.mean()
    both_scenarios = set(group.scenario) == {"effect", "near_null"}
    passed = bool(both_scenarios and group.converged.all()
                  and (coverage >= RECOVERY_COVERAGE_MIN).all()
                  and (discrimination >= RECOVERY_COVERAGE_MIN).all())
    RECOVERY_OK[spec_name] = passed
    recovery_summary.append({
        "spec": spec_name, "both_scenarios": both_scenarios,
        "all_converged": bool(group.converged.all()),
        "min_coverage": float(coverage.min()),
        "min_discrimination": float(discrimination.min()),
        "mean_width": float(group.width.mean()),
        "mean_separation": float(group.separation.mean()),
        "recovery_ok": passed})
RECOVERY_SUMMARY = save_table(pd.DataFrame(recovery_summary), "03_recovery_summary.parquet")
show(RECOVERY_SUMMARY,
     caption=f"a family passes when both scenarios ran, every fit converged, at least "
             f"{RECOVERY_COVERAGE_MIN:.0%} of repeats cover their own truth, and at least "
             f"{RECOVERY_COVERAGE_MIN:.0%} exclude the other scenario's. A mean width comparable "
             f"to the mean separation is the warning sign the second criterion exists to catch.")

# a family's gate applies to every spec of that family
for family_prefix in ("D2_stride", "D2_patch"):
    base = f"{family_prefix}_f1"
    if base in RECOVERY_OK:
        for name in SPECS:
            if name.startswith(family_prefix):
                RECOVERY_OK[name] = RECOVERY_OK[base]
for mode in VERDICT_MODES + REFERENCE_MODES:
    if f"D1_{VERDICT_MODES[0]}" in RECOVERY_OK:
        RECOVERY_OK[f"D1_{mode}"] = RECOVERY_OK[f"D1_{VERDICT_MODES[0]}"]

if all(RECOVERY_OK.values()):
    note("<b>Parameter recovery: PASS.</b> On the real design axes, every family recovers the "
         "effect its claim is about, returns a posterior on the state the claim denies when handed "
         "that instead, and in both cases returns an interval narrow enough to exclude the other "
         "scenario. A null reported later is therefore a statement about the data rather than "
         "about a design that could not have seen an effect.", "good")
else:
    failed = [name for name, ok in RECOVERY_OK.items() if not ok]
    note(f"<b>Parameter recovery failed for {failed}.</b> Those claims are reported as NOT "
         "REPORTABLE in Part 5.6: an estimand from a model that cannot recover a known effect on "
         "this design carries no evidence either way.", "bad")

figure, axis = plt.subplots(figsize=(8.5, 0.34 * len(RECOVERY) + 1.4))
order = RECOVERY.sort_values(["spec", "scenario", "parameter", "repeat"]).reset_index(drop=True)
y = np.arange(len(order))
colours = np.where(order.covered & order.excludes_other, INK["primary"],
                   np.where(order.covered, INK["accent"], INK["bad"]))
axis.hlines(y, order.low, order.high, color=colours, lw=3, alpha=0.55)
axis.scatter(order["median"], y, color=colours, zorder=3, s=18,
             label="posterior median and 95% interval")
axis.scatter(order.truth, y, color=INK["line"], marker="D", zorder=4, s=26, label="the truth")
axis.scatter(order.truth_other, y, facecolors="none", edgecolors=INK["muted"], marker="D",
             zorder=4, s=26, label="the other scenario's truth, which the interval must exclude")
axis.set_yticks(y)
axis.set_yticklabels([f"{r.spec}  {r.scenario}  {r.parameter}  #{r.repeat}"
                      for r in order.itertuples()], fontsize=6)
axis.invert_yaxis()
axis.legend(fontsize=8, loc="best")
axis.set_title("Part 3.7 parameter recovery: SYNTHETIC, NON-REPORTABLE")
figure.tight_layout()
savefig(figure, "P3_recovery.png")
plt.show()

---
# Part 4, Posterior inference

The priors are fixed, the observations are in, the likelihood and its reconstruction are proved.
This part fits the five families and reports, for each, what Deliverable 2's validation paragraph
asks for: the estimand on its natural scale with a 95 per cent credible interval, the posterior
probability of each preregistered threshold, and the nested alternatives that the leave-one-out
comparisons of Part 5.4 need.

Nothing is decided here. A posterior is a conditional statement, and the conditions, convergence,
recovery, predictive adequacy, prior sensitivity and the reliability of the comparisons, are Part 5.
Every number printed in this part is repeated there beside the gates that decide whether it may be
read.

In [ ]:
#@title 4.0  Bookkeeping for the fits
banner("PART 4, POSTERIOR INFERENCE")

if MODE == "preflight":
    raise RuntimeError(
        "PREFLIGHT COMPLETE, and this stop is the end of it rather than a failure.\n"
        "Everything that can be checked without sampling has been: the design against "
        "tab:hfModels and tab:branchSites, the priors against tab:bayesDecisions, the prior "
        "predictive of all five families, the collected design against what these models need, "
        "the log-likelihood reconstruction against PyMC, the PSIS port against ArviZ, and "
        "parameter recovery on the real design axes.\n"
        "Set MODE to 'smoke' for a fast end-to-end rehearsal that cannot produce a verdict, or "
        "to 'full' for the reportable run.")

FITS: dict[str, object] = {}
FIT_SPECS: dict[str, Spec] = {}
FIT_OPTIONS: dict[str, dict] = {}
ESTIMANDS: list[dict] = []
CLAIM_PROBS: list[dict] = []


def register(name: str, spec: Spec, options: dict, idata) -> None:
    FITS[name] = idata
    FIT_SPECS[name] = spec
    FIT_OPTIONS[name] = dict(options)


def report(idata, variable: str, transform=None, label: str = "", fit: str = "") -> dict:
    values = idata.posterior[variable].values.ravel()
    if transform is not None:
        values = transform(values)
    low, high = np.quantile(values, [0.025, 0.975])
    row = {"fit": fit, "parameter": label or variable, "median": float(np.median(values)),
           "low": float(low), "high": float(high), "sd": float(values.std())}
    print(f"  {row['parameter']:<34s} median {row['median']:+.4f}   "
          f"95% CrI [{row['low']:+.4f}, {row['high']:+.4f}]")
    ESTIMANDS.append(row)
    return row


def probability(idata, variable: str, predicate) -> float:
    return float(np.mean(predicate(idata.posterior[variable].values.ravel())))


def claim(claim_name: str, model: str, parameter: str, p_support: float, p_refute: float,
          rule: str, fits: tuple[str, ...], loo_group: str | None = None,
          identified: bool = True, reason: str = "") -> None:
    CLAIM_PROBS.append({"claim": claim_name, "model": model, "parameter": parameter,
                        "p_support": float(p_support), "p_refute": float(p_refute),
                        "rule": rule, "fits": fits, "loo_group": loo_group,
                        "identified": bool(identified), "identification_note": reason})

## 4.1, Model A, H1 at the behavioural level, and M1

**Estimand.** $\bar\beta$, the population log-ratio of forecast amplitude recovery at a candidate
frequency against its matched controls. $e^{\bar\beta}<1$ means the candidate recovers *less*, which
is what H1 predicts.

**Reported.** $e^{\bar\beta}$ with a 95 per cent credible interval; $\Pr(\bar\beta<\log 0.8)$, the
probability of at least a fifth of the recovery being lost; and $\Pr(|\bar\beta|<\log 1.1)$, the
probability that the effect is practically nil, which is the refutation leg. The per-configuration
$\beta_c$ are shown as a forest plot, because H1 is not a claim that the effect is uniform across
geometries.

**M1** is the slope $\delta_O$ of the same fit, so its posterior comes from the same chains on the
same rows. Its rule has two legs, and the second is not decoration: a sign probability starts at
0.50 under a symmetric prior, so $\Pr(\delta_O>0)\ge 0.95$ alone would let a slope that is tiny but
consistently signed count as mitigation. The model carrying the overlap term must also win the
leave-one-out comparison against the same model without it, which is the patch-only fit below.

Four configuration levels are fitted: both covariates, overlap only, patch size only, and neither.
The pair that decides M1 is **both against patch-only**, since that is the comparison in which the
overlap term is the only difference. The four-way ranking is reported as description.

In [ ]:
#@title 4.1  Model A: H1 behavioural, and the configuration level that carries M1
A_VARIANTS = {
    "both": {},
    "overlap": {"patch": False},
    "patch": {"overlap": False},
    "none": {"overlap": False, "patch": False},
}

if FIT_MODEL_A:
    for level, options in A_VARIANTS.items():
        with PROGRESS.task(f"fit_A_{level}"):
            register(f"A_{level}", SPECS["A"], options,
                     sample_blocked(SPECS["A"], f"04_A_{level}", options))

    idata_A = FITS["A_both"]
    print("\nModel A, H1 behavioural and M1")
    report(idata_A, "beta_bar", label="beta_bar  (log recovery ratio)", fit="A_both")
    report(idata_A, "recovery_ratio", label="exp(beta_bar)  (recovery ratio)", fit="A_both")
    report(idata_A, "delta_O", label="delta_O  (overlap slope, M1)", fit="A_both")
    report(idata_A, "delta_P", label="delta_P  (log patch-size slope)", fit="A_both")
    report(idata_A, "tau", label="tau  (spread between geometries)", fit="A_both")

    pA_support = probability(idata_A, "beta_bar", lambda x: x < LOG08)
    pA_refute = probability(idata_A, "beta_bar", lambda x: np.abs(x) < LOG11)
    pA_negative = probability(idata_A, "beta_bar", lambda x: x < 0)
    pM1_support = probability(idata_A, "delta_O", lambda x: x > 0)
    print(f"\n  P(beta_bar < log 0.8 | D) = {pA_support:.3f}     (H1 support leg)")
    print(f"  P(|beta_bar| < log 1.1 | D) = {pA_refute:.3f}     (H1 refutation leg)")
    print(f"  P(beta_bar < 0 | D)       = {pA_negative:.3f}     (direction only)")
    print(f"  P(delta_O > 0 | D)        = {pM1_support:.3f}     (M1 first leg)")

    claim("H1_behavioural", "A", "exp(beta_bar)", pA_support, pA_refute,
          "Pr(beta_bar < log 0.8) >= 0.95; refuted when Pr(|beta_bar| < log 1.1) >= 0.95",
          ("A_both",))
    claim("M1", "A'", "delta_O", pM1_support, 1 - pM1_support,
          "Pr(delta_O > 0) >= 0.95 and the overlap term wins LOO against patch-only",
          ("A_both", "A_patch"), loo_group="M1")

    beta_posterior = idata_A.posterior["beta"]
    levels = [str(value) for value in beta_posterior.coords["config"].values]
    flat = beta_posterior.transpose("chain", "draw", "config").values.reshape(-1, len(levels))
    low, median, high = np.quantile(flat, [0.025, 0.5, 0.975], axis=0)
    order = np.argsort(GEOM.set_index("model").loc[levels, "overlap"].to_numpy())

    figure, axes = plt.subplots(1, 2, figsize=(13.5, 4.0),
                               gridspec_kw={"width_ratios": [1.15, 1]})
    y = np.arange(len(levels))
    axes[0].hlines(y, low[order], high[order], color=INK["primary"], lw=2.4)
    axes[0].plot(median[order], y, "o", color=INK["primary"], ms=5, zorder=3)
    axes[0].axvline(0, color=INK["line"], lw=1)
    axes[0].axvline(LOG08, color=INK["bad"], ls="--", lw=1, label="log 0.8")
    axes[0].set_yticks(y)
    axes[0].set_yticklabels([f"{levels[i]}  O={GEOM.set_index('model').overlap[levels[i]]:.2f}"
                             for i in order], fontsize=7)
    axes[0].invert_yaxis()
    axes[0].set(title=r"Model A: configuration effects $\beta_c$, ordered by overlap",
                xlabel=r"$\beta_c$")
    axes[0].legend(fontsize=7)

    x_overlap = GEOM.set_index("model").loc[levels, "x_overlap"].to_numpy()
    axes[1].errorbar(x_overlap, median, yerr=[median - low, high - median], fmt="o",
                     color=INK["primary"], ms=5, capsize=3, lw=1.2, label=r"$\beta_c$")
    grid = np.linspace(x_overlap.min() - 0.1, x_overlap.max() + 0.1, 60)
    beta_bar_draws = idata_A.posterior["beta_bar"].values.ravel()
    delta_O_draws = idata_A.posterior["delta_O"].values.ravel()
    lines = beta_bar_draws[:, None] + delta_O_draws[:, None] * grid[None, :]
    band = np.quantile(lines, [0.025, 0.5, 0.975], axis=0)
    axes[1].fill_between(grid, band[0], band[2], color=INK["accent"], alpha=0.18,
                         label="the M1 slope, 95%")
    axes[1].plot(grid, band[1], color=INK["accent"], lw=2)
    axes[1].axhline(0, color=INK["line"], lw=1)
    axes[1].set(title=r"M1: the configuration level against centred overlap",
                xlabel=r"$\widetilde O_c$", ylabel=r"$\beta_c$")
    axes[1].legend(fontsize=7)
    figure.tight_layout()
    savefig(figure, "P4_A.png")
    plt.show()
    note("The right panel is what M1 reads. The slope is estimated across fifteen independently "
         "trained variants, so it is evidence that the effect varies with geometry in the "
         "predicted direction across this grid, and not the effect of changing the overlap of a "
         "trained model. Appendix B states that limit and it is not removable by any fit.", "info")
else:
    print("FIT_MODEL_A is False: Model A is skipped in this session")

## 4.2, Model B, H1 at the representational level

**Estimand.** $\theta_{\mathrm{lock}}$, the log factor by which the frequency-local prequential
codelength expands at a candidate frequency. $e^{\theta_{\mathrm{lock}}}>1$ means the representation
needs *more bits* to separate a candidate frequency from its neighbours, which is information loss
in the sense of Voita and Titov, whether or not the forecast happens to recover the tone.

This is the half of the framework that can disagree with Model A, and the disagreement would be the
finding: Model A asks whether the forecast rebuilds the tone, Model B whether the frequency is still
readable inside the network. A candidate that is rebuilt but no longer distinguishable from its
neighbours appears here and nowhere else.

The unadjusted form, Eq. (10) with the stage and geometry offsets removed, is fitted alongside and
compared by LOO, so the blocking decision is reported rather than assumed. Codelengths differ
several-fold between probe stages, and without the offsets that variation is available to the lock
indicator.

In [ ]:
#@title 4.2  Model B: H1 representational
if FIT_MODEL_B:
    for label, options in (("adjusted", {}), ("unadjusted", {"adjusted": False})):
        with PROGRESS.task(f"fit_B_{label}"):
            register(f"B_{label}", SPECS["B"], options,
                     sample_blocked(SPECS["B"], f"04_B_{label}", options))

    idata_B = FITS["B_adjusted"]
    print("\nModel B, H1 representational")
    report(idata_B, "theta_lock", label="theta_lock  (log codelength ratio)", fit="B_adjusted")
    report(idata_B, "codelength_ratio", label="exp(theta_lock)  (codelength ratio)",
           fit="B_adjusted")
    report(idata_B, "r", label="r  (Gamma shape)", fit="B_adjusted")

    pB_support = probability(idata_B, "theta_lock", lambda x: x > LOG12)
    pB_refute = probability(idata_B, "theta_lock", lambda x: np.abs(x) < LOG11)
    print(f"\n  P(theta_lock > log 1.2 | D) = {pB_support:.3f}     (support leg)")
    print(f"  P(|theta_lock| < log 1.1 | D) = {pB_refute:.3f}     (refutation leg)")
    claim("H1_representational", "B", "exp(theta_lock)", pB_support, pB_refute,
          "Pr(theta_lock > log 1.2) >= 0.95; refuted when Pr(|theta_lock| < log 1.1) >= 0.95",
          ("B_adjusted",), loo_group="B_blocking")

    figure, axes = plt.subplots(1, 2, figsize=(12.5, 3.4))
    draws = idata_B.posterior["theta_lock"].values.ravel()
    axes[0].hist(draws, bins=80, density=True, color=INK["accent"], alpha=0.85,
                 label="posterior")
    axes[0].plot(np.linspace(-1.5, 1.5, 200),
                 stats.norm.pdf(np.linspace(-1.5, 1.5, 200), 0, 0.5), ls="--",
                 color=INK["muted"], label="prior")
    axes[0].axvline(LOG12, color=INK["bad"], ls="--", label="log 1.2")
    axes[0].axvspan(-LOG11, LOG11, color=INK["good"], alpha=0.12, label="refutation ROPE")
    axes[0].set(title=r"$\theta_{lock}$: prior against posterior", xlabel=r"$\theta_{lock}$")
    axes[0].legend(fontsize=7)

    stage_effects = idata_B.posterior["u_stage"]
    stage_levels = [str(v) for v in stage_effects.coords["stage"].values]
    stage_flat = stage_effects.transpose("chain", "draw", "stage").values.reshape(
        -1, len(stage_levels))
    s_low, s_med, s_high = np.quantile(stage_flat, [0.025, 0.5, 0.975], axis=0)
    y = np.arange(len(stage_levels))
    axes[1].hlines(y, s_low, s_high, color=INK["primary"], lw=2.2)
    axes[1].plot(s_med, y, "o", color=INK["primary"], ms=4)
    axes[1].axvline(0, color=INK["line"], lw=1)
    axes[1].set_yticks(y)
    axes[1].set_yticklabels(stage_levels, fontsize=6)
    axes[1].invert_yaxis()
    axes[1].set(title="probe-stage offsets (sum-to-zero)", xlabel="log codelength offset")
    figure.tight_layout()
    savefig(figure, "P4_B.png")
    plt.show()
else:
    print("FIT_MODEL_B is False: Model B is skipped in this session")

## 4.3, Model C, H2, is the deficit set by the geometry or by the phase?

**Estimand.** $\sigma_\phi$, the spread of the per-phase offsets. The phase circle is cut into eight
equal slices, each slice gets its own offset, and $\sigma_\phi$ says how far apart those offsets
are, that is, how much the deficit moves as the signal slides through its cycle. Slices are
preferred to a fitted sinusoid because they presuppose no shape: any systematic pattern around the
circle, of whatever form, widens the spread.

**Two readings are reported, and they can disagree.** The magnitude reading is
$\Pr(\sigma_\phi<\log 1.1)$, whether the dependence is practically negligible. The evidential
reading is the LOO comparison against the same model with the phase term removed, whether allowing
phase to matter predicts held-out data better. A small but consistently detected dependence would
show a low equivalence probability together with a LOO preference for the phase model. The rule of
`tab:bayesDecisions` is the magnitude reading alone, so a LOO preference does not by itself deny H2;
it is reported beside the verdict.

Appendix B adds one condition that is easy to overlook: where the population effect is itself
indistinguishable from zero, little remains for phase to move, so H2 is read **conditionally**. The
deficit scale $\bar\beta$ of this same fit is printed for exactly that reason.

In [ ]:
#@title 4.3  Model C: H2 phase invariance
if FIT_MODEL_C:
    for label, options in (("phase", {}), ("nophase", {"phase": False})):
        with PROGRESS.task(f"fit_C_{label}"):
            register(f"C_{label}", SPECS["C"], options,
                     sample_blocked(SPECS["C"], f"04_C_{label}", options))

    idata_C = FITS["C_phase"]
    print("\nModel C, H2 phase invariance")
    report(idata_C, "sigma_phase", label="sigma_phase  (the estimand)", fit="C_phase")
    report(idata_C, "sd_u_phase", label="sd of the eight realised offsets", fit="C_phase")
    report(idata_C, "beta_bar", label="beta_bar  (the deficit scale)", fit="C_phase")

    pC_support = probability(idata_C, "sigma_phase", lambda x: x < LOG11)
    print(f"\n  P(sigma_phase < log 1.1 | D) = {pC_support:.3f}")
    claim("H2", "C", "sigma_phase", pC_support, 1 - pC_support,
          "Pr(sigma_phase < log 1.1) >= 0.95", ("C_phase",), loo_group="H2")

    offsets = idata_C.posterior["u_phase"]
    flat = offsets.transpose("chain", "draw", "phasebin").values.reshape(-1, N_PHASE_BINS)
    low, median, high = np.quantile(flat, [0.025, 0.5, 0.975], axis=0)
    centres = (np.arange(N_PHASE_BINS) + 0.5) * (2 * np.pi / N_PHASE_BINS)

    figure, axes = plt.subplots(1, 2, figsize=(12.5, 3.4))
    observed = (CONTRASTS.groupby("phase_bin").y_deficit.mean()
                .reindex(range(N_PHASE_BINS)))
    axes[0].plot(centres, observed.to_numpy() - observed.mean(), "s", ms=5,
                 color=INK["muted"], label="observed deficit, centred")
    axes[0].errorbar(centres, median, yerr=[median - low, high - median], fmt="o",
                     color=INK["bad"], capsize=3, label=r"$u_p$ with 95% CrI")
    axes[0].axhline(0, color=INK["line"], lw=0.9)
    axes[0].set(title="the eight per-phase offsets", xlabel=r"phase $\phi$ [rad]",
                ylabel="offset")
    axes[0].legend(fontsize=7)

    posterior = idata_C.posterior["sigma_phase"].values.ravel()
    axes[1].hist(posterior, bins=80, density=True, color=INK["bad"], alpha=0.8,
                 label="posterior")
    axes[1].hist(SIGMA_PHASE_PRIOR, bins=80, density=True, histtype="step",
                 color=INK["muted"], label="prior")
    axes[1].axvline(LOG11, color=INK["line"], ls="--", label=r"ROPE edge $\log 1.1$")
    axes[1].set(title=r"$\sigma_\phi$: prior against posterior", xlabel=r"$\sigma_\phi$")
    axes[1].legend(fontsize=7)
    figure.tight_layout()
    savefig(figure, "P4_C.png")
    plt.show()
else:
    print("FIT_MODEL_C is False: Model C is skipped in this session")

## 4.4, Model D1, H3, where the degradation sites are

**Estimand.** Not an effect size but a location law: is the token dispersion systematically lower on
one of the two predicted grids than off it? Each swept frequency carries two binary labels and four
labellings are fitted to the same profiles and compared by LOO, the two-label model H3 states and the
three alternatives, stride only, patch only and neither.

H3 predicts $\theta_S<0$ **and** $\theta_P<0$, so the support probability is the probability of the
**joint** event and not of two marginals each above the cutoff. Two marginals at 0.96 can accompany a
joint probability well below it if the two are negatively correlated in the posterior, and the
hypothesis is the conjunction.

The comparison is meaningful only because the sweep grid is the union over every geometry, so each
labelling is scored where its rival predicts a dip as well as where it does. It is identified only
because the design contains configurations with $S \nmid P$: on the $P=S$ diagonal the two combs
coincide by construction and no amount of data can separate them. Part 2.5 checked that rank.

Each generator is fitted separately and the verdict takes the **less favourable** of the two, because
a location law that held on one corpus and not on the other would not be a property of the
tokenisation. The one-hertz labelling is fitted as a declared robustness reading; it is expected to
attenuate both coefficients, because it also labels the integer neighbours of the non-integer sites,
which lie on no comb.

In [ ]:
#@title 4.4  Model D1: H3 site location
D1_LABELLINGS = {
    "both": {},
    "stride": {"use_p": False},
    "patch": {"use_s": False},
    "none": {"use_s": False, "use_p": False},
}

if FIT_MODEL_D1:
    with PROGRESS.task("fit_D1"):
        for mode in D1_MODES:
            spec = SPECS[f"D1_{mode}"]
            for labelling, options in D1_LABELLINGS.items():
                register(f"D1_{mode}_{labelling}", spec, options,
                         sample_blocked(spec, f"04_D1_{mode}_{labelling}", options))
    with PROGRESS.task("fit_D1_alt_tol"):
        for mode in VERDICT_MODES:
            spec = SPECS[f"D1_{mode}"]
            options = {"alt_tolerance": True}
            register(f"D1_{mode}_both_alt", spec, options,
                     sample_blocked(spec, f"04_D1_{mode}_both_alt", options))

    print("\nModel D1, H3 site location")
    d1_rows = []
    for mode in D1_MODES:
        for suffix, note_text in (("both", "exact grid labels"),
                                  ("both_alt", f"{GRID_TOL_ALT_HZ:g} Hz labels, robustness")):
            fit_name = f"D1_{mode}_{suffix}"
            if fit_name not in FITS:
                continue
            idata = FITS[fit_name]
            theta_S = idata.posterior["theta_S"].values.ravel()
            theta_P = idata.posterior["theta_P"].values.ravel()
            joint = float(np.mean((theta_S < 0) & (theta_P < 0)))
            d1_rows.append({
                "mode": mode, "labelling": note_text,
                "in_verdict": mode in VERDICT_MODES,
                "theta_S median": float(np.median(theta_S)),
                "theta_S low": float(np.quantile(theta_S, 0.025)),
                "theta_S high": float(np.quantile(theta_S, 0.975)),
                "theta_P median": float(np.median(theta_P)),
                "theta_P low": float(np.quantile(theta_P, 0.025)),
                "theta_P high": float(np.quantile(theta_P, 0.975)),
                "P(theta_S<0)": float(np.mean(theta_S < 0)),
                "P(theta_P<0)": float(np.mean(theta_P < 0)),
                "P(both<0) joint": joint})
            if suffix == "both":
                report(idata, "theta_S", label=f"theta_S  {mode}", fit=fit_name)
                report(idata, "theta_P", label=f"theta_P  {mode}", fit=fit_name)
    D1_TABLE = save_table(pd.DataFrame(d1_rows), "04_D1_coefficients.parquet")
    show(D1_TABLE, caption="the joint probability is the one H3 is read from; the marginals are "
                           "shown because they can both exceed it")

    primary = D1_TABLE[(D1_TABLE.labelling == "exact grid labels") & D1_TABLE.in_verdict]
    pD1_support = float(primary["P(both<0) joint"].min())
    pD1_refute = float(max(1 - primary["P(theta_S<0)"].min(), 1 - primary["P(theta_P<0)"].min()))
    print(f"\n  joint P(theta_S < 0 and theta_P < 0 | D) = {pD1_support:.3f}  "
          f"(the less favourable generator)")
    claim("H3_location", "D1", "theta_S < 0 and theta_P < 0", pD1_support, pD1_refute,
          "the two-label fit wins LOO and both coefficients are negative on both generators",
          tuple(f"D1_{mode}_{labelling}" for mode in VERDICT_MODES
                for labelling in D1_LABELLINGS),
          loo_group="H3_location")

    figure, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    width = 0.34
    positions = np.arange(len(primary))
    for offset, (parameter, colour) in enumerate((("theta_S", INK["grid_s"]),
                                                 ("theta_P", INK["grid_p"]))):
        medians = primary[f"{parameter} median"].to_numpy()
        lows = primary[f"{parameter} low"].to_numpy()
        highs = primary[f"{parameter} high"].to_numpy()
        axes[0].errorbar(positions + (offset - 0.5) * width, medians,
                         yerr=[medians - lows, highs - medians], fmt="o", color=colour,
                         capsize=4, ms=6, label=parameter)
    axes[0].axhline(0, color=INK["line"], lw=1)
    axes[0].set_xticks(positions)
    axes[0].set_xticklabels(primary["mode"])
    axes[0].set(title="D1: dip depth on each grid, by generator",
                ylabel="log depth relative to off-grid")
    axes[0].legend(fontsize=8)

    idata = FITS[f"D1_{VERDICT_MODES[0]}_both"]
    theta_S = idata.posterior["theta_S"].values.ravel()
    theta_P = idata.posterior["theta_P"].values.ravel()
    axes[1].scatter(theta_S, theta_P, s=3, alpha=0.12, color=INK["primary"], linewidths=0)
    axes[1].axvline(0, color=INK["line"], lw=1)
    axes[1].axhline(0, color=INK["line"], lw=1)
    axes[1].fill_betweenx([theta_P.min(), 0], theta_S.min(), 0, color=INK["good"], alpha=0.10)
    axes[1].set(title=f"the joint event H3 claims, {VERDICT_MODES[0]}\n"
                      f"shaded quadrant holds {100 * np.mean((theta_S < 0) & (theta_P < 0)):.1f}% "
                      f"of the posterior",
                xlabel=r"$\theta_S$", ylabel=r"$\theta_P$")
    figure.tight_layout()
    savefig(figure, "P4_D1.png")
    plt.show()
else:
    print("FIT_MODEL_D1 is False: Model D1 is skipped in this session")

## 4.5, Model D2, H3a and H3b, whether the sites move

Part 4.4 says where the dips sit. It does not yet say they **move**: each geometry's grid is fixed, so
a model that fits that grid well is still consistent with dips that merely happen to coincide with
it. H3's claim is a law across geometries, and it is two laws, one per branch.

$$\hat f^{\,F}_{1,g} \sim \mathcal N\!\left(\kappa_F\,\Delta^F_g,\ \sqrt{\sigma_F^2+\Delta f^2}\right),
\qquad \Delta^S_g=f_s/S_g,\quad \Delta^P_g=f_s/P_g.$$

The residual carries the floor $\Delta f$, the sweep step, because a spacing read off a discrete grid
is no more precise than one step. Without it, sites landing exactly on the prediction drive
$\sigma_F$ to zero and the posterior develops a funnel the sampler cannot traverse; with it a
near-zero $\sigma_F$ is a result rather than a sampler failure.

**Two limitations of this measurement are reported rather than adjusted for, and both are real.**

*Conditional on the selection.* Dips are detected from the profile alone, which uses neither $P$ nor
$S$, but each detected dip is then **assigned** to the branch whose grid it is near. Regressing the
fundamental of a branch on that branch's predicted spacing is therefore conditional on a selection
that used the same spacing. This is what Appendix B specifies and it is what is fitted; it means
$\kappa_F\approx1$ says that, given a dip was detected and fell near this branch's comb, its
position equals the predicted spacing, which is weaker than an unconditional measurement of the comb.

*The lowest site need not be the fundamental.* If the first dip of a comb is not detected, the lowest
detected site is a higher harmonic, and $\hat f_1/\Delta$ is then near an integer above one rather
than near one. Dividing by that integer to recover a slope of one would be fitting the answer. The
audit below reports $\mathrm{round}(\hat f_1/\Delta)$ per geometry and branch, and a second fit
restricted to the geometries whose lowest site **is** the fundamental is reported beside the
preregistered one as a declared robustness reading.

Both responses of Appendix B are fitted: the fundamental, which is the specified one, and the median
gap between successive sites, as a robustness check. With three or four sites in band one spurious
detection inserts a short interval and flips the median, to which the fundamental is immune, so a
disagreement between them is itself informative about the stability of the detection.

In [ ]:
#@title 4.5  Model D2: the harmonic-order audit, then H3a and H3b
mean_sites = SITES[(SITES.rep == -1) & SITES["mode"].isin(VERDICT_MODES)].copy()
if "first_harmonic" not in mean_sites:
    raise RuntimeError(
        "this collection predates the harmonic-order audit in collect.derive_sites. Re-merge it "
        "with `python collect.py --out <data dir> --merge-only --response contrast`, which "
        "rebuilds the sites table from the existing shards without a single forward pass.")

# One row per geometry and branch, pooling the two generator modes. The order is read from the
# column derive_sites computes rather than recomputed here, so the notebook and the collection
# cannot disagree about which harmonic a detected site is.
D2_AUDIT = (mean_sites
            .groupby(["model", "branch"], as_index=False)
            .agg(P=("P", "first"), S=("S", "first"),
                 predicted_spacing=("predicted_spacing", "first"),
                 f1=("f1", "median"), n_sites=("n_sites", "sum"),
                 harmonic_order=("first_harmonic", "max"),
                 harmonic_order_min=("first_harmonic", "min"),
                 modes=("mode", "nunique")))
D2_AUDIT["status"] = np.where(
    D2_AUDIT.harmonic_order == 0, "no site detected",
    np.where(D2_AUDIT.harmonic_order_min != D2_AUDIT.harmonic_order,
             "the two generators disagree about which harmonic the lowest site is",
             np.where(D2_AUDIT.harmonic_order == 1, "lowest site is the fundamental",
                      "lowest site is a higher harmonic; the fundamental was not detected")))
D2_AUDIT = save_table(D2_AUDIT, "04_D2_site_audit.parquet")
show(D2_AUDIT, caption="the harmonic-order audit on the replicate-averaged curves, pooling both "
                       "generators; harmonic_order is the collected first_harmonic column")

# A geometry qualifies for the restricted fit only when its lowest detected site is the
# fundamental on EVERY fitted generator, so a disagreement between the two excludes it.
FUNDAMENTAL_MODELS = {
    branch: set(D2_AUDIT.loc[(D2_AUDIT.branch == branch)
                             & (D2_AUDIT.harmonic_order == 1)
                             & (D2_AUDIT.harmonic_order_min == 1), "model"])
    for branch in ("stride", "patch")}
for branch, models in FUNDAMENTAL_MODELS.items():
    print(f"  {branch}: the lowest detected site is the fundamental on both generators in "
          f"{len(models)} of {len(TAGS)} geometries")

if FIT_MODEL_D2:
    with PROGRESS.task("fit_D2"):
        for name, spec in list(SPECS.items()):
            if not name.startswith("D2_"):
                continue
            register(name, spec, {}, sample_blocked(spec, f"04_{name}", {}))
            branch = name.split("_")[1]
            restricted = spec.frame.model.isin(FUNDAMENTAL_MODELS[branch]).to_numpy()
            if restricted.sum() >= 3 and restricted.sum() < len(spec.frame):
                sub = subset_spec(spec, np.flatnonzero(restricted))
                register(f"{name}_fundamental", sub, {},
                         sample_blocked(sub, f"04_{name}_fundamental", {}))

    print("\nModel D2, H3a and H3b")
    d2_rows = []
    for name in [n for n in FITS if n.startswith("D2_")]:
        branch = name.split("_")[1]
        idata = FITS[name]
        kappa = idata.posterior["kappa"].values.ravel()
        d2_rows.append({
            "fit": name, "branch": branch,
            "response": "f1" if "_f1" in name else "delta_hat",
            "subset": "fundamental only" if name.endswith("_fundamental") else "all detected",
            "observations": len(FIT_SPECS[name].frame),
            "kappa median": float(np.median(kappa)),
            "kappa low": float(np.quantile(kappa, 0.025)),
            "kappa high": float(np.quantile(kappa, 0.975)),
            "P(|kappa-1|<0.1)": float(np.mean(np.abs(kappa - 1) < ROPE_SLOPE)),
            "identified": bool(FIT_SPECS[name].identified)})
    D2_TABLE = save_table(pd.DataFrame(d2_rows), "04_D2_coefficients.parquet")
    show(D2_TABLE, caption="the preregistered fit is response f1 on all detected sites; the other "
                           "rows are the declared robustness readings")

    for branch, claim_name in (("stride", "H3a"), ("patch", "H3b")):
        primary_name = f"D2_{branch}_f1"
        identified = D2_IDENTIFICATION[branch]["identified"]
        if primary_name not in FITS:
            claim(claim_name, "D2", "kappa", np.nan, np.nan,
                  f"Pr(|kappa - 1| < {ROPE_SLOPE}) >= 0.95", tuple(), identified=False,
                  reason="no replicate-level sites for this branch")
            continue
        report(FITS[primary_name], "kappa", label=f"kappa_{branch[0].upper()}  (primary, f1)",
               fit=primary_name)
        support = probability(FITS[primary_name], "kappa",
                             lambda x: np.abs(x - 1) < ROPE_SLOPE)
        print(f"  P(|kappa_{branch[0].upper()} - 1| < {ROPE_SLOPE} | D) = {support:.3f}")
        claim(claim_name, "D2", f"kappa_{branch[0].upper()}", support, 1 - support,
              f"Pr(|kappa - 1| < {ROPE_SLOPE}) >= 0.95, after the identification gate",
              (primary_name,), identified=identified,
              reason="" if identified else
              (f"{D2_IDENTIFICATION[branch]['unambiguous_sites']} unambiguous sites against a bar "
               f"of {MIN_D2_SITES}"))

    figure, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for axis, branch in zip(axes, ("stride", "patch")):
        name = f"D2_{branch}_f1"
        if name not in FITS:
            axis.axis("off")
            continue
        spec = FIT_SPECS[name]
        kappa = FITS[name].posterior["kappa"].values.ravel()
        sigma_F = FITS[name].posterior["sigma_F"].values.ravel()
        axis.scatter(spec.arrays["x"], spec.arrays["y"], s=18, alpha=0.6,
                     color=INK["primary"], label="detected fundamental")
        grid = np.linspace(0, float(np.max(spec.arrays["x"])) * 1.08, 60)
        lines = kappa[:, None] * grid[None, :]
        band = np.quantile(lines, [0.025, 0.5, 0.975], axis=0)
        axis.fill_between(grid, band[0], band[2], color=INK["accent"], alpha=0.2,
                          label=r"$\kappa_F \Delta^F$, 95%")
        axis.plot(grid, band[1], color=INK["accent"], lw=2)
        axis.plot(grid, grid, ls="--", color=INK["line"], lw=1.2, label=r"$\kappa_F=1$")
        axis.set(title=f"D2 {branch} branch: "
                       f"{'identified' if D2_IDENTIFICATION[branch]['identified'] else 'NOT identified'}"
                       f"  ({D2_IDENTIFICATION[branch]['unambiguous_sites']} sites)",
                 xlabel=r"predicted spacing $\Delta^F$ [Hz]",
                 ylabel=r"measured $\hat f_1$ [Hz]")
        axis.legend(fontsize=7)
    figure.tight_layout()
    savefig(figure, "P4_D2.png")
    plt.show()

    if any(order > 1 for order in D2_AUDIT.harmonic_order):
        note("At least one geometry's lowest detected site is a higher harmonic of its branch's "
             "comb, so its fundamental was not detected. Those rows pull kappa above one and the "
             "restricted fit in the table above is what isolates the effect. Neither reading is "
             "corrected by dividing by the harmonic order, because that would impose the slope "
             "the model is estimating.", "warn")
else:
    print("FIT_MODEL_D2 is False: Model D2 is skipped in this session")

save_table(pd.DataFrame(ESTIMANDS), "04_estimands.parquet")
show(pd.DataFrame(CLAIM_PROBS).drop(columns=["fits"], errors="ignore"),
     caption="the probability of each rule, before any gate is applied")

---
# Part 5, Checks, and only then verdicts

A posterior is a conditional statement: *given* that the sampler converged, that the model can
reproduce the data, that the design could have seen the effect, and that the answer does not hinge on
an arbitrary prior scale. This part tests all four and only then states the verdicts.

| Section | Gate | Threshold | Applies to |
|---|---|---|---|
| 5.1 | convergence | $\hat R<1.01$, bulk and tail ESS $>1000$, zero divergences, over every parameter | every fit the claim reads |
| 5.2 | posterior predictive | the observed statistic inside the 95 per cent replicated interval for at least 90 per cent of the levels of every stratum | every fit the claim reads |
| 5.3 | prior sensitivity | each rule probability moving by at most 0.10 over the ladder, with the same decision class throughout | every claim |
| 5.4 | LOO reliability | every Pareto $k$ below ArviZ's threshold, and a win declared only at twice the standard error of the difference | **only M1 and H3 location**, whose rules have a comparison leg |
| 5.5 | the verdicts | pre-gate reading, each gate separately, post-gate verdict | every claim |

Two of those rows are easy to get wrong and both were, in an earlier draft of this notebook. A gate
that names one representative fit passes a claim whose other fit failed, so **every gate is a
conjunction over every fit the claim is read from**: H3 location pools both generators and takes the
less favourable, so a generator whose model fails its predictive check fails the gate. And a
comparison that the rule does not mention must not decide the verdict, so for H2 and for Model B's
blocking the leave-one-out reading is reported beside the result and marked not applicable as a
gate, which is what `tab:bayesDecisions` specifies.

The first three thresholds in that table are the ones the document preregisters. The 90 per cent
predictive coverage, the 0.10 sensitivity band and the 80 per cent recovery coverage of Part 3.7 are
operational thresholds declared by this notebook; they are recorded in `analysis_manifest.json` and
they are not attributed to the report.

Interpretation is fail-closed. A gate that fails does not weaken a claim into a hint: the estimand is
reported as **NOT REPORTABLE**, and a D2 branch below its site bar as **NOT IDENTIFIED**, neither of
which is a null result.

## 5.1, Convergence

$\hat R$ measures agreement between the four chains, and values near one indicate that each reached
the same distribution. Bulk and tail effective sample size count the genuinely independent draws left
after autocorrelation, the tail figure guarding the interval endpoints the credible intervals are read
from. Zero divergences requires that the sampler met no geometry it could not traverse.

The three are read over **every** parameter of a fit, not only the estimand. A population effect can
mix perfectly while a group scale does not, and the group scale is part of the model the estimand is
conditional on.

One diagnostic pattern is worth naming, because it is what the earlier runs of this project hit: zero
divergences together with a very low effective sample size is a **ridge**, not a funnel. A funnel
announces itself through divergences; a ridge is a direction along which parameters trade off
exactly, which the sampler traverses slowly without ever failing a step. Part 3.2's sum-to-zero
constraint is the fix for the ridge these models had, and the table below is where that either shows
or does not.

In [ ]:
#@title 5.1  The convergence gate
banner("PART 5, CHECKS")

_CONVERGENCE_COLUMNS = ["fit", "max_rhat", "min_ess_bulk", "min_ess_tail", "divergences",
                        "chains", "diagnostics_ok", "reason", "observations", "options"]
CONVERGENCE = save_table(
    pd.DataFrame(FIT_LOG) if FIT_LOG else pd.DataFrame(columns=_CONVERGENCE_COLUMNS),
    "05_convergence.parquet")
if not FIT_LOG:
    raise RuntimeError(
        "no model was fitted or reloaded in this session, so there is nothing to check. Turn on "
        "at least one FIT_MODEL_* flag in Part 0.4, or point RUN_FOLDER at a finished run.")
show(CONVERGENCE.drop(columns=["options", "reason"], errors="ignore"),
     caption=f"thresholds: R-hat < {RHAT_MAX}, bulk and tail ESS > {ESS_MIN}, "
             f"divergences <= {MAX_DIVERGENCES}")

_converged = dict(zip(CONVERGENCE.fit, CONVERGENCE.diagnostics_ok))


def converged(*names: str) -> bool:
    """Whether every named fit passed the convergence gate."""
    return bool(names and all(_converged.get(name, False) for name in names))


failed_fits = CONVERGENCE.loc[~CONVERGENCE.diagnostics_ok, "fit"].tolist()
if failed_fits:
    note(f"<b>Convergence failed for {failed_fits}.</b> Every claim that reads one of these fits "
         "becomes NOT REPORTABLE in Part 5.5. The columns above say which diagnostic failed: a "
         "low effective sample size with zero divergences is a ridge, and divergences with a "
         "healthy effective sample size is a funnel.", "bad")
else:
    note("<b>Convergence: PASS for every fit.</b> Four chains, R-hat below "
         f"{RHAT_MAX}, effective sample size above {ESS_MIN} in bulk and tail, and no "
         "divergences, over every parameter of every fit.", "good")

figure, axes = plt.subplots(1, 3, figsize=(14, max(2.6, 0.22 * len(CONVERGENCE))))
y = np.arange(len(CONVERGENCE))
colours = np.where(CONVERGENCE.diagnostics_ok, INK["good"], INK["bad"])
axes[0].barh(y, CONVERGENCE.max_rhat, color=colours, alpha=0.8)
axes[0].axvline(RHAT_MAX, color=INK["line"], ls="--", lw=1)
axes[0].set(title=r"max $\hat R$", xlim=(0.995, max(1.02, CONVERGENCE.max_rhat.max() * 1.02)))
axes[1].barh(y, CONVERGENCE[["min_ess_bulk", "min_ess_tail"]].min(axis=1), color=colours,
             alpha=0.8)
axes[1].axvline(ESS_MIN, color=INK["line"], ls="--", lw=1)
axes[1].set(title="min ESS (bulk and tail)")
axes[2].barh(y, CONVERGENCE.divergences.fillna(0), color=colours, alpha=0.8)
axes[2].set(title="divergences")
for axis in axes:
    axis.set_yticks(y)
    axis.set_yticklabels(CONVERGENCE.fit, fontsize=6)
    axis.invert_yaxis()
figure.tight_layout()
savefig(figure, "P5_convergence.png")
plt.show()

## 5.2, Posterior predictive checks

A model that reproduces the pooled histogram and fails a stratum is fitting the average and missing
the structure, so the check is stratified by every factor the design varies: geometry, generator,
candidate frequency, phase slice, patch size, stride, probe stage, signal mode and grid class,
whichever of those a given model carries. Two statistics are read per level, the mean and the standard
deviation, because a model can match every level's mean while getting the dispersion wrong, and three
global quantiles are read as well.

The replicates are drawn from the same linear predictor that Part 3.5 proved against PyMC, with the
family's own sampling step applied to it, in small batches of draws, so
the full replicate array is never built. That matters for the same reason the log likelihood is never
built: a Model A replicate set at 400 draws would be 1.1 GiB, and at the 2000 draws of the full
posterior, 5.9 GiB.

This is a model-adequacy gate. It is not a hypothesis result, and passing it is not evidence for any
claim.

In [ ]:
#@title 5.2  Stratified posterior predictive checks, from the reconstruction
def ppc_check(name: str, n_draw: int = PPC_DRAWS, batch: int = PPC_BATCH) -> pd.DataFrame:
    """Observed against replicated mean and sd for every level of every stratum, plus quantiles."""
    spec, options, idata = FIT_SPECS[name], FIT_OPTIONS[name], FITS[name]
    draws = posterior_draws(idata)
    n_samples = len(next(iter(draws.values())))
    picked = np.linspace(0, n_samples - 1, min(n_draw, n_samples)).astype(int)
    y = np.asarray(spec.arrays["y"], float)
    rows_all = np.arange(len(y))
    rng = np.random.default_rng(SEED)

    plans, observed, replicated = [], {}, {}
    for column in spec.strata:
        if column not in spec.frame or spec.frame[column].isna().any():
            raise ValueError(f"PPC {name}: stratum {column!r} is missing or incomplete")
        group, levels = pd.factorize(spec.frame[column], sort=True)
        counts = np.bincount(group, minlength=len(levels)).astype(float)
        mean = np.bincount(group, weights=y, minlength=len(levels)) / counts
        variance = np.bincount(group, weights=y * y, minlength=len(levels)) / counts - mean ** 2
        sd = np.sqrt(np.maximum(0.0, variance))
        plans.append((column, group, levels, counts))
        for position, level in enumerate(levels):
            for metric, values in (("mean", mean), ("sd", sd)):
                if metric == "sd" and counts[position] < 2:
                    continue
                key = (column, str(level), metric)
                observed[key] = float(values[position])
                replicated[key] = []
    for metric, quantile in (("q05", 0.05), ("q50", 0.5), ("q95", 0.95)):
        key = ("global", "all", metric)
        observed[key] = float(np.quantile(y, quantile))
        replicated[key] = []

    for start in range(0, len(picked), batch):
        index = picked[start:start + batch]
        sample = simulate_replicates(spec, draws, options, rows_all, index, rng)
        if not np.isfinite(sample).all():
            raise ValueError(f"PPC {name}: a replicate is not finite")
        for column, group, levels, counts in plans:
            means = np.stack([np.bincount(group, weights=row, minlength=len(levels)) / counts
                              for row in sample])
            squares = np.stack([np.bincount(group, weights=row * row, minlength=len(levels))
                                / counts for row in sample])
            sds = np.sqrt(np.maximum(0.0, squares - means ** 2))
            for position, level in enumerate(levels):
                for metric, values in (("mean", means), ("sd", sds)):
                    key = (column, str(level), metric)
                    if key in replicated:
                        replicated[key].extend(values[:, position].tolist())
        for metric, quantile in (("q05", 0.05), ("q50", 0.5), ("q95", 0.95)):
            replicated[("global", "all", metric)].extend(
                np.quantile(sample, quantile, axis=1).tolist())
        del sample
        gc.collect()

    rows = []
    for key, value in observed.items():
        low, high = np.quantile(replicated[key], [0.025, 0.975])
        rows.append({"fit": name, "stratum": key[0], "level": key[1], "metric": key[2],
                     "observed": value, "rep_low": float(low), "rep_high": float(high),
                     "ppc_ok": bool(low <= value <= high)})
    return pd.DataFrame(rows)


PPC_FITS = [name for name in ("A_both", "B_adjusted", "C_phase") if name in FITS]
PPC_FITS += [f"D1_{mode}_both" for mode in D1_MODES if f"D1_{mode}_both" in FITS]
PPC_FITS += [name for name in FITS if name.startswith("D2_") and name.endswith("_f1")]

PPC_FILE = "05_ppc.parquet"
if have(PPC_FILE):
    PPC = load_df(PPC_FILE)
else:
    with PROGRESS.task("part5_ppc"):
        PPC = save_table(pd.concat([ppc_check(name) for name in PPC_FITS], ignore_index=True),
                         PPC_FILE)

PPC_RATES = (PPC.groupby(["fit", "stratum", "metric"]).ppc_ok.mean()
             .rename("coverage").reset_index())
PPC_OK = {name: bool(len(group) and (group.coverage >= PPC_COVERAGE_MIN).all())
          for name, group in PPC_RATES.groupby("fit")}
save_table(PPC_RATES, "05_ppc_rates.parquet")
show(PPC_RATES.pivot_table(index="fit", columns=["stratum", "metric"], values="coverage")
     .reset_index(), caption=f"share of levels covered, per stratum and statistic; the gate is "
                             f"{PPC_COVERAGE_MIN:.0%} on every one")
show(pd.DataFrame([{"fit": name, "ppc_ok": ok} for name, ok in PPC_OK.items()]))

worst = PPC[~PPC.ppc_ok].sort_values("fit")
if len(worst):
    show(worst.head(20), caption=f"{len(worst)} of {len(PPC)} checks fall outside the interval; "
                                 "the first twenty are listed")
if all(PPC_OK.values()):
    note("<b>Posterior predictive: PASS.</b> Every model reproduces the mean and the dispersion of "
         "every stratum of the design to within its replicated interval, at the declared rate.",
         "good")
else:
    note(f"<b>Posterior predictive failed for "
         f"{[name for name, ok in PPC_OK.items() if not ok]}.</b> The failing strata above say "
         "where the model does not reproduce the data; the claims those fits carry become NOT "
         "REPORTABLE.", "bad")

figure, axis = plt.subplots(figsize=(9, max(3, 0.34 * len(PPC_RATES))))
pivot = PPC_RATES.assign(key=PPC_RATES.stratum + " / " + PPC_RATES.metric).pivot(
    index="fit", columns="key", values="coverage")
image = axis.imshow(pivot.to_numpy(dtype=float), vmin=0, vmax=1, cmap="Blues", aspect="auto")
axis.set_xticks(np.arange(len(pivot.columns)))
axis.set_xticklabels(pivot.columns, rotation=40, ha="right", fontsize=7)
axis.set_yticks(np.arange(len(pivot.index)))
axis.set_yticklabels(pivot.index, fontsize=7)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        value = pivot.iloc[i, j]
        axis.text(j, i, "n/a" if pd.isna(value) else f"{value:.0%}", ha="center", va="center",
                  fontsize=6, color="white" if pd.notna(value) and value > 0.62 else "#222222")
axis.set_title(f"posterior predictive coverage by stratum (gate {PPC_COVERAGE_MIN:.0%})")
axis.grid(False)
figure.colorbar(image, ax=axis, label="share of levels covered")
figure.tight_layout()
savefig(figure, "P5_ppc.png")
plt.show()

## 5.3, Prior sensitivity

Every effect prior in the document is centred on no effect and its scale is varied over the
sensitivity ladder $\{0.25, 0.5, 1.0\}$, sceptical, primary and wide, "so that a conclusion resting
on the choice of scale rather than on the data is detected as such". That statement is about every
effect prior, so the ladder is applied to all five families as a **factor** on every prior scale:
$\{0.5, 1, 2\}$ times the tabulated value, which reproduces $\{0.25, 0.5, 1.0\}$ for Model A's
effect priors and $\{0.125, 0.25, 0.5\}$ for $\sigma_\phi$.

Nuisance scales move with the same factor. The two centres that are not effect sizes stay fixed:
Model B's data-informed intercept $\log\bar L$, and the Gamma shape prior, which carries the
dispersion rather than an effect.

A claim passes when all three variants converge, the rule's probability moves by at most 0.10 across
them, **and** the decision class is the same at every rung. Two converged variants do not excuse a
third that failed.

Model A's likelihood is checked at the same time. Appendix B argues for Student-$t_4$ over a Gaussian
from the shape of the response; refitting under a Gaussian reports what that argument is worth on
these data rather than leaving it as an assertion.

In [ ]:
#@title 5.3  The sensitivity ladder, on every family
SENSITIVITY_TARGETS = {
    "H1_behavioural": ("A_both", "beta_bar", lambda x: x < LOG08, lambda x: np.abs(x) < LOG11),
    "M1": ("A_both", "delta_O", lambda x: x > 0, lambda x: x < 0),
    "H1_representational": ("B_adjusted", "theta_lock", lambda x: x > LOG12,
                            lambda x: np.abs(x) < LOG11),
    "H2": ("C_phase", "sigma_phase", lambda x: x < LOG11, lambda x: x >= LOG11),
}
for mode in VERDICT_MODES:
    SENSITIVITY_TARGETS[f"H3_location:{mode}"] = (f"D1_{mode}_both", None, None, None)
for branch, claim_name in (("stride", "H3a"), ("patch", "H3b")):
    if f"D2_{branch}_f1" in FITS:
        SENSITIVITY_TARGETS[claim_name] = (
            f"D2_{branch}_f1", "kappa", lambda x: np.abs(x - 1) < ROPE_SLOPE,
            lambda x: np.abs(x - 1) >= ROPE_SLOPE)


def decision_class(p_support: float, p_refute: float) -> str:
    if not (np.isfinite(p_support) and np.isfinite(p_refute)):
        return "NOT VERIFIED"
    if p_support >= PROB:
        return "SUPPORTED"
    if p_refute >= PROB:
        return "REFUTED"
    return "INCONCLUSIVE"


def rule_probabilities(claim_name: str, idata) -> tuple[float, float]:
    """The support and refutation probabilities of one rule, on any fit of the same model."""
    if claim_name.startswith("H3_location"):
        theta_S = idata.posterior["theta_S"].values.ravel()
        theta_P = idata.posterior["theta_P"].values.ravel()
        return (float(np.mean((theta_S < 0) & (theta_P < 0))),
                float(max(np.mean(theta_S >= 0), np.mean(theta_P >= 0))))
    _, parameter, support, refute = SENSITIVITY_TARGETS[claim_name]
    draws = idata.posterior[parameter].values.ravel()
    return float(np.mean(support(draws))), float(np.mean(refute(draws)))


SENSITIVITY_FILE = "05_sensitivity.parquet"
if have(SENSITIVITY_FILE):
    SENSITIVITY = load_df(SENSITIVITY_FILE)
else:
    with PROGRESS.task("part5_sensitivity"):
        rows = []
        for claim_name, (fit_name, _, _, _) in SENSITIVITY_TARGETS.items():
            if fit_name not in FITS:
                continue
            spec, options = FIT_SPECS[fit_name], FIT_OPTIONS[fit_name]
            for factor in PRIOR_FACTORS:
                if factor == 1.0:
                    idata, label = FITS[fit_name], fit_name
                    diagnostic_ok = converged(fit_name)
                else:
                    label = f"05_sens_{fit_name}_f{factor:g}".replace(".", "p")
                    idata = sample_blocked(spec, label, {**options, "factor": factor},
                                          record_diagnostics=False)
                    diagnostic_ok = diagnostics(idata, label)["diagnostics_ok"]
                support, refute = rule_probabilities(claim_name, idata)
                rows.append({"claim": claim_name, "fit": fit_name, "factor": factor,
                             "p_support": support, "p_refute": refute,
                             "decision": decision_class(support, refute),
                             "converged": bool(diagnostic_ok)})
                if factor != 1.0:
                    del idata
                    gc.collect()
        # Model A's likelihood, as a declared robustness reading rather than a ladder rung
        if "A_both" in FITS:
            gaussian = sample_blocked(SPECS["A"], "05_A_gaussian",
                                      {"likelihood": "normal"}, record_diagnostics=False)
            beta = gaussian.posterior["beta_bar"].values.ravel()
            delta = gaussian.posterior["delta_O"].values.ravel()
            rows.append({"claim": "H1_behavioural", "fit": "A_gaussian_likelihood",
                         "factor": np.nan, "p_support": float(np.mean(beta < LOG08)),
                         "p_refute": float(np.mean(np.abs(beta) < LOG11)),
                         "decision": decision_class(float(np.mean(beta < LOG08)),
                                                    float(np.mean(np.abs(beta) < LOG11))),
                         "converged": bool(diagnostics(gaussian, "A_gaussian")["diagnostics_ok"])})
            rows.append({"claim": "M1", "fit": "A_gaussian_likelihood", "factor": np.nan,
                         "p_support": float(np.mean(delta > 0)),
                         "p_refute": float(np.mean(delta < 0)),
                         "decision": decision_class(float(np.mean(delta > 0)),
                                                    float(np.mean(delta < 0))),
                         "converged": bool(diagnostics(gaussian, "A_gaussian")["diagnostics_ok"])})
            del gaussian
            gc.collect()
        SENSITIVITY = save_table(pd.DataFrame(rows), SENSITIVITY_FILE)

show(SENSITIVITY, caption="factor 1 is the reported fit; the rows with no factor are the Gaussian "
                          "likelihood, reported as robustness and not as a ladder rung")

LADDER = SENSITIVITY[SENSITIVITY.factor.notna()]
SENSITIVITY_OK = {}
for claim_name, group in LADDER.groupby("claim"):
    SENSITIVITY_OK[claim_name] = bool(
        set(group.factor) == set(PRIOR_FACTORS)
        and group.converged.all()
        and (group.p_support.max() - group.p_support.min()) <= SENSITIVITY_SPREAD_MAX
        and (group.p_refute.max() - group.p_refute.min()) <= SENSITIVITY_SPREAD_MAX
        and group.decision.nunique() == 1)
# a per-generator H3 rung gates the single joint claim
_h3_rungs = [key for key in SENSITIVITY_OK if key.startswith("H3_location:")]
if _h3_rungs:
    SENSITIVITY_OK["H3_location"] = all(SENSITIVITY_OK[key] for key in _h3_rungs)
show(pd.DataFrame([{"claim": name, "sensitivity_ok": ok}
                   for name, ok in SENSITIVITY_OK.items()]))

figure, axis = plt.subplots(figsize=(8.5, 3.4))
for claim_name, group in LADDER.groupby("claim"):
    ordered = group.sort_values("factor")
    axis.plot(ordered.factor, ordered.p_support, marker="o", ms=5, lw=1.6, label=claim_name)
axis.axhline(PROB, color=INK["bad"], ls="--", lw=1, label=f"the {PROB:.2f} cutoff")
axis.set(xscale="log", xticks=list(PRIOR_FACTORS),
         xlabel="factor on every prior scale (1 is the tabulated prior)",
         ylabel="probability of the support rule", ylim=(-0.03, 1.03),
         title="does the answer move with the prior?")
axis.set_xticklabels([f"{factor:g}" for factor in PRIOR_FACTORS])
axis.legend(fontsize=7, ncol=2)
figure.tight_layout()
savefig(figure, "P5_sensitivity.png")
plt.show()

if all(SENSITIVITY_OK.values()):
    note("<b>Prior sensitivity: PASS.</b> No rule probability moves by more than "
         f"{SENSITIVITY_SPREAD_MAX:.2f} across the ladder and the decision class is the same at "
         "every rung, so no conclusion here is the prior's conclusion.", "good")
else:
    note(f"<b>Prior sensitivity failed for "
         f"{[name for name, ok in SENSITIVITY_OK.items() if not ok]}.</b> A conclusion that moves "
         "with the prior scale is a property of the prior; those claims become NOT REPORTABLE.",
         "bad")

## 5.4, Leave-one-out comparisons

Where two formulations compete they are compared by leave-one-out cross-validation. The quantity read
is the difference in expected log pointwise predictive density with its standard error, and Appendix B
fixes the reading rule: **a difference smaller than twice its standard error is reported as
inconclusive rather than rounded into a decision**, so a nominal ranking is not mistaken for evidence.

The standard error is computed on **paired** per-observation differences,
$\mathrm{dse}=\sqrt{N\,\mathrm{Var}(\ell^{(1)}_i-\ell^{(0)}_i)}$, which is what makes it the error of
the difference rather than the difference of two errors. A comparison is also gated on reliability: if
any Pareto $k$ exceeds ArviZ's threshold for the number of draws, the importance-sampling
approximation behind the whole calculation is not trustworthy and the comparison returns inconclusive
whatever the margin.

These comparisons run on the **whole design**, 371,000 observations for Models A and C, because
Part 3.6 proved the streaming implementation exact. Nothing is subsampled, so `elpd_loo` here is the
expected log predictive density of the fitted population and not of a declared subset of it.

Four comparisons matter, and each answers a different question.

| Group | Members | What it decides |
|---|---|---|
| M1 | both covariates against patch size only | the second leg of the M1 rule; the overlap term is the only difference |
| M1, four-way | both, overlap only, patch only, neither | descriptive: which configuration level the data prefer |
| H2 | with the phase term against without | the evidential reading of H2, reported beside the magnitude reading and not merged with it |
| H3 location | both grids, stride only, patch only, neither | the model-comparison leg of the H3 rule, per generator |
| B blocking | stage and geometry offsets against Eq. (10) taken literally | whether blocking on probe stage was worth it |

In [ ]:
#@title 5.4  Streaming LOO over the whole design, and the comparisons
def loo_cached(fit_name: str) -> dict:
    """LOO for one fit, with its per-observation arrays checkpointed.

    elpd_i and pareto_k are one float per observation, a few megabytes against a design of 371,000
    rows, so persisting them makes each fit's LOO individually resumable and lets Part 6 draw the
    Pareto diagnostics without recomputing anything.
    """
    pointwise_name = f"05_loo_pointwise_{fit_name}.parquet"
    scalars_name = f"05_loo_scalars_{fit_name}.json"
    if have(pointwise_name) and have(scalars_name):
        frame = load_df(pointwise_name)
        result = dict(load_json(scalars_name))
        result["elpd_i"] = frame.elpd_i.to_numpy(float)
        result["pareto_k"] = frame.pareto_k.to_numpy(float)
        return result
    result = loo_reconstructed(FIT_SPECS[fit_name], FITS[fit_name], FIT_OPTIONS[fit_name],
                              label=fit_name)
    save_df(pd.DataFrame({"elpd_i": result["elpd_i"], "pareto_k": result["pareto_k"]}),
            pointwise_name)
    save_json({key: value for key, value in result.items()
               if key not in ("elpd_i", "pareto_k")}, scalars_name)
    return result


def loo_compare(group: str, members: dict[str, str], reference: str) -> tuple[pd.DataFrame, dict]:
    """Compare the fits of one group, with paired differences against the reference."""
    results = {label: loo_cached(fit_name) for label, fit_name in members.items()}
    best = max(results, key=lambda label: results[label]["elpd_loo"])
    rows = []
    for label, result in sorted(results.items(), key=lambda item: -item[1]["elpd_loo"]):
        difference = results[best]["elpd_i"] - result["elpd_i"]
        against_reference = results[reference]["elpd_i"] - result["elpd_i"]
        rows.append({
            "group": group, "member": label, "fit": members[label],
            "elpd_loo": result["elpd_loo"], "se": result["se"], "p_loo": result["p_loo"],
            "n_obs": result["n_obs"],
            "elpd_diff_vs_best": float(difference.sum()),
            "dse_vs_best": float(np.sqrt(len(difference) * np.var(difference))),
            "elpd_diff_vs_reference": float(against_reference.sum()),
            "dse_vs_reference": float(np.sqrt(len(against_reference)
                                              * np.var(against_reference))),
            "max_pareto_k": result["max_pareto_k"], "good_k": result["good_k"],
            "loo_reliable": result["reliable"]})
    table = pd.DataFrame(rows)
    reliable = bool(table.loo_reliable.all())
    alternatives = table[table.member != reference]
    separated = bool(len(alternatives)
                     and (alternatives.elpd_diff_vs_reference
                          > 2 * alternatives.dse_vs_reference).all())
    runner_up = table.iloc[1] if len(table) > 1 else None
    decision = {
        "group": group, "reference": reference, "best": best, "loo_reliable": reliable,
        "reference_wins": bool(reliable and separated),
        "best_margin": float(runner_up.elpd_diff_vs_best) if runner_up is not None else np.nan,
        "best_dse": float(runner_up.dse_vs_best) if runner_up is not None else np.nan,
    }
    decision["separated_2dse"] = bool(
        np.isfinite([decision["best_margin"], decision["best_dse"]]).all()
        and abs(decision["best_margin"]) >= 2 * decision["best_dse"])
    if not reliable:
        decision["reading"] = "INCONCLUSIVE: PSIS-LOO is not reliable (inspect the Pareto k)"
    elif not decision["separated_2dse"]:
        decision["reading"] = (f"INCONCLUSIVE: the best model leads by "
                               f"{abs(decision['best_margin']):.1f} against a 2*dse of "
                               f"{2 * decision['best_dse']:.1f}")
    else:
        decision["reading"] = f"{best} preferred"
    return table, decision


LOO_GROUPS: dict[str, tuple[dict[str, str], str]] = {}
if "A_both" in FITS and "A_patch" in FITS:
    LOO_GROUPS["M1"] = ({"overlap + patch size": "A_both", "patch size only": "A_patch"},
                        "overlap + patch size")
    LOO_GROUPS["M1_fourway"] = ({"overlap + patch size": "A_both", "overlap only": "A_overlap",
                                 "patch size only": "A_patch", "neither": "A_none"},
                                "overlap + patch size")
if "C_phase" in FITS and "C_nophase" in FITS:
    LOO_GROUPS["H2"] = ({"phase-dependent": "C_phase", "phase-free": "C_nophase"},
                        "phase-dependent")
if "B_adjusted" in FITS and "B_unadjusted" in FITS:
    LOO_GROUPS["B_blocking"] = ({"stage and geometry adjusted": "B_adjusted",
                                 "Eq. (10) taken literally": "B_unadjusted"},
                                "stage and geometry adjusted")
for mode in D1_MODES:
    if f"D1_{mode}_both" in FITS:
        LOO_GROUPS[f"H3_location:{mode}"] = (
            {"both grids": f"D1_{mode}_both", "stride only": f"D1_{mode}_stride",
             "patch only": f"D1_{mode}_patch", "no grid": f"D1_{mode}_none"}, "both grids")

LOO_FILE = "05_loo.parquet"
if have(LOO_FILE) and have("05_loo_decisions.json"):
    LOO = load_df(LOO_FILE)
    LOO_DECISIONS = load_json("05_loo_decisions.json")
else:
    with PROGRESS.task("part5_loo"):
        tables, decisions = [], {}
        for group, (members, reference) in LOO_GROUPS.items():
            table, decision = loo_compare(group, members, reference)
            tables.append(table)
            decisions[group] = decision
            print(f"  {group}: {decision['reading']}")
        LOO = save_table(pd.concat(tables, ignore_index=True), LOO_FILE)
        LOO_DECISIONS = save_json(decisions, "05_loo_decisions.json")

show(LOO, caption="elpd_diff_vs_reference is the paired difference the rule reads; a win needs it "
                  "to exceed twice dse_vs_reference and every member to be reliable")
show(pd.DataFrame(LOO_DECISIONS).T.reset_index(drop=True)[
        ["group", "reference", "best", "loo_reliable", "reference_wins", "reading"]],
     caption="the reading rule of Appendix B, applied")

LOO_OK = {group: bool(decision["loo_reliable"]) for group, decision in LOO_DECISIONS.items()}
LOO_WIN = {group: bool(decision["reference_wins"]) for group, decision in LOO_DECISIONS.items()}
# Only the modes a verdict is read from roll up into the claim; the pure-sinusoid comparison is
# reported on its own row and gates nothing, for the reason given in Part 3.3.
_h3_groups = [f"H3_location:{mode}" for mode in VERDICT_MODES
              if f"H3_location:{mode}" in LOO_DECISIONS]
if _h3_groups:
    LOO_OK["H3_location"] = all(LOO_OK[group] for group in _h3_groups)
    LOO_WIN["H3_location"] = all(LOO_WIN[group] for group in _h3_groups)

figure, axis = plt.subplots(figsize=(9, max(2.8, 0.36 * len(LOO))))
plot_rows = LOO[LOO.member != LOO.group.map(
    {group: reference for group, (_, reference) in LOO_GROUPS.items()})].reset_index(drop=True)
y = np.arange(len(plot_rows))
colours = np.where(plot_rows.elpd_diff_vs_reference > 2 * plot_rows.dse_vs_reference,
                   INK["good"], INK["muted"])
axis.barh(y, plot_rows.elpd_diff_vs_reference, color=colours, alpha=0.85,
          xerr=2 * plot_rows.dse_vs_reference, error_kw={"ecolor": INK["line"], "lw": 1})
axis.axvline(0, color=INK["line"], lw=1)
axis.set_yticks(y)
axis.set_yticklabels([f"{r.group}: reference - {r.member}" for r in plot_rows.itertuples()],
                     fontsize=7)
axis.invert_yaxis()
axis.set(title="how far the reference formulation leads, with two standard errors drawn",
         xlabel="paired elpd difference (positive favours the reference)")
figure.tight_layout()
savefig(figure, "P5_loo.png")
plt.show()

if all(LOO_OK.values()):
    note("<b>LOO reliability: PASS.</b> Every Pareto k is below ArviZ's threshold for this number "
         "of draws, so every comparison above is an approximation the diagnostics support.", "good")
else:
    note(f"<b>Unreliable PSIS-LOO in {[g for g, ok in LOO_OK.items() if not ok]}.</b> Those "
         "comparisons return inconclusive whatever their margin, and the claims whose rules "
         "include a comparison leg become NOT REPORTABLE.", "bad")

## 5.5, The verdicts

One row per claim, and three readings of each.

**Pre-gate** is the probability rule applied on its own: what the posterior says, before asking
whether the posterior may be believed. It is reported because the difference between the pre-gate and
post-gate columns is the whole content of the validation: a claim that is `SUPPORTED` pre-gate and
`NOT REPORTABLE` post-gate is a claim whose evidence exists but whose conditions were not met, and
that is a different situation from one that is inconclusive on the numbers.

**Each gate, independently.** Every gate is shown as its own column, evaluated on its own, so a
failure can be attributed rather than merely noted. The gates are not combined into a score: they are
a conjunction, and the `gates_failed` column names the ones that did not hold.

**Post-gate** is the reportable verdict. The order of resolution is fixed: a non-reportable mode
first, then identification, then the gates, then the probability rule, and last the comparison leg
for the two claims whose rules include one.

| Verdict | Meaning |
|---|---|
| `SUPPORTED` | the support rule reached 0.95 and every gate held |
| `REFUTED` | the refutation rule reached 0.95 and every gate held |
| `INCONCLUSIVE` | the gates held and the rule did not decide; a verdict in its own right |
| `NOT REPORTABLE` | a gate failed, or the run is not in `full` mode; not a null result |
| `NOT IDENTIFIED` | the design cannot answer the question, for a reason stated in the note; not a null result |

### Model D2 needs one gate the document does not name

Eq. (13) regresses the **fundamental** of a branch on the spacing that branch predicts. Where the
first dip of a comb was not detected, the lowest detected site is a higher harmonic and the ratio
sits near an integer above one: a geometry whose sites follow the grid perfectly can still push
$\kappa$ towards two. Part 4.5 reports that harmonic order per geometry and fits a restricted
version alongside the preregistered one.

Reporting the audit and then ignoring it in the verdict would be the worst of both, so the two
readings are required to **agree**. Neither is an independent verification of the movement law,
because the branch assignment already used the predicted grid, and ten unambiguous sites do not
change that; what the agreement establishes is the weaker and honest claim that the reading does not
depend on which of the two sets of sites is used. A claim that survives one reading and not the
other is inconclusive, not resolved by choosing.

In [ ]:
#@title 5.5  Pre-gate reading, every gate, post-gate verdict
PRIOR_GATE_OK = True   # Part 0.8 and Part 1 raise on failure, so reaching here is the pass

CLAIM_ESTIMATE = {
    "H1_behavioural": ("A_both", "recovery_ratio", "exp(beta_bar)"),
    "M1": ("A_both", "delta_O", "delta_O"),
    "H1_representational": ("B_adjusted", "codelength_ratio", "exp(theta_lock)"),
    "H2": ("C_phase", "sigma_phase", "sigma_phase"),
    "H3a": ("D2_stride_f1", "kappa", "kappa_S"),
    "H3b": ("D2_patch_f1", "kappa", "kappa_P"),
}
# Every gate is a conjunction over EVERY fit the claim is read from, not over a representative
# one. H3_location pools both generators and takes the less favourable, so a generator whose
# model fails its predictive check has to fail the gate: naming only the first would let it pass.
CLAIM_RECOVERY_SPEC = {
    "H1_behavioural": ("A",), "M1": ("A",), "H1_representational": ("B",), "H2": ("C",),
    "H3_location": tuple(f"D1_{mode}" for mode in VERDICT_MODES),
    "H3a": ("D2_stride_f1",), "H3b": ("D2_patch_f1",),
}
CLAIM_PPC_FIT = {
    "H1_behavioural": ("A_both",), "M1": ("A_both",), "H1_representational": ("B_adjusted",),
    "H2": ("C_phase",),
    "H3_location": tuple(f"D1_{mode}_both" for mode in VERDICT_MODES),
    "H3a": ("D2_stride_f1",), "H3b": ("D2_patch_f1",),
}


def all_pass(results: dict, names: tuple) -> bool:
    """True when every named fit is present and passed. An empty list is a failure, not a pass."""
    return bool(names) and all(results.get(name, False) for name in names)


def d2_consistency(branch: str) -> tuple:
    """Whether Model D2's two readings agree, and what they say.

    Eq. (13) regresses the FUNDAMENTAL of a branch on the spacing that branch predicts. Where the
    first dip of a comb was not detected, the lowest detected site is a higher harmonic and the
    ratio sits near an integer above one, which pushes kappa away from one for a reason that is a
    property of the detection rather than of the model. The preregistered fit keeps those rows,
    because that is what the appendix specifies; the restricted fit drops them. Neither is an
    independent verification of the movement law, since the branch assignment already used the
    predicted grid, so what is required here is that the two agree: a claim that depends on which
    of the two is read is reported as inconclusive rather than resolved by choosing.
    """
    primary = f"D2_{branch}_f1"
    restricted = f"{primary}_fundamental"
    if primary not in FITS:
        return None, "no fit for this branch"
    contaminated = D2_AUDIT[(D2_AUDIT.branch == branch) & (D2_AUDIT.harmonic_order > 1)]
    if contaminated.empty:
        return True, "every lowest detected site is its branch's fundamental"
    if restricted not in FITS:
        return False, (f"{len(contaminated)} geometry(ies) contribute a higher harmonic and too "
                       "few remain for a restricted fit, so the two readings cannot be compared")
    readings = {}
    for label, name in (("all detected", primary), ("fundamental only", restricted)):
        draws = FITS[name].posterior["kappa"].values.ravel()
        probability = float(np.mean(np.abs(draws - 1) < ROPE_SLOPE))
        readings[label] = (decision_class(probability, 1 - probability), probability,
                           float(np.median(draws)))
    agree = readings["all detected"][0] == readings["fundamental only"][0]
    detail = "; ".join(f"{label}: {cls} at {prob:.3f}, kappa median {median:+.3f}"
                       for label, (cls, prob, median) in readings.items())
    return agree, detail
# The two claims whose rule of tab:bayesDecisions has a comparison leg. For H2 and for Model B's
# blocking the comparison is reported beside the verdict but does not gate it, which is why they
# are absent from this set rather than listed in a second one.
LOO_REQUIRED = {"M1", "H3_location"}


def resolve_verdict(claim_name: str, pre_gate: str, gates: dict, identified: bool,
                    identification_note: str, loo_won: bool, reportable_mode: bool) -> tuple:
    """The post-gate verdict, in a fixed order of resolution.

    Mode first, because a smoke run cannot produce evidence whatever it computes. Then
    identification, because a question the design cannot answer is not answered by a converged
    fit. Then the gates, as a conjunction. Then the probability rule. And last the comparison leg,
    for the two claims whose rule has one: a claim that reaches 0.95 on the probability and loses
    the comparison is inconclusive, not supported.
    """
    failed = sorted(name for name, ok in gates.items() if ok is False)
    if not reportable_mode:
        return "NOT REPORTABLE", f"mode {MODE}", failed
    if not identified:
        return "NOT IDENTIFIED", identification_note, failed
    if failed:
        return "NOT REPORTABLE", ", ".join(name[5:] for name in failed), failed
    if pre_gate == "SUPPORTED" and claim_name in LOO_REQUIRED and not loo_won:
        return "INCONCLUSIVE", "the comparison leg of the rule is not satisfied", failed
    return pre_gate, "", failed


verdict_rows = []
for entry in CLAIM_PROBS:
    claim_name = entry["claim"]
    fits = tuple(entry["fits"])
    group = entry.get("loo_group")

    gates = {
        "gate_prior": bool(PRIOR_GATE_OK),
        "gate_convergence": converged(*fits) if fits else False,
        "gate_recovery": all_pass(RECOVERY_OK, CLAIM_RECOVERY_SPEC.get(claim_name, ())),
        "gate_ppc": all_pass(PPC_OK, CLAIM_PPC_FIT.get(claim_name, ())),
        "gate_sensitivity": bool(SENSITIVITY_OK.get(claim_name, False)),
    }
    # The comparison is a gate only where the rule of tab:bayesDecisions has a comparison leg.
    # For H2 and for Model B's blocking it is reported beside the verdict and must not decide it,
    # so it is marked not applicable rather than required.
    gates["gate_loo_reliable"] = (bool(LOO_OK.get(group, False))
                                  if (group is not None and claim_name in LOO_REQUIRED) else None)
    robustness = "-"
    if claim_name in ("H3a", "H3b"):
        branch = "stride" if claim_name == "H3a" else "patch"
        agree, robustness = d2_consistency(branch)
        gates["gate_d2_readings_agree"] = agree

    pre_gate = decision_class(entry["p_support"], entry["p_refute"])
    post_gate, reason, failed = resolve_verdict(
        claim_name, pre_gate, gates, entry["identified"], entry["identification_note"],
        bool(LOO_WIN.get(group, False)), IS_FULL)

    fit_name, variable, label = CLAIM_ESTIMATE.get(claim_name, (None, None, None))
    median = low = high = np.nan
    if fit_name in FITS and variable:
        draws = FITS[fit_name].posterior[variable].values.ravel()
        median, low, high = (float(np.median(draws)), float(np.quantile(draws, 0.025)),
                             float(np.quantile(draws, 0.975)))

    verdict_rows.append({
        "claim": claim_name, "model": entry["model"],
        "estimand": label or entry["parameter"],
        "median": median, "low": low, "high": high,
        "p_support": entry["p_support"], "p_refute": entry["p_refute"],
        "pre_gate": pre_gate, **gates,
        "gates_failed": ", ".join(name[5:] for name in failed) or "-",
        "loo_reading": (LOO_DECISIONS.get(group, {}).get("reading", "-")
                        if group else "not applicable"),
        "loo_leg_required": claim_name in LOO_REQUIRED,
        "robustness": robustness,
        "post_gate": post_gate, "reason": reason,
        "rule": entry["rule"]})

VERDICTS = save_table(pd.DataFrame(verdict_rows), "05_verdicts.parquet")
GATE_COLUMNS = [column for column in VERDICTS.columns if column.startswith("gate_")]

show(VERDICTS[["claim", "model", "estimand", "median", "low", "high", "p_support", "p_refute",
               "pre_gate", "post_gate", "gates_failed"]],
     title="Verdicts", caption="the full table, with every gate as its own column and the "
                               "decision rules, is in tables/05_verdicts.csv")
show(VERDICTS[["claim"] + GATE_COLUMNS + ["gates_failed"]],
     caption="each gate evaluated on its own. None means the gate does not apply to that claim: "
             "the comparison is not a leg of its rule, or it is not a D2 branch.")
if (VERDICTS.robustness != "-").any():
    show(VERDICTS.loc[VERDICTS.robustness != "-", ["claim", "robustness"]],
         caption="Model D2's two readings, which the verdict requires to agree")

_PALETTE = {"SUPPORTED": ("#E6F2EA", "#2F6B4A"), "REFUTED": ("#F7E9E9", "#8E3B3B"),
            "INCONCLUSIVE": ("#F3F5F7", "#4A5560"),
            "NOT REPORTABLE": ("#FBF1E4", "#A85D21"),
            "NOT IDENTIFIED": ("#EFEDF6", "#4F4585")}
if _RICH:
    cards = []
    for row in VERDICTS.itertuples():
        background, edge = _PALETTE.get(row.post_gate, ("#F3F5F7", "#4A5560"))
        interval = ("" if not np.isfinite(row.median)
                    else f"<div style='font-size:12px;color:#3B4550;margin-top:4px'>"
                         f"{row.estimand} = <b>{row.median:.3f}</b> "
                         f"[{row.low:.3f}, {row.high:.3f}]</div>")
        cards.append(
            f"<div style='flex:1 1 236px;background:{background};border:1px solid {edge}33;"
            f"border-left:5px solid {edge};border-radius:6px;padding:10px 12px'>"
            f"<div style='font-size:11px;letter-spacing:.06em;color:#66707A;"
            f"text-transform:uppercase'>{row.claim} &nbsp;|&nbsp; model {row.model}</div>"
            f"<div style='font-size:17px;font-weight:700;color:{edge};margin-top:3px'>"
            f"{row.post_gate}</div>{interval}"
            f"<div style='font-size:11px;color:#5A6570;margin-top:6px'>pre-gate "
            f"<b>{row.pre_gate}</b> &nbsp;&middot;&nbsp; support {row.p_support:.3f} "
            f"&nbsp;&middot;&nbsp; refute {row.p_refute:.3f}</div>"
            f"<div style='font-size:11px;color:#5A6570;margin-top:2px'>gates failed: "
            f"{row.gates_failed}</div>"
            + (f"<div style='font-size:11px;color:{edge};margin-top:2px'>{row.reason}</div>"
               if row.reason else "")
            + "</div>")
    display(HTML(
        f"<div style='font:13px system-ui,-apple-system,sans-serif'>"
        f"<div style='font-size:15px;font-weight:700;margin:10px 0 8px'>"
        f"Deliverable 2 claims, run {RUN_ID}</div>"
        f"<div style='display:flex;flex-wrap:wrap;gap:10px'>{''.join(cards)}</div>"
        f"<div style='font-size:11px;color:#66707A;margin-top:10px'>"
        f"NOT REPORTABLE and NOT IDENTIFIED are not null results. "
        f"Tone SNR {TONE_SNR}, {len(CONTRASTS):,} triplets, {CHAINS} chains of {DRAWS} draws."
        f"</div></div>"))

if not IS_FULL:
    note(f"<b>MODE is {MODE!r}, so every verdict is NON-REPORTABLE by construction.</b> "
         "Only a full run on the complete design can produce evidence about Chronos.", "warn")

## 5.6, Provenance

Everything needed to say where a number came from: the code, the data, the environment, the gates and
the limitations this design cannot remove.

In [ ]:
#@title 5.6  The run summary
LIMITATIONS = [
    "One training seed per geometry, so a difference between two configurations is a difference "
    "between two trained models as well as between two geometries, and the design cannot separate "
    "them. An effect large relative to its posterior spread is unlikely to be an artefact of that "
    "choice; an effect near a decision threshold could be.",
    "The analysis context is truncated: of the 480 samples the patch grid spans only L_tok, and the "
    "remainder enters no token. The truncation cancels within a triplet, which shares context "
    "length, background and phase, but not between geometries, and it is non-zero on exactly the "
    "six runs with S not dividing P, which are the runs that identify the stride branch.",
    "The token count N = 1 + floor((L - P)/S) moves with the stride, so the M1 slope cannot "
    "separate overlap from sequence length; lowering S also changes the mixture of stride- and "
    "patch-derived candidates along the same axis. Neither confound could create a mitigation, but "
    "either could inflate one.",
    "Model D2's sites are detected from the profile alone but assigned to a branch using that "
    "branch's own grid, so kappa_F is conditional on that selection. Part 4.5 reports the "
    "harmonic-order audit and a restricted fit rather than correcting the slope.",
    "The tone amplitude is a choice, not a quantity the report fixes. It is a field of the "
    "collection's configuration, so it enters the collection's design fingerprint, and a run at a "
    "different amplitude is refused rather than merged.",
    "M1's slope is a comparison between independently trained variants over fifteen design points, "
    "not the effect of changing the overlap of a trained model.",
]

RUN_SUMMARY = {
    "run_id": RUN_ID, "mode": MODE, "reportable": IS_FULL,
    "analysis_fingerprint": ANALYSIS_FINGERPRINT,
    "repository_commit": REPO_COMMIT,
    "tone_snr": TONE_SNR,
    "collection": {"directory": str(DATA_DIR), "hashes": SOURCE_HASHES,
                   "triplets": int(len(CONTRASTS)), "mdl_rows": int(len(MDL)),
                   "collapse_rows": int(len(COLLAPSE))},
    "sampling": ANALYSIS_SPEC["sampling"],
    "gates": {
        "prior_calibration": bool(PRIOR_GATE_OK),
        "design": bool(DESIGN_CHECK.ok.all()),
        "reconstruction_parity": bool(PARITY.agrees.all()),
        "psis_parity": bool(PSIS_PARITY.agrees.all()),
        "recovery": {name: bool(ok) for name, ok in RECOVERY_OK.items()},
        "convergence": {name: bool(ok) for name, ok in _converged.items()},
        "ppc": {name: bool(ok) for name, ok in PPC_OK.items()},
        "sensitivity": {name: bool(ok) for name, ok in SENSITIVITY_OK.items()},
        "loo_reliable": {name: bool(ok) for name, ok in LOO_OK.items()},
    },
    "verdicts": {row["claim"]: row["post_gate"] for row in verdict_rows},
    "d2_identification": D2_IDENTIFICATION,
    "timings_seconds": dict(_TIMINGS),
    "limitations": LIMITATIONS,
}
save_json(RUN_SUMMARY, "05_run_summary.json")
print(json.dumps({key: RUN_SUMMARY[key] for key in
                  ("run_id", "mode", "reportable", "tone_snr", "verdicts")}, indent=2))
for index, limitation in enumerate(LIMITATIONS, 1):
    print(f"\n  ({index}) {limitation}")

---
# Part 6, Reading the results

Part 5 gates every fit on numbers. This part looks at them. A model can pass every scalar threshold
while a picture shows something a table cannot: a chain that mixed well on average but drifted early, a
posterior that never moved from its prior, one observation dominating a comparison, or a claim whose
support probability sits just at the cutoff.

**This part depends only on saved artifacts.** Every figure is built from the checkpointed posteriors
and the checkpointed tables in the run folder, never from anything held in memory by an earlier part.
To regenerate the figures alone, in a fresh runtime with no Chronos and no sampling, set
`STANDALONE_FIGURES = True` in Part 0.4 and run all: collection and sampling are then refused, every
stage reloads from its checkpoint in seconds, and a missing artifact is reported by name instead of
being recomputed.

The figures Parts 2, 4 and 5 produce are part of the same set and are already written to
`figures/`; this part adds the cross-cutting and diagnostic ones and closes with an index of all of
them.

In [ ]:
#@title 6.0  Reload every artifact this part reads
banner("PART 6, READING THE RESULTS")

# Part 6 spans several cells, so its cost is measured from here to the end of 6.7 rather than
# bracketed by a context manager that would close before the figures were drawn.
PART6_STARTED = time.time()


def fit_of(key: str):
    """The posterior of one fit, from memory if it is there and from its checkpoint otherwise."""
    if key in FITS:
        return FITS[key]
    return load_idata(f"04_{key}.nc")


# One row per stage of the run. Loading them all here is both what Part 6 reads and a completeness
# check: a stage that produced nothing is named before any figure is drawn.
REQUIRED_TABLES = {
    "0  decision rules": "00_decision_spec.parquet",
    "1  prior predictive": "01_prior_predictive.parquet",
    "2  design gate": "02_design_check.parquet",
    "3  parameter recovery": "03_recovery.parquet",
    "4  estimands": "04_estimands.parquet",
    "5  convergence": "05_convergence.parquet",
    "5  posterior predictive": "05_ppc_rates.parquet",
    "5  prior sensitivity": "05_sensitivity.parquet",
    "5  LOO comparisons": "05_loo.parquet",
    "5  verdicts": "05_verdicts.parquet",
}
TABLES, missing = {}, []
for label, name in REQUIRED_TABLES.items():
    try:
        TABLES[label] = load_df(name)
    except FileNotFoundError:
        missing.append(f"{label} -> {name}")
if missing:
    raise FileNotFoundError(
        f"Part 6 reads the run's own result tables and these are absent from {CKPT_DIR}: "
        f"{missing}. In a figures-only session, point RUN_FOLDER at a finished run.")
show(pd.DataFrame([{"stage": label, "table": REQUIRED_TABLES[label], "rows": len(frame)}
                   for label, frame in TABLES.items()]),
     caption="every stage of the run produced output, and this is what Part 6 reads")

CANDIDATE_FITS = {"H1_behavioural / M1": "A_both", "H1_representational": "B_adjusted",
                  "H2": "C_phase"}
for mode in VERDICT_MODES:
    CANDIDATE_FITS[f"H3_location {mode}"] = f"D1_{mode}_both"
for branch in ("stride", "patch"):
    CANDIDATE_FITS[f"H3{'a' if branch == 'stride' else 'b'}"] = f"D2_{branch}_f1"

# A family that this session did not fit and that has no checkpoint is reported as absent rather
# than raising: the figures that do not need it are still worth drawing.
PRIMARY_FITS = {label: key for label, key in CANDIDATE_FITS.items()
                if key in FITS or have(f"04_{key}.nc")}
ABSENT_FITS = {label: key for label, key in CANDIDATE_FITS.items()
               if label not in PRIMARY_FITS}
POSTERIORS = {key: fit_of(key) for key in PRIMARY_FITS.values()}
print("\nprimary fits available to this part:")
for claim_label, key in PRIMARY_FITS.items():
    print(f"  {claim_label:<26s} <- 04_{key}.nc")
if ABSENT_FITS:
    note("These fits are not in this run folder, so the figures that read them are skipped: "
         + ", ".join(f"{label} ({key})" for label, key in ABSENT_FITS.items()), "warn")

## 6.1, Every estimand against its own threshold

One figure for the whole study. Each row is a claim's estimand on its natural scale, with the posterior
median, the 95 per cent credible interval, and the threshold its rule is written against. The colour is
the post-gate verdict, so a wide interval next to a green label and a narrow one next to an orange
label are both visible for what they are: the interval says what the data determined, and the colour
says whether the conditions for reading it were met.

In [ ]:
#@title 6.1  Every estimand, one figure
VERDICTS_VIEW = TABLES["5  verdicts"]
THRESHOLDS = {"H1_behavioural": (0.8, "recovery ratio 0.8"),
              "M1": (0.0, "no slope"),
              "H1_representational": (1.2, "codelength ratio 1.2"),
              "H2": (LOG11, r"$\log 1.1$"),
              "H3a": (1.0, r"$\kappa_S=1$"), "H3b": (1.0, r"$\kappa_P=1$")}
plottable = VERDICTS_VIEW[np.isfinite(VERDICTS_VIEW["median"])].reset_index(drop=True)

figure, axes = plt.subplots(len(plottable), 1, figsize=(9, 1.25 * len(plottable)), squeeze=False)
for position, row in plottable.iterrows():
    axis = axes[position, 0]
    background, edge = _PALETTE.get(row.post_gate, ("#F3F5F7", "#4A5560"))
    axis.set_facecolor(background)
    axis.errorbar(row["median"], 0, xerr=[[row["median"] - row.low], [row.high - row["median"]]],
                  fmt="o", color=edge, ms=8, capsize=5, lw=2)
    threshold = THRESHOLDS.get(row.claim)
    if threshold is not None:
        axis.axvline(threshold[0], color=INK["line"], ls="--", lw=1.2)
        axis.annotate(threshold[1], (threshold[0], 0.32), fontsize=7, color=INK["line"],
                      ha="center")
    axis.set_yticks([])
    axis.set_ylim(-0.45, 0.45)
    axis.set_xlabel(row.estimand, fontsize=8)
    axis.set_title(f"{row.claim}  ({row.model})    {row.post_gate}     "
                   f"median {row['median']:.3f}  [{row.low:.3f}, {row.high:.3f}]     "
                   f"support {row.p_support:.3f}",
                   loc="left", fontsize=9, color=edge)
figure.suptitle(f"The estimands of Deliverable 2, run {RUN_ID}", y=1.002, fontsize=11)
figure.tight_layout()
savefig(figure, "P6_estimands.png")
plt.show()

## 6.2, Prior against posterior: how much the data moved each estimand

A posterior that has not moved from its prior is a report that the data were uninformative about that
parameter, and it is not the same thing as evidence that the effect is absent. The document says so
explicitly in the caption of `tab:bayesPriors`. These panels put the two densities side by side for
every estimand, with the decision threshold drawn, so the reader can see which of the two is doing the
work.

The number printed on each panel is the ratio of the posterior's standard deviation to the prior's:
small means the data determined the parameter, near one means they did not.

In [ ]:
#@title 6.2  Prior against posterior for every estimand
PRIOR_DRAWS_BY_ESTIMAND = {
    ("A_both", "beta_bar"): stats.t.rvs(NU, scale=PRIOR_SCALE, size=40000, random_state=SEED),
    ("A_both", "delta_O"): stats.t.rvs(NU, scale=PRIOR_SCALE, size=40000, random_state=SEED + 1),
    ("A_both", "delta_P"): stats.t.rvs(NU, scale=PRIOR_SCALE, size=40000, random_state=SEED + 2),
    ("A_both", "tau"): np.abs(stats.t.rvs(NU, scale=PRIOR_SCALE, size=40000,
                                          random_state=SEED + 3)),
    ("B_adjusted", "theta_lock"): RNG.normal(0, 0.5, 40000),
    ("C_phase", "sigma_phase"): SIGMA_PHASE_PRIOR,
}
for branch in ("stride", "patch"):
    key = f"D2_{branch}_f1"
    if key in POSTERIORS:
        PRIOR_DRAWS_BY_ESTIMAND[(key, "kappa")] = RNG.normal(0, 1, 40000)
for mode in VERDICT_MODES:
    PRIOR_DRAWS_BY_ESTIMAND[(f"D1_{mode}_both", "theta_S")] = RNG.normal(0, 1, 40000)
    PRIOR_DRAWS_BY_ESTIMAND[(f"D1_{mode}_both", "theta_P")] = RNG.normal(0, 1, 40000)

entries = [(key, variable, prior) for (key, variable), prior
           in PRIOR_DRAWS_BY_ESTIMAND.items() if key in POSTERIORS]
columns = 3
rows_needed = math.ceil(len(entries) / columns)
figure, axes = plt.subplots(rows_needed, columns, figsize=(4.4 * columns, 2.5 * rows_needed),
                            squeeze=False)
THRESHOLD_LINES = {"beta_bar": LOG08, "delta_O": 0.0, "theta_lock": LOG12,
                   "sigma_phase": LOG11, "kappa": 1.0, "theta_S": 0.0, "theta_P": 0.0}
UPDATING = []
for index, (key, variable, prior) in enumerate(entries):
    axis = axes[index // columns, index % columns]
    posterior = POSTERIORS[key].posterior[variable].values.ravel()
    low, high = np.quantile(np.concatenate([posterior, prior]), [0.001, 0.999])
    bins = np.linspace(low, high, 90)
    axis.hist(prior, bins=bins, density=True, histtype="step", lw=1.3, color=INK["muted"],
              label="prior")
    axis.hist(posterior, bins=bins, density=True, color=INK["primary"], alpha=0.7,
              label="posterior")
    if variable in THRESHOLD_LINES:
        axis.axvline(THRESHOLD_LINES[variable], color=INK["bad"], ls="--", lw=1)
    shrink = float(posterior.std() / prior.std())
    UPDATING.append({"fit": key, "parameter": variable, "prior_sd": float(prior.std()),
                     "posterior_sd": float(posterior.std()), "sd_ratio": shrink})
    axis.set_title(f"{key}  {variable}\nposterior sd / prior sd = {shrink:.3f}", fontsize=8)
    axis.set_yticks([])
    if index == 0:
        axis.legend(fontsize=7)
for index in range(len(entries), rows_needed * columns):
    axes[index // columns, index % columns].axis("off")
figure.tight_layout()
savefig(figure, "P6_prior_posterior.png")
plt.show()
save_table(pd.DataFrame(UPDATING), "06_updating.parquet")
show(pd.DataFrame(UPDATING), caption="a ratio near one is a parameter the data did not determine")

## 6.3, Chain behaviour: traces and rank plots

$\hat R$ and the effective sample size say *whether* the chains agree. A trace shows *how*, and would
catch a chain that drifted through warm-up and then happened to land where the others were. The rank
plot is the sharper of the two: the draws of all chains are pooled and ranked, and each chain's ranks
are histogrammed. Under convergence every chain's histogram is uniform, so a chain that explores a
narrower or shifted region shows as a sloped or humped histogram even when $\hat R$ is close to one.

In [ ]:
#@title 6.3  Traces and rank plots for the primary estimands
TRACE_TARGETS = [("A_both", ["beta_bar", "delta_O", "delta_P", "tau"]),
                 ("B_adjusted", ["theta_lock", "r"]),
                 ("C_phase", ["sigma_phase", "beta_bar"])]
TRACE_TARGETS += [(f"D1_{mode}_both", ["theta_S", "theta_P"]) for mode in VERDICT_MODES]
TRACE_TARGETS += [(key, ["kappa", "sigma_F"]) for key in POSTERIORS if key.startswith("D2_")]

for key, variables in TRACE_TARGETS:
    if key not in POSTERIORS:
        continue
    idata = POSTERIORS[key]
    figure, axes = plt.subplots(len(variables), 2, figsize=(12, 1.7 * len(variables)),
                                squeeze=False, gridspec_kw={"width_ratios": [2, 1]})
    for row, variable in enumerate(variables):
        array = idata.posterior[variable]
        n_chains = int(array.sizes["chain"])
        for chain in range(n_chains):
            axes[row, 0].plot(np.asarray(array.isel(chain=chain)), lw=0.5, alpha=0.8,
                              label=f"chain {chain}")
        axes[row, 0].set_ylabel(variable, fontsize=8)
        flat = array.transpose("chain", "draw").values
        ranks = stats.rankdata(flat.ravel()).reshape(flat.shape)
        edges = np.linspace(0, flat.size, 21)
        for chain in range(n_chains):
            axes[row, 1].hist(ranks[chain], bins=edges, histtype="step", lw=1.1,
                              label=f"chain {chain}")
        axes[row, 1].axhline(flat.shape[1] / 20, color=INK["line"], ls="--", lw=1)
        axes[row, 1].set_yticks([])
        if row == 0:
            axes[row, 0].legend(fontsize=6, ncol=n_chains)
            axes[row, 1].set_title("rank plot: uniform under convergence", fontsize=8)
    axes[-1, 0].set_xlabel("draw")
    axes[-1, 1].set_xlabel("pooled rank")
    figure.suptitle(f"chain behaviour, {key}", y=1.01, fontsize=10)
    figure.tight_layout()
    savefig(figure, f"P6_trace_{key}.png")
    plt.show()

## 6.4, The pairs where a funnel would appear

Two places in this analysis are named in advance as geometrically awkward, and both are worth looking
at rather than trusting to a divergence count.

Model D2's residual carries the sweep-resolution floor precisely because, without it, $\sigma_F$ is
driven towards zero when the sites land on the prediction, and the posterior develops a funnel the
sampler cannot traverse. The zero-divergence gate says that did not happen here; the
$\kappa$ against $\sigma_F$ panel is where that claim becomes visible.

Model C's $\sigma_\phi$ is predicted near zero by the hypothesis itself, which is the regime in which
a centred group scale funnels. It is written non-centred for that reason, and the $\sigma_\phi$ against
$u_p$ panel shows whether the reparameterisation did its job.

Model A's population effect against its hierarchical scale is included as a comparison: a pair with no
expected pathology, so the reader can see what an unremarkable one looks like.

In [ ]:
#@title 6.4  Pair plots on the funnel-risk pairs
PAIR_TARGETS = [("A_both", "beta_bar", "tau", "A: the population effect against its scale")]
if "C_phase" in POSTERIORS:
    PAIR_TARGETS.append(("C_phase", "sigma_phase", "beta_bar",
                         r"C: $\sigma_\phi$, predicted near zero, against the deficit scale"))
for key in POSTERIORS:
    if key.startswith("D2_"):
        PAIR_TARGETS.append((key, "kappa", "sigma_F",
                             f"{key}: the pair the resolution floor exists to fix"))

available = [target for target in PAIR_TARGETS if target[0] in POSTERIORS]
figure, axes = plt.subplots(1, len(available), figsize=(4.6 * len(available), 3.6), squeeze=False)
for position, (key, x_name, y_name, title) in enumerate(available):
    axis = axes[0, position]
    idata = POSTERIORS[key]
    x = idata.posterior[x_name].values.ravel()
    y = idata.posterior[y_name].values.ravel()
    diverging = np.asarray(idata.sample_stats["diverging"]).ravel().astype(bool)
    axis.scatter(x[~diverging], y[~diverging], s=4, alpha=0.18, color=INK["primary"],
                 linewidths=0, label="draws")
    if diverging.any():
        axis.scatter(x[diverging], y[diverging], s=22, color=INK["bad"], marker="x",
                     label=f"divergences ({int(diverging.sum())})")
    axis.set(xlabel=x_name, ylabel=y_name, title=title)
    axis.legend(fontsize=7)
figure.tight_layout()
savefig(figure, "P6_pairs.png")
plt.show()

## 6.5, Pareto $k$ per observation

Part 5.4 gates each comparison on the **largest** Pareto $k$. Plotting all of them shows which
observations drive it, which is the difference between one awkward row and a systematic problem. For
Models A and C it is also where the Student-$t_4$ choice of Part 3.1 is visible: under a Gaussian the
dead-frequency contrasts would be the high-$k$ points, and the heavy tail is what keeps them from
being.

The arrays are read from the pointwise LOO checkpoints, so nothing is recomputed.

In [ ]:
#@title 6.5  Pareto k per observation, from the LOO checkpoints
pointwise_available = [(key, f"05_loo_pointwise_{key}.parquet")
                       for key in POSTERIORS if have(f"05_loo_pointwise_{key}.parquet")]
if pointwise_available:
    columns = min(3, len(pointwise_available))
    rows_needed = math.ceil(len(pointwise_available) / columns)
    figure, axes = plt.subplots(rows_needed, columns,
                                figsize=(4.6 * columns, 2.8 * rows_needed), squeeze=False)
    for index, (key, name) in enumerate(pointwise_available):
        axis = axes[index // columns, index % columns]
        frame = load_df(name)
        scalars = load_json(f"05_loo_scalars_{key}.json")
        k = frame.pareto_k.to_numpy(float)
        step = max(1, len(k) // 20000)      # plot at most 20k points, the extremes are kept
        axis.scatter(np.arange(0, len(k), step), k[::step], s=3, alpha=0.4,
                     color=INK["primary"], linewidths=0)
        axis.axhline(scalars["good_k"], color=INK["bad"], ls="--", lw=1,
                     label=f"ArviZ threshold {scalars['good_k']:.2f}")
        axis.axhline(0.5, color=INK["accent"], ls=":", lw=1, label="0.5")
        axis.set(title=f"{key}\nmax k = {scalars['max_pareto_k']:.3f}, "
                       f"{len(k):,} observations",
                 xlabel="observation", ylabel="Pareto k")
        axis.legend(fontsize=6)
    for index in range(len(pointwise_available), rows_needed * columns):
        axes[index // columns, index % columns].axis("off")
    figure.tight_layout()
    savefig(figure, "P6_pareto_k.png")
    plt.show()
else:
    print("no pointwise LOO checkpoints in this run folder")

## 6.6, The decision dashboard

The gate matrix and the two verdict columns in one place. Reading it left to right: what the posterior
said, whether each condition for reading it held, and what may therefore be reported. The point of
separating the columns is that a failure is attributable. A row that is green in every gate and
`INCONCLUSIVE` in the last column is a genuine inconclusive result; a row that is `SUPPORTED` pre-gate
and `NOT REPORTABLE` post-gate is an unmet condition, and the failing gate names it.

In [ ]:
#@title 6.6  The gate matrix and the verdicts
gate_columns = [column for column in VERDICTS_VIEW.columns if column.startswith("gate_")]


def gate_value(value) -> float:
    """A gate as a number for the heat map: 1 passed, 0 failed, NaN not applicable."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    return float(bool(value))


matrix = np.array([[gate_value(VERDICTS_VIEW[column].iloc[row]) for column in gate_columns]
                   for row in range(len(VERDICTS_VIEW))], dtype=float)

figure, axes = plt.subplots(1, 2, figsize=(13.5, max(3.0, 0.5 * len(VERDICTS_VIEW))),
                            gridspec_kw={"width_ratios": [1.5, 1]})
axes[0].imshow(np.ma.masked_invalid(matrix), cmap="RdYlGn", vmin=-0.4, vmax=1.4, aspect="auto")
axes[0].set_xticks(np.arange(len(gate_columns)))
axes[0].set_xticklabels([column[5:] for column in gate_columns], rotation=35, ha="right",
                        fontsize=8)
axes[0].set_yticks(np.arange(len(VERDICTS_VIEW)))
axes[0].set_yticklabels(VERDICTS_VIEW.claim, fontsize=8)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]
        text = "n/a" if np.isnan(value) else ("PASS" if value > 0.5 else "FAIL")
        axes[0].text(j, i, text, ha="center", va="center", fontsize=7, color="#1F2933")
axes[0].set_title("every gate, evaluated on its own")
axes[0].grid(False)

y = np.arange(len(VERDICTS_VIEW))
for offset, (column, marker) in enumerate((("pre_gate", "o"), ("post_gate", "s"))):
    for position, value in enumerate(VERDICTS_VIEW[column]):
        _, edge = _PALETTE.get(value, ("#F3F5F7", "#4A5560"))
        axes[1].scatter(offset, position, marker=marker, s=260, color=edge, alpha=0.85)
        axes[1].text(offset, position, value.split()[0][:4], ha="center", va="center",
                     fontsize=6, color="white", fontweight="bold")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["pre-gate", "post-gate"])
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_yticks(y)
axes[1].set_yticklabels(VERDICTS_VIEW.claim, fontsize=8)
axes[1].set_title("what the posterior said, and what may be reported")
axes[1].grid(False)
figure.tight_layout()
savefig(figure, "P6_dashboard.png")
plt.show()
show(VERDICTS_VIEW[["claim", "pre_gate", "post_gate", "gates_failed", "loo_reading", "reason"]],
     caption="the reason column names the gate or the rule leg that decided the post-gate verdict")

## 6.7, Where the time went, and the figure index

The timings are what the run actually cost, per task, which is the honest answer to "how long does
this take" and the input to any future estimate. The index lists every figure the run produced, across
all six parts.

In [ ]:
#@title 6.7  Timings and the index of every figure
_TIMINGS["part6_figures"] = time.time() - PART6_STARTED
atomic_json(ckpt(TIMINGS_FILE), _TIMINGS)
_record(TIMINGS_FILE)

timing_frame = PROGRESS.frame().dropna(subset=["seconds"]).sort_values("seconds",
                                                                      ascending=True)
if len(timing_frame):
    figure, axis = plt.subplots(figsize=(8.5, max(2.6, 0.3 * len(timing_frame))))
    axis.barh(np.arange(len(timing_frame)), timing_frame.seconds / 60, color=INK["primary"],
              alpha=0.8)
    axis.set_yticks(np.arange(len(timing_frame)))
    axis.set_yticklabels(timing_frame.task, fontsize=7)
    axis.set(xlabel="minutes", title=f"measured cost of this run: "
                                     f"{timing_frame.seconds.sum() / 3600:.1f} h in total")
    figure.tight_layout()
    savefig(figure, "P6_timings.png")
    plt.show()
show(PROGRESS.frame(), caption="declared weight against measured seconds")

FIGURE_INDEX = pd.DataFrame(
    [{"figure": path.name, "kilobytes": round(path.stat().st_size / 1024, 1)}
     for path in sorted(FIG_DIR.glob("*.png"))])
save_table(FIGURE_INDEX, "06_figure_index.parquet")
show(FIGURE_INDEX, caption=f"{len(FIGURE_INDEX)} figures in {FIG_DIR}")

TABLE_INDEX = pd.DataFrame(
    [{"table": path.name, "kilobytes": round(path.stat().st_size / 1024, 1)}
     for path in sorted(TAB_DIR.glob("*.csv"))])
show(TABLE_INDEX, caption=f"{len(TABLE_INDEX)} result tables in {TAB_DIR}")

flush_hash_cache()
banner("RUN COMPLETE")
print(f"mode        : {MODE}" + ("" if IS_FULL else "   (NON-REPORTABLE)"))
print(f"run folder  : {CKPT_DIR}")
print(f"artifacts   : {len(MANIFEST['artifacts'])} tracked in analysis_manifest.json")
print(f"figures     : {len(FIGURE_INDEX)}   tables: {len(TABLE_INDEX)}")
print("\nverdicts:")
for row in VERDICTS_VIEW.itertuples():
    print(f"  {row.claim:<22s} {row.post_gate:<16s} "
          f"(pre-gate {row.pre_gate}{', gates failed: ' + row.gates_failed if row.gates_failed != '-' else ''})")

---
## What can and cannot be concluded

`SUPPORTED` and `REFUTED` require `full` mode, identification, and every gate that applies to that
claim. The thresholds behind them are the ones `tab:bayesDecisions` preregisters, and Part 0.8 checked
that the priors this notebook fits are the priors those thresholds were calibrated against.

`INCONCLUSIVE` means the checks are usable and the rule did not decide. It is a result, not a failure.

`NOT REPORTABLE` means a gate failed or the run is not reportable by construction. It is not a null
result, and it must not be read as one: the estimand is unavailable, not zero.

`NOT IDENTIFIED` means the design cannot answer the question. For a D2 branch that is the site bar of
`tab:bayesDecisions`, and Part 2.5 says how many unambiguous sites the branch actually supplied.

Three things bound every reading above, and none of them can be fixed by any fit. One training seed
per geometry, so a difference between geometries is also a difference between trained models. A
truncated analysis context that is non-zero on exactly the six runs identifying the stride branch. And
a token count that moves with the stride, so M1 cannot separate overlap from sequence length. Part 5.6
prints all of them with the run summary, and they belong beside any number taken from this notebook
into the report.

Keep the executed notebook, `analysis_manifest.json`, the CSV tables and the checkpoints together.
The verdict table on its own does not carry the conditions that made it readable.